# Adapter + MR — development, final và official-val

Pilot42/43/44 là kết quả lịch sử18/11; final và official-val đã chạy20/11 trên chính notebook này. Mặc định mọi cờ chạy tắt. Encoder/Adapter luôn đóng băng trong MR; không FT/TME full.

Runtime mới: chạy cấu hình/bootstrap và định nghĩa ban đầu, rồi các định nghĩa final; chỉ bật `RUN_FINAL_PIPELINE` ở cell điều phối cuối khi chủ động chạy/resume. Pipeline tự xác minh → lock → final42/43/44 → benchmark → tổng hợp. Pilot/diagnostic cũ được guard để Run all không ghi lại artifact cũ. Kết quả cuối và test_report nằm ở cell đọc lại cuối.


In [ ]:
CFG = {'preparation_root': '/content/drive/MyDrive/ĐATN/data/processed/benchmark_preparation/FashionIQ/fashioniq_prep_v1_20261103', 'output_root': '/content/drive/MyDrive/ĐATN/data/processed/adapter_mr_pilot/FashionIQ', 'campaign_id': 'adapter_mr_pilot_v1_seed44', 'local_root': '/content/visfit_adapter_mr_pilot', 'legacy_conventional_root': '/content/drive/MyDrive/ĐATN/data/processed/tme_development/FashionIQ/tme_dev_v1_seed42', 'legacy_adapter_root': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_development/FashionIQ/retrieval_only_dev_v1_base42_adapter42_checked_v2_storagefix', 'seed': 44, 'epochs': 3, 'hp': {'batch_size': 32, 'learning_rate': 0.0001, 'weight_decay': 0.01, 'gradient_clip': 1.0, 'retrieval_temperature': 0.07, 'checkpoint_every_steps': 64}, 'adapter': {'width': 128, 'heads': 4}, 'eval_batch_size': 32, 'export_top_k': 50, 'min_local_free_gib': 15, 'min_drive_free_gib': 4, 'RUN_PILOT': False}

RUN_PILOT_VALIDATION = False  # Historical pilot validation/reporting is opt-in.


In [ ]:
# Chỉ stdlib trước import model; bootstrap fix: dependency-owner-and-loaded-version-v2.
import sys, subprocess, importlib.util, importlib.metadata, json
from pathlib import Path
BOOTSTRAP = {'python':sys.version,'installs':[],'pins':{},'passed':False,'bootstrap_fix':'dependency-owner-and-loaded-version-v2'}
local=Path('/content/visfit_adapter_mr_pilot_bootstrap');local.mkdir(parents=True,exist_ok=True)
BOOTSTRAP['loaded_before_bootstrap']={k:{'version':getattr(v,'__version__',None),'path':getattr(v,'__file__',None)} for k,v in list(sys.modules.items()) if k in ('torch','transformers','tokenizers','huggingface_hub')}
PINS = {'transformers':'4.57.1','tokenizers':'0.22.1','huggingface-hub':'0.36.0'}
for package in ('torch','numpy','pandas','Pillow'):
    try: BOOTSTRAP['pins'][package]=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        raise RuntimeError('Colab base package missing: '+package+'; use a standard GPU runtime') from None
needed=[]
if importlib.util.find_spec('psutil') is None:needed.append('psutil')
for package,version in PINS.items():
    try: installed=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: installed=None
    if installed!=version:needed.append(package+'=='+version)
    module={'huggingface-hub':'huggingface_hub'}.get(package,package)
    loaded=any(n==module or n.startswith(module+'.') for n in sys.modules)
    loaded_version=getattr(sys.modules.get(module),'__version__',None)
    if loaded and (installed!=version or loaded_version!=version):
        BOOTSTRAP.update(restart_required=True,blocked_package=package,installed_version=installed,loaded_version=loaded_version)
        (local/'bootstrap_report.json').write_text(json.dumps(BOOTSTRAP,indent=2))
        raise RuntimeError('A dependency requiring a version change is already loaded: '+package+'. Save the notebook and use Runtime > Restart session, then run from the top. No automatic install/restart loop.')
    BOOTSTRAP['pins'][package]=version
if needed:
    p=subprocess.run([sys.executable,'-m','pip','install','--quiet',*needed],capture_output=True,text=True)
    if p.returncode:raise RuntimeError('Dependency installation failed; stopped before importing models. No torch/CUDA upgrade requested.')
    BOOTSTRAP['installs']=needed
check=subprocess.run([sys.executable,'-m','pip','check'],capture_output=True,text=True)
BOOTSTRAP['pip_check']={'returncode':check.returncode,'report':check.stdout+check.stderr}
# Scope by dependent package owner, not a substring of the dependency mentioned in the error.
# Gradio/Diffusers are not used by this pilot; their conflicts remain visible in the report.
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name
pending=['torch','torchvision','transformers','tokenizers','huggingface-hub','numpy','pandas','Pillow','safetensors','packaging']
core=set()
while pending:
    owner=canonicalize_name(pending.pop())
    if owner in core:continue
    core.add(owner)
    for raw in importlib.metadata.requires(owner) or []:
        req=Requirement(raw)
        if req.marker is None or req.marker.evaluate({'extra':''}):pending.append(req.name)
lines=[line for line in (check.stdout+check.stderr).splitlines() if line.strip() and 'No broken requirements found' not in line]
critical=[line for line in lines if canonicalize_name(line.split()[0]) in core]
BOOTSTRAP['dependency_scope']={'core_packages':sorted(core),'critical_conflicts':critical,'outside_pilot_conflicts':[line for line in lines if line not in critical]}
local=Path('/content/visfit_adapter_mr_pilot_bootstrap');local.mkdir(parents=True,exist_ok=True)
(local/'bootstrap_report.json').write_text(json.dumps(BOOTSTRAP,indent=2))
if check.returncode and critical:raise RuntimeError('pip check found Pilot dependency conflicts: '+'; '.join(critical))
probe=subprocess.run([sys.executable,'-c',
    'import torch,transformers; from transformers import CLIPModel,CLIPTokenizer,CLIPImageProcessor; from transformers.generation import GenerationMixin; print(transformers.__version__); print(transformers.__file__)'],capture_output=True,text=True)
BOOTSTRAP['import_probe']={'returncode':probe.returncode,'stdout':probe.stdout,'traceback':probe.stderr}
(local/'bootstrap_report.json').write_text(json.dumps(BOOTSTRAP,indent=2))
if probe.returncode:raise RuntimeError('CLIP dependency import failed in a clean subprocess. Full traceback: /content/visfit_adapter_mr_pilot_bootstrap/bootstrap_report.json. Diagnose before dataset/training.')
BOOTSTRAP['passed']=True
(local/'bootstrap_report.json').write_text(json.dumps(BOOTSTRAP,indent=2))
print('Bootstrap đạt; giữ nguyên torch/CUDA:',BOOTSTRAP['pins']['torch'])
print('Pilot dependency conflicts:',len(critical),'| clean CLIP import:',probe.returncode==0,'| packages installed:',BOOTSTRAP['installs'])
if check.returncode:print('pip check có xung đột ngoài dependency graph của pilot; giữ nguyên đầy đủ trong bootstrap_report.json. Không coi toàn môi trường là pip-clean.')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
SOURCE = {'source_verification': 'Actual Drive metadata and raw small artifacts read; no Drive mutations', 'encoder': {'encoder_checkpoint_sha256': None, 'epochs': 3, 'format': 'legacy_conventional_v1', 'kind': 'encoder', 'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_development/FashionIQ/tme_dev_v1_seed42/conventional/best.pt', 'phase': 'development', 'seed': 42, 'sha256': 'f62513032c086ce5c2e09a9774add5bc590e6ded81c5b9d92d0229cd526f080b', 'signature': 'c6cf0642cdcf5b7c7c5fc8fe6246fd7f0c668377036fefb12d45cfb1c10dc3d3'}, 'adapter': {'encoder_checkpoint_sha256': 'f62513032c086ce5c2e09a9774add5bc590e6ded81c5b9d92d0229cd526f080b', 'epochs': 10, 'format': 'legacy_adapter_v2', 'kind': 'adapter', 'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_development/FashionIQ/retrieval_only_dev_v1_base42_adapter42_checked_v2_storagefix/best.pt', 'phase': 'development', 'seed': 42, 'sha256': '7d92ef45dfa5cec81ad9042b6bc1eceb2abfdfc7cc957595083d8d2c3c923783', 'signature': '0d0fd5fbacb1bb940499a2046e1c18a489792f38dce6406134c3de6f0df0e0d9'}, 'expected_metrics': {'A': {'overall': {'R@1': 25.833096037059164, 'R@5': 49.44016690768524, 'R@10': 61.020876488685154, 'R@50': 84.24027038747225}, 'Mean': 72.6305734380787, 'per_category': {'dress': {'R@1': 22.073578595317723, 'R@5': 43.97993311036789, 'R@10': 54.515050167224075, 'R@50': 79.76588628762542}, 'shirt': {'R@1': 27.545909849749584, 'R@5': 51.41903171953256, 'R@10': 61.769616026711184, 'R@50': 84.47412353923205}, 'toptee': {'R@1': 27.879799666110184, 'R@5': 52.92153589315526, 'R@10': 66.7779632721202, 'R@50': 88.48080133555926}}, 'n_query': 1796}, 'B': {'overall': {'R@1': 26.946713493131437, 'R@5': 52.167864314921, 'R@10': 63.47107684118644, 'R@50': 86.3000951046244}, 'Mean': 74.88558597290542, 'per_category': {'dress': {'R@1': 23.244147157190636, 'R@5': 45.65217391304348, 'R@10': 57.52508361204013, 'R@50': 81.27090301003345}, 'shirt': {'R@1': 29.883138564273793, 'R@5': 53.08848080133556, 'R@10': 64.44073455759599, 'R@50': 86.97829716193655}, 'toptee': {'R@1': 27.712854757929883, 'R@5': 57.76293823038397, 'R@10': 68.44741235392321, 'R@50': 90.65108514190318}}, 'n_query': 1796}}, 'tolerances': {'A_metric_reproduction_pp': 0.01, 'A_source_file': 'baseline_verification.json', 'adapter_gate_off_atol': 1e-06, 'B_rerun_tolerance_explicitly_defined_in_artifact': None, 'B_saved_prediction_metric_consistency_atol': 1e-10, 'B_note': 'No separate numerical rerun tolerance exists for B; source completion recomputes saved ranks with <1e-10 metric consistency. Do not invent relaxed B tolerance.'}, 'manifests': {'train': {'filename': 'development_train.csv', 'sha256': '465f758604249e2005cc58c0dfa79edf236081613e908c1e5942fd61abf17086', 'count': 16204}, 'validation': {'filename': 'development_val.csv', 'sha256': 'ace52e94935adacfd19aeaea7a99314602954bce9bff43458ceb564fdf05a8b5', 'count': 1796}, 'gallery': {'filename': 'development_gallery.csv', 'sha256': '83c08f14b606227c1a406b5074163ef70e42eb025144d19e2b1de4f38d234dc5', 'count': 1796}}, 'cache': {'root': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_development/FashionIQ/retrieval_only_dev_v1_base42_adapter42_checked_v1/cache', 'fingerprint': 'b6fbd67a1d203ced8626b25afd592cba578b196a2d9b9b6bff5e2ad0e875daff', 'index_sha256': 'c90c8f77acc6d48e7d3adc6df8deaa1cd56aec49b7b84f8ab2388a1bf881f841', 'index_drive_id': '1CkZxYsIOkA1RzPea8-MyGoNWXPMzLbq9', 'counts': {'eos': 18000, 'input_ids': 18000, 'mask': 18000, 'reference': 12525, 'reference_pooled': 12525, 'target_pooled': 17839, 'text': 18000}, 'contract': {'amp': False, 'attention_backend': 'eager', 'base_checkpoint_sha256': 'f62513032c086ce5c2e09a9774add5bc590e6ded81c5b9d92d0229cd526f080b', 'cache_shard_size': 128, 'caption_policy': 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1', 'dtype': 'float32', 'extractor_schema': 'trained-conventional-finalLN-final-image-v1', 'extractor_source_sha256': '82a37f5162dfd31ac5bbdd37bc903cf260ce8983c4788331399620274b1cac7b', 'feature_fingerprint': '1e86518f73f1640b6907474749b86e5d175b64c343b4de9f2f2e2e71af926037', 'gallery_sha256': '83c08f14b606227c1a406b5074163ef70e42eb025144d19e2b1de4f38d234dc5', 'image_extraction_contract': {'attention_backend': 'eager', 'autocast': False, 'batch_size': 32, 'caption_policy': 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1', 'cuda': '13.0', 'device': 'cuda', 'device_name': 'Tesla T4', 'dtype': 'float32', 'encoder_id': 'patrickjohncyh/fashion-clip', 'encoder_weight_sha256': '7b2e2766a4372167f9db95876b897dcecfe1ca5963f2bf620b3a8393f3ad6366', 'feature_shard_size': 256, 'implementation': 'fashioniq-preparation-v1.0.0', 'libraries': {'Pillow': '11.3.0', 'huggingface-hub': '0.36.0', 'numpy': '2.4.6', 'pandas': '2.2.3', 'tokenizers': '0.22.1', 'torch': '2.11.0+cu130', 'transformers': '4.57.1'}, 'processor': {'_processor_class': None, 'crop_size': {'height': 224, 'width': 224}, 'do_center_crop': True, 'do_convert_rgb': True, 'do_normalize': True, 'do_rescale': True, 'do_resize': True, 'image_mean': [0.48145466, 0.4578275, 0.40821073], 'image_processor_type': 'CLIPImageProcessor', 'image_std': [0.26862954, 0.26130258, 0.27577711], 'resample': 3, 'rescale_factor': 0.00392156862745098, 'size': {'shortest_edge': 224}}, 'reference': 'vision_model.last_hidden_state INCLUDING CLS; no projection or L2', 'repeat_acceptance': {'atol': 6.103515625e-05, 'note': 'Repeat same input and execution contract; not a historical feature acceptance threshold.', 'predeclared': True, 'rtol': 6.103515625e-05}, 'revision': '7e3ba62ce16b379a1ab479346b66f192e76f51b7', 'target': 'visual_projection(vision_model.pooler_output) then L2, [512]', 'text': 'text_model.last_hidden_state, [77,512], no projection/normalization + binary attention_mask', 'tf32': False, 'tokenizer': {'class': 'CLIPTokenizer', 'max_length': 77, 'special_tokens': {'bos_token': '<|startoftext|>', 'eos_token': '<|endoftext|>', 'pad_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, 'vocab_sha256': 'f9190aead40a965a09db1ba2e556dccac0eda8423cb4926bcd994722899ec68e'}, 'truncation_policy': 'error_if_over_77_preserve_all_source_rows'}, 'implementation': 'visfit-adapter-development-cache-v1', 'runtime': {'device': 'cuda', 'gpu': 'Tesla T4', 'packages': {'numpy': '2.4.6', 'torch': '2.14.1+cu130', 'transformers': '4.57.1'}}, 'source_image_index_sha256': {'reference': 'b0391ce423bb89b335aec95d9b2c752bd146d6e79bb6f4d1319ccd6898e5ad4b', 'target': '71a80c213d06a0605451c354386a9d41614d4a8f281e52aaddaa167bc046c1fb'}, 'text': 'strict trained conventional last stream -> final LN, padded outputs zero; frozen trained projection applied later', 'text_encode_batch_size': 16, 'tf32': False, 'train_sha256': '465f758604249e2005cc58c0dfa79edf236081613e908c1e5942fd61abf17086', 'validation_sha256': 'ace52e94935adacfd19aeaea7a99314602954bce9bff43458ceb564fdf05a8b5'}, 'shard_size': 128, 'role_shapes': {'eos': {'shape': [], 'dtype': 'int64'}, 'input_ids': {'shape': [77], 'dtype': 'int64'}, 'mask': {'shape': [77], 'dtype': 'int64'}, 'reference': {'shape': [50, 768], 'dtype': 'float32'}, 'reference_pooled': {'shape': [512], 'dtype': 'float32'}, 'target_pooled': {'shape': [512], 'dtype': 'float32'}, 'text': {'shape': [77, 512], 'dtype': 'float32'}}}, 'prediction_sha256': {'A': '00712dcc75ac63ded64e75465e566c1c183365eda3df7b7f832ddc0f411568a2', 'B': '750967b2753da6026514051bfee5825a7335f10efbbe32e2da0e50623e73b5c9'}, 'actual_checkpoint_binary_verified': ['adapter'], 'checkpoint_binary_deferred_to_colab': ['encoder (~894MB)'], 'source_file_sha256': {'adapter_cache_audit.json': 'e5578b21ada12bab97966322929b0c3c0726cb1abe0150b85aff4be990ac7f2b', 'prep_feature_metadata.json': '48ee1866dc22f07fa588ea815318ddc9995cfa8734ffd0f2acc88c430f855342', 'development_full_cache_index.json': 'c90c8f77acc6d48e7d3adc6df8deaa1cd56aec49b7b84f8ab2388a1bf881f841', 'adapter_base_preflight.json': 'a0953b19940bcc5edc0aaec676a39277a7c168b5bc2c6713d5398168cec30368', 'adapter_best_result_verification.json': '5f2dd56bdd671c17db222c77966b93f7f0c39ff8d10af3dd3ccd7cd2c4809508', 'adapter_runtime_contract.json': '02696d2a36995ffd90791671203673b689bede949c5f05d92c20c4741e65ee2d', 'baseline_verification.json': '1ddd249b7d0674dae29e582a8383b7b7b96848994bc1b79be33213a8d2ab68a2', 'encoder_config.json': 'b3d48c8550a8f36ee1bbbc575878d10ac10e416509b8a599ba23019c7561935f', 'prep_protocol_config.json': '2fa2322dee2ebe9964e4ce63ebf0cd8dabad83392a39e288d0e43334e40faffb', 'encoder_best.json': '9ec6117b48e0df64e49247d9f41da1cc035c1a504e00551abc1aad654aadbca1', 'adapter_development_summary.json': '96da4b19138b4e2e687d1f8623a7411cfeedf0d6ba901a923639c2b829766139', 'adapter_locked_training_config.json': '4db8657b94ba13e5b0321af2b01986468f7e8949f7d2c539276cf8d84bf3ed3e', 'adapter_training_config.json': 'e8ccdc60d5e7c655a3a9009bb8b093a4e95d2d0074d18442a6ed3d77da5dfe5d', 'adapter_verified_trial_reference.json': 'dfaf7c892539869b9a001280e8379ac5c381635eddfa12b8e24a2cfc6a5cc833', 'full_development_seed42_result.json': '8300050811f23178f3e49562877452f94b23c8a2d01b8330d33d1afc98cbbbd1', 'adapter_full_cache_reuse.json': '3deb322a2d0679fb2ad5ca1bd2e87b0fe1b9c4c4e20baba920a1f01287e894b9', 'adapter_source_provenance.json': 'da564ddd55c8d82da1f1347c8ebd2948366ec050f797ec80bf228d0bdc02689f', 'encoder_metric_history.json': '253717bafb9c38d8dd1f0cf484d78b6165c2708233afec225bc7a905e3d8829e', 'adapter_experiment_config.json': '6a8a18859e4253cb67311d071caa15fe8c385bd48c8bffa9f9f9d68e7618901e', 'adapter_best.json': 'ee3be0b8e8ef22bac279a2de1584915314d3323088b5bca67d30ddc641c73e5c'}, 'warnings': ['Baseline verification records one rank difference, with identical aggregate metrics and 0.01pp configured source tolerance; do not require exact baseline rank equality without diagnosing that known evidence.', 'Trial2048 cache is not sufficient; checked_v1 full cache verified18000 query IDs and manifest alignment.', 'No official-val/final checkpoint is part of these source records.']}
PREPARATION_ANCHORS = {'data_audit.json': 'fbaa7eae7b99c5e5aeeddcb0bc6027d3870e8c4f9be5658559caa95662321e95', 'feature_metadata.json': '48ee1866dc22f07fa588ea815318ddc9995cfa8734ffd0f2acc88c430f855342', 'extraction_contract.json': 'bbd53b94172aa5c07a358a0697163c620a823b297c6fec6209167bdb59d1f0a1', 'training_plan.json': 'e5496feae33ab0e9dc832e29a1a9097b2d7fed6c2683bd9d4314d6e4973bdc21', 'development_gallery.csv': '83c08f14b606227c1a406b5074163ef70e42eb025144d19e2b1de4f38d234dc5', 'development_val.csv': 'ace52e94935adacfd19aeaea7a99314602954bce9bff43458ceb564fdf05a8b5', 'protocol_config.json': '2fa2322dee2ebe9964e4ce63ebf0cd8dabad83392a39e288d0e43334e40faffb', 'development_train.csv': '465f758604249e2005cc58c0dfa79edf236081613e908c1e5942fd61abf17086', 'feature_views/development_gallery.csv': '283199425c5fc88fb8e063ea28be629e611e44513689e97f22ee3d0ec5a33927', 'feature_views/development_val.csv': 'cbd8d7b72fa7fb4e3c38d64f3bb8ceeeba73be946e9918c5a77c856a42217619', 'feature_views/development_train.csv': '6f7a5e6ab8792dd8fd08d6fea8c472a9d9b56b59af38ba146467d797f8a98ff0', 'official_train_queries.csv': '539c53d61e9b3d01bf010d089ffa16cef2cd437229feee4a475884586188fd11'}
EXTRACTION_CONTRACT = {'implementation': 'fashioniq-preparation-v1.0.0', 'encoder_id': 'patrickjohncyh/fashion-clip', 'revision': '7e3ba62ce16b379a1ab479346b66f192e76f51b7', 'encoder_weight_sha256': '7b2e2766a4372167f9db95876b897dcecfe1ca5963f2bf620b3a8393f3ad6366', 'processor': {'_processor_class': None, 'do_resize': True, 'size': {'shortest_edge': 224}, 'resample': 3, 'do_center_crop': True, 'crop_size': {'height': 224, 'width': 224}, 'do_rescale': True, 'rescale_factor': 0.00392156862745098, 'do_normalize': True, 'image_mean': [0.48145466, 0.4578275, 0.40821073], 'image_std': [0.26862954, 0.26130258, 0.27577711], 'do_convert_rgb': True, 'image_processor_type': 'CLIPImageProcessor'}, 'tokenizer': {'class': 'CLIPTokenizer', 'vocab_sha256': 'f9190aead40a965a09db1ba2e556dccac0eda8423cb4926bcd994722899ec68e', 'special_tokens': {'bos_token': '<|startoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, 'max_length': 77}, 'caption_policy': 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1', 'truncation_policy': 'error_if_over_77_preserve_all_source_rows', 'dtype': 'float32', 'attention_backend': 'eager', 'autocast': False, 'tf32': False, 'libraries': {'torch': '2.11.0+cu130', 'transformers': '4.57.1', 'tokenizers': '0.22.1', 'huggingface-hub': '0.36.0', 'numpy': '2.4.6', 'pandas': '2.2.3', 'Pillow': '11.3.0'}, 'device': 'cuda', 'device_name': 'Tesla T4', 'cuda': '13.0', 'batch_size': 32, 'feature_shard_size': 256, 'reference': 'vision_model.last_hidden_state INCLUDING CLS; no projection or L2', 'text': 'text_model.last_hidden_state, [77,512], no projection/normalization + binary attention_mask', 'target': 'visual_projection(vision_model.pooler_output) then L2, [512]', 'repeat_acceptance': {'atol': 6.103515625e-05, 'rtol': 6.103515625e-05, 'predeclared': True, 'note': 'Repeat same input and execution contract; not a historical feature acceptance threshold.'}}
LEGACY = {'adapter_config_sha256': 'e8ccdc60d5e7c655a3a9009bb8b093a4e95d2d0074d18442a6ed3d77da5dfe5d', 'adapter_code_sha256': '63c5666a61a6fb4afbee2b84f49dbc3d77fa8a2ee14bc47e91f34afd23674dd2', 'adapter_best': {'phase': 'retrieval_only_development', 'sha256': '7d92ef45dfa5cec81ad9042b6bc1eceb2abfdfc7cc957595083d8d2c3c923783', 'signature': '0d0fd5fbacb1bb940499a2046e1c18a489792f38dce6406134c3de6f0df0e0d9', 'step': 5070}, 'adapter_last': {'phase': 'retrieval_only_development', 'sha256': '7d92ef45dfa5cec81ad9042b6bc1eceb2abfdfc7cc957595083d8d2c3c923783', 'signature': '0d0fd5fbacb1bb940499a2046e1c18a489792f38dce6406134c3de6f0df0e0d9', 'step': 5070}, 'encoder_best': {'phase': 'conventional', 'sha256': 'f62513032c086ce5c2e09a9774add5bc590e6ded81c5b9d92d0229cd526f080b', 'signature': 'c6cf0642cdcf5b7c7c5fc8fe6246fd7f0c668377036fefb12d45cfb1c10dc3d3', 'step': 12153}, 'encoder_config_sha256': 'b3d48c8550a8f36ee1bbbc575878d10ac10e416509b8a599ba23019c7561935f', 'encoder_metrics': {'Mean': 72.6305734380787, 'gallery_counts': {'dress': 598, 'shirt': 599, 'toptee': 599}, 'n_query': 1796, 'overall': {'R@1': 25.833096037059164, 'R@10': 61.020876488685154, 'R@5': 49.44016690768524, 'R@50': 84.24027038747225}, 'per_category': {'dress': {'R@1': 22.073578595317723, 'R@10': 54.515050167224075, 'R@5': 43.97993311036789, 'R@50': 79.76588628762542}, 'shirt': {'R@1': 27.545909849749584, 'R@10': 61.769616026711184, 'R@5': 51.41903171953256, 'R@50': 84.47412353923205}, 'toptee': {'R@1': 27.879799666110184, 'R@10': 66.7779632721202, 'R@5': 52.92153589315526, 'R@50': 88.48080133555926}}, 'query_counts': {'dress': 598, 'shirt': 599, 'toptee': 599}, 'scope': 'full locked development validation; NOT official-val', 'search': 'exact L2 cosine FP32, reference retained, exact ties original gallery_index ascending', 'unit': 'percent'}, 'encoder_text_hash': 'acc814a8bb4c419c3a7489bc3efbf000564d5e8eae3660dfa42ceb74b332d265', 'adapter_initial_sha256': 'd6df8bce9821c85b8c57e8b235954bdde3bfc833fb6ec064a01e145be60afb1e'}
PILOT_HP = {'batch_size': 32, 'learning_rate': 0.0001, 'weight_decay': 0.01, 'gradient_clip': 1.0, 'retrieval_temperature': 0.07, 'checkpoint_every_steps': 64}
CAPTION_POLICY = 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1'
CATEGORIES = ('dress', 'shirt', 'toptee')
DVR_HIDDEN = 256
DVR_ALPHA = 1.0
NATIVE_ATOL = 3e-05
NATIVE_RTOL = 3e-05
GATE_ATOL = 1e-06
SOURCE_FILE_LOCKS = {'/content/drive/MyDrive/ĐATN/data/processed/tme_development/FashionIQ/tme_dev_v1_seed42/conventional/config.json': 'b3d48c8550a8f36ee1bbbc575878d10ac10e416509b8a599ba23019c7561935f', '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_development/FashionIQ/retrieval_only_dev_v1_base42_adapter42_checked_v2_storagefix/training_config.json': 'e8ccdc60d5e7c655a3a9009bb8b093a4e95d2d0074d18442a6ed3d77da5dfe5d'}
SOURCE_NOTEBOOKS = {'taskb_dvr_dev.ipynb': 'ce1ca7c31865101577ce0ce44335e9e73921d3445b1ff86c8c90bd5f3af34f6f', 'taskb_final.ipynb': 'd7501736afcb0b8f97e2afb565ef8cfb389a2e5f60b285e7708c712e5fe4a851', 'taskb_adapter_full.ipynb': 'd18d9cd05d6c039b39d668ac75a97caa66647078d6198751a1d64de216037432', 'taskb_adapter_dev.ipynb': '99020095c41dad5d3d9ce289154ae5778b447e390a5ec9d7ed0a3f2614152404', 'taskb_prepare.ipynb': '6b24bddf6fdfdc49f3610e612245696b6b576d5ef689b526e6f55b96a0a9ca34', 'taskb_xaa_dev.ipynb': 'cade489341533dd2e767dc5ba0cbd400b7a801d9a0684fea6abe400a5ce1051d', 'taskb_encoder_dev.ipynb': '5b81001a4a82a62a51dcd1233701d2497a8c97af250391213cd79c88944e3904'}
SOURCE_REGISTRY = {42: {'source_verification': 'Actual Drive metadata and raw small artifacts read; no Drive mutations', 'encoder': {'encoder_checkpoint_sha256': None, 'epochs': 3, 'format': 'legacy_conventional_v1', 'kind': 'encoder', 'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_development/FashionIQ/tme_dev_v1_seed42/conventional/best.pt', 'phase': 'development', 'seed': 42, 'sha256': 'f62513032c086ce5c2e09a9774add5bc590e6ded81c5b9d92d0229cd526f080b', 'signature': 'c6cf0642cdcf5b7c7c5fc8fe6246fd7f0c668377036fefb12d45cfb1c10dc3d3'}, 'adapter': {'encoder_checkpoint_sha256': 'f62513032c086ce5c2e09a9774add5bc590e6ded81c5b9d92d0229cd526f080b', 'epochs': 10, 'format': 'legacy_adapter_v2', 'kind': 'adapter', 'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_development/FashionIQ/retrieval_only_dev_v1_base42_adapter42_checked_v2_storagefix/best.pt', 'phase': 'development', 'seed': 42, 'sha256': '7d92ef45dfa5cec81ad9042b6bc1eceb2abfdfc7cc957595083d8d2c3c923783', 'signature': '0d0fd5fbacb1bb940499a2046e1c18a489792f38dce6406134c3de6f0df0e0d9'}, 'expected_metrics': {'A': {'overall': {'R@1': 25.833096037059164, 'R@5': 49.44016690768524, 'R@10': 61.020876488685154, 'R@50': 84.24027038747225}, 'Mean': 72.6305734380787, 'per_category': {'dress': {'R@1': 22.073578595317723, 'R@5': 43.97993311036789, 'R@10': 54.515050167224075, 'R@50': 79.76588628762542}, 'shirt': {'R@1': 27.545909849749584, 'R@5': 51.41903171953256, 'R@10': 61.769616026711184, 'R@50': 84.47412353923205}, 'toptee': {'R@1': 27.879799666110184, 'R@5': 52.92153589315526, 'R@10': 66.7779632721202, 'R@50': 88.48080133555926}}, 'n_query': 1796}, 'B': {'overall': {'R@1': 26.946713493131437, 'R@5': 52.167864314921, 'R@10': 63.47107684118644, 'R@50': 86.3000951046244}, 'Mean': 74.88558597290542, 'per_category': {'dress': {'R@1': 23.244147157190636, 'R@5': 45.65217391304348, 'R@10': 57.52508361204013, 'R@50': 81.27090301003345}, 'shirt': {'R@1': 29.883138564273793, 'R@5': 53.08848080133556, 'R@10': 64.44073455759599, 'R@50': 86.97829716193655}, 'toptee': {'R@1': 27.712854757929883, 'R@5': 57.76293823038397, 'R@10': 68.44741235392321, 'R@50': 90.65108514190318}}, 'n_query': 1796}}, 'tolerances': {'A_metric_reproduction_pp': 0.01, 'A_source_file': 'baseline_verification.json', 'adapter_gate_off_atol': 1e-06, 'B_rerun_tolerance_explicitly_defined_in_artifact': None, 'B_saved_prediction_metric_consistency_atol': 1e-10, 'B_note': 'No separate numerical rerun tolerance exists for B; source completion recomputes saved ranks with <1e-10 metric consistency. Do not invent relaxed B tolerance.'}, 'manifests': {'train': {'filename': 'development_train.csv', 'sha256': '465f758604249e2005cc58c0dfa79edf236081613e908c1e5942fd61abf17086', 'count': 16204}, 'validation': {'filename': 'development_val.csv', 'sha256': 'ace52e94935adacfd19aeaea7a99314602954bce9bff43458ceb564fdf05a8b5', 'count': 1796}, 'gallery': {'filename': 'development_gallery.csv', 'sha256': '83c08f14b606227c1a406b5074163ef70e42eb025144d19e2b1de4f38d234dc5', 'count': 1796}}, 'cache': {'root': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_development/FashionIQ/retrieval_only_dev_v1_base42_adapter42_checked_v1/cache', 'fingerprint': 'b6fbd67a1d203ced8626b25afd592cba578b196a2d9b9b6bff5e2ad0e875daff', 'index_sha256': 'c90c8f77acc6d48e7d3adc6df8deaa1cd56aec49b7b84f8ab2388a1bf881f841', 'index_drive_id': '1CkZxYsIOkA1RzPea8-MyGoNWXPMzLbq9', 'counts': {'eos': 18000, 'input_ids': 18000, 'mask': 18000, 'reference': 12525, 'reference_pooled': 12525, 'target_pooled': 17839, 'text': 18000}, 'contract': {'amp': False, 'attention_backend': 'eager', 'base_checkpoint_sha256': 'f62513032c086ce5c2e09a9774add5bc590e6ded81c5b9d92d0229cd526f080b', 'cache_shard_size': 128, 'caption_policy': 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1', 'dtype': 'float32', 'extractor_schema': 'trained-conventional-finalLN-final-image-v1', 'extractor_source_sha256': '82a37f5162dfd31ac5bbdd37bc903cf260ce8983c4788331399620274b1cac7b', 'feature_fingerprint': '1e86518f73f1640b6907474749b86e5d175b64c343b4de9f2f2e2e71af926037', 'gallery_sha256': '83c08f14b606227c1a406b5074163ef70e42eb025144d19e2b1de4f38d234dc5', 'image_extraction_contract': {'attention_backend': 'eager', 'autocast': False, 'batch_size': 32, 'caption_policy': 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1', 'cuda': '13.0', 'device': 'cuda', 'device_name': 'Tesla T4', 'dtype': 'float32', 'encoder_id': 'patrickjohncyh/fashion-clip', 'encoder_weight_sha256': '7b2e2766a4372167f9db95876b897dcecfe1ca5963f2bf620b3a8393f3ad6366', 'feature_shard_size': 256, 'implementation': 'fashioniq-preparation-v1.0.0', 'libraries': {'Pillow': '11.3.0', 'huggingface-hub': '0.36.0', 'numpy': '2.4.6', 'pandas': '2.2.3', 'tokenizers': '0.22.1', 'torch': '2.11.0+cu130', 'transformers': '4.57.1'}, 'processor': {'_processor_class': None, 'crop_size': {'height': 224, 'width': 224}, 'do_center_crop': True, 'do_convert_rgb': True, 'do_normalize': True, 'do_rescale': True, 'do_resize': True, 'image_mean': [0.48145466, 0.4578275, 0.40821073], 'image_processor_type': 'CLIPImageProcessor', 'image_std': [0.26862954, 0.26130258, 0.27577711], 'resample': 3, 'rescale_factor': 0.00392156862745098, 'size': {'shortest_edge': 224}}, 'reference': 'vision_model.last_hidden_state INCLUDING CLS; no projection or L2', 'repeat_acceptance': {'atol': 6.103515625e-05, 'note': 'Repeat same input and execution contract; not a historical feature acceptance threshold.', 'predeclared': True, 'rtol': 6.103515625e-05}, 'revision': '7e3ba62ce16b379a1ab479346b66f192e76f51b7', 'target': 'visual_projection(vision_model.pooler_output) then L2, [512]', 'text': 'text_model.last_hidden_state, [77,512], no projection/normalization + binary attention_mask', 'tf32': False, 'tokenizer': {'class': 'CLIPTokenizer', 'max_length': 77, 'special_tokens': {'bos_token': '<|startoftext|>', 'eos_token': '<|endoftext|>', 'pad_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, 'vocab_sha256': 'f9190aead40a965a09db1ba2e556dccac0eda8423cb4926bcd994722899ec68e'}, 'truncation_policy': 'error_if_over_77_preserve_all_source_rows'}, 'implementation': 'visfit-adapter-development-cache-v1', 'runtime': {'device': 'cuda', 'gpu': 'Tesla T4', 'packages': {'numpy': '2.4.6', 'torch': '2.14.1+cu130', 'transformers': '4.57.1'}}, 'source_image_index_sha256': {'reference': 'b0391ce423bb89b335aec95d9b2c752bd146d6e79bb6f4d1319ccd6898e5ad4b', 'target': '71a80c213d06a0605451c354386a9d41614d4a8f281e52aaddaa167bc046c1fb'}, 'text': 'strict trained conventional last stream -> final LN, padded outputs zero; frozen trained projection applied later', 'text_encode_batch_size': 16, 'tf32': False, 'train_sha256': '465f758604249e2005cc58c0dfa79edf236081613e908c1e5942fd61abf17086', 'validation_sha256': 'ace52e94935adacfd19aeaea7a99314602954bce9bff43458ceb564fdf05a8b5'}, 'shard_size': 128, 'role_shapes': {'eos': {'shape': [], 'dtype': 'int64'}, 'input_ids': {'shape': [77], 'dtype': 'int64'}, 'mask': {'shape': [77], 'dtype': 'int64'}, 'reference': {'shape': [50, 768], 'dtype': 'float32'}, 'reference_pooled': {'shape': [512], 'dtype': 'float32'}, 'target_pooled': {'shape': [512], 'dtype': 'float32'}, 'text': {'shape': [77, 512], 'dtype': 'float32'}}}, 'prediction_sha256': {'A': '00712dcc75ac63ded64e75465e566c1c183365eda3df7b7f832ddc0f411568a2', 'B': '750967b2753da6026514051bfee5825a7335f10efbbe32e2da0e50623e73b5c9'}, 'actual_checkpoint_binary_verified': ['adapter'], 'checkpoint_binary_deferred_to_colab': ['encoder (~894MB)'], 'source_file_sha256': {'adapter_cache_audit.json': 'e5578b21ada12bab97966322929b0c3c0726cb1abe0150b85aff4be990ac7f2b', 'prep_feature_metadata.json': '48ee1866dc22f07fa588ea815318ddc9995cfa8734ffd0f2acc88c430f855342', 'development_full_cache_index.json': 'c90c8f77acc6d48e7d3adc6df8deaa1cd56aec49b7b84f8ab2388a1bf881f841', 'adapter_base_preflight.json': 'a0953b19940bcc5edc0aaec676a39277a7c168b5bc2c6713d5398168cec30368', 'adapter_best_result_verification.json': '5f2dd56bdd671c17db222c77966b93f7f0c39ff8d10af3dd3ccd7cd2c4809508', 'adapter_runtime_contract.json': '02696d2a36995ffd90791671203673b689bede949c5f05d92c20c4741e65ee2d', 'baseline_verification.json': '1ddd249b7d0674dae29e582a8383b7b7b96848994bc1b79be33213a8d2ab68a2', 'encoder_config.json': 'b3d48c8550a8f36ee1bbbc575878d10ac10e416509b8a599ba23019c7561935f', 'prep_protocol_config.json': '2fa2322dee2ebe9964e4ce63ebf0cd8dabad83392a39e288d0e43334e40faffb', 'encoder_best.json': '9ec6117b48e0df64e49247d9f41da1cc035c1a504e00551abc1aad654aadbca1', 'adapter_development_summary.json': '96da4b19138b4e2e687d1f8623a7411cfeedf0d6ba901a923639c2b829766139', 'adapter_locked_training_config.json': '4db8657b94ba13e5b0321af2b01986468f7e8949f7d2c539276cf8d84bf3ed3e', 'adapter_training_config.json': 'e8ccdc60d5e7c655a3a9009bb8b093a4e95d2d0074d18442a6ed3d77da5dfe5d', 'adapter_verified_trial_reference.json': 'dfaf7c892539869b9a001280e8379ac5c381635eddfa12b8e24a2cfc6a5cc833', 'full_development_seed42_result.json': '8300050811f23178f3e49562877452f94b23c8a2d01b8330d33d1afc98cbbbd1', 'adapter_full_cache_reuse.json': '3deb322a2d0679fb2ad5ca1bd2e87b0fe1b9c4c4e20baba920a1f01287e894b9', 'adapter_source_provenance.json': 'da564ddd55c8d82da1f1347c8ebd2948366ec050f797ec80bf228d0bdc02689f', 'encoder_metric_history.json': '253717bafb9c38d8dd1f0cf484d78b6165c2708233afec225bc7a905e3d8829e', 'adapter_experiment_config.json': '6a8a18859e4253cb67311d071caa15fe8c385bd48c8bffa9f9f9d68e7618901e', 'adapter_best.json': 'ee3be0b8e8ef22bac279a2de1584915314d3323088b5bca67d30ddc641c73e5c'}, 'warnings': ['Baseline verification records one rank difference, with identical aggregate metrics and 0.01pp configured source tolerance; do not require exact baseline rank equality without diagnosing that known evidence.', 'Trial2048 cache is not sufficient; checked_v1 full cache verified18000 query IDs and manifest alignment.', 'No official-val/final checkpoint is part of these source records.'], 'prediction_paths': {'A': '/content/drive/MyDrive/ĐATN/data/processed/tme_development/FashionIQ/tme_dev_v1_seed42/conventional/epoch_03_per_query.csv', 'B': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_development/FashionIQ/retrieval_only_dev_v1_base42_adapter42_checked_v2_storagefix/per_query_results.csv'}, 'trained_text_cache': {'root': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_development/FashionIQ/retrieval_only_dev_v1_base42_adapter42_checked_v1/cache/b6fbd67a1d203ced8626b25afd592cba578b196a2d9b9b6bff5e2ad0e875daff', 'index_sha256': 'c90c8f77acc6d48e7d3adc6df8deaa1cd56aec49b7b84f8ab2388a1bf881f841', 'contract': {'amp': False, 'attention_backend': 'eager', 'base_checkpoint_sha256': 'f62513032c086ce5c2e09a9774add5bc590e6ded81c5b9d92d0229cd526f080b', 'cache_shard_size': 128, 'caption_policy': 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1', 'dtype': 'float32', 'extractor_schema': 'trained-conventional-finalLN-final-image-v1', 'extractor_source_sha256': '82a37f5162dfd31ac5bbdd37bc903cf260ce8983c4788331399620274b1cac7b', 'feature_fingerprint': '1e86518f73f1640b6907474749b86e5d175b64c343b4de9f2f2e2e71af926037', 'gallery_sha256': '83c08f14b606227c1a406b5074163ef70e42eb025144d19e2b1de4f38d234dc5', 'image_extraction_contract': {'attention_backend': 'eager', 'autocast': False, 'batch_size': 32, 'caption_policy': 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1', 'cuda': '13.0', 'device': 'cuda', 'device_name': 'Tesla T4', 'dtype': 'float32', 'encoder_id': 'patrickjohncyh/fashion-clip', 'encoder_weight_sha256': '7b2e2766a4372167f9db95876b897dcecfe1ca5963f2bf620b3a8393f3ad6366', 'feature_shard_size': 256, 'implementation': 'fashioniq-preparation-v1.0.0', 'libraries': {'Pillow': '11.3.0', 'huggingface-hub': '0.36.0', 'numpy': '2.4.6', 'pandas': '2.2.3', 'tokenizers': '0.22.1', 'torch': '2.11.0+cu130', 'transformers': '4.57.1'}, 'processor': {'_processor_class': None, 'crop_size': {'height': 224, 'width': 224}, 'do_center_crop': True, 'do_convert_rgb': True, 'do_normalize': True, 'do_rescale': True, 'do_resize': True, 'image_mean': [0.48145466, 0.4578275, 0.40821073], 'image_processor_type': 'CLIPImageProcessor', 'image_std': [0.26862954, 0.26130258, 0.27577711], 'resample': 3, 'rescale_factor': 0.00392156862745098, 'size': {'shortest_edge': 224}}, 'reference': 'vision_model.last_hidden_state INCLUDING CLS; no projection or L2', 'repeat_acceptance': {'atol': 6.103515625e-05, 'note': 'Repeat same input and execution contract; not a historical feature acceptance threshold.', 'predeclared': True, 'rtol': 6.103515625e-05}, 'revision': '7e3ba62ce16b379a1ab479346b66f192e76f51b7', 'target': 'visual_projection(vision_model.pooler_output) then L2, [512]', 'text': 'text_model.last_hidden_state, [77,512], no projection/normalization + binary attention_mask', 'tf32': False, 'tokenizer': {'class': 'CLIPTokenizer', 'max_length': 77, 'special_tokens': {'bos_token': '<|startoftext|>', 'eos_token': '<|endoftext|>', 'pad_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, 'vocab_sha256': 'f9190aead40a965a09db1ba2e556dccac0eda8423cb4926bcd994722899ec68e'}, 'truncation_policy': 'error_if_over_77_preserve_all_source_rows'}, 'implementation': 'visfit-adapter-development-cache-v1', 'runtime': {'device': 'cuda', 'gpu': 'Tesla T4', 'packages': {'numpy': '2.4.6', 'torch': '2.14.1+cu130', 'transformers': '4.57.1'}}, 'source_image_index_sha256': {'reference': 'b0391ce423bb89b335aec95d9b2c752bd146d6e79bb6f4d1319ccd6898e5ad4b', 'target': '71a80c213d06a0605451c354386a9d41614d4a8f281e52aaddaa167bc046c1fb'}, 'text': 'strict trained conventional last stream -> final LN, padded outputs zero; frozen trained projection applied later', 'text_encode_batch_size': 16, 'tf32': False, 'train_sha256': '465f758604249e2005cc58c0dfa79edf236081613e908c1e5942fd61abf17086', 'validation_sha256': 'ace52e94935adacfd19aeaea7a99314602954bce9bff43458ceb564fdf05a8b5'}, 'schema': 'legacy'}}, 43: {'source_verification': 'Actual Drive metadata and raw small artifacts read; no Drive mutations', 'encoder': {'encoder_checkpoint_sha256': None, 'epochs': 3, 'format': 'native', 'kind': 'encoder', 'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/development/seed_43/encoder/best.pt', 'phase': 'development', 'seed': 43, 'sha256': '7fbac15a8d3b83ae88cdd6bedd50fcf05076f620e1df27d76aa4218ac9b0a33f', 'signature': '74821c9bb15a88f3d95b1c20d9d56ca826ac1d8001240e3a06827bbaa0ef95f0'}, 'adapter': {'encoder_checkpoint_sha256': '7fbac15a8d3b83ae88cdd6bedd50fcf05076f620e1df27d76aa4218ac9b0a33f', 'epochs': 9, 'format': 'native', 'kind': 'adapter', 'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/development/seed_43/adapter/best.pt', 'phase': 'development', 'seed': 43, 'sha256': 'a8764653490de2b512152c3e3970072eb28c0e07e916b3e2fa58fd1feefbab19', 'signature': 'bbdee079c956a48b82867d124303b2566055be46d6f3f8a09552c4b3fd8068ff'}, 'expected_metrics': {'A': {'Mean': 71.43339047055386, 'gallery_counts': {'dress': 598, 'shirt': 599, 'toptee': 599}, 'n_query': 1796, 'overall': {'R@1': 23.994468670377795, 'R@10': 59.351706578969406, 'R@5': 46.93478353926927, 'R@50': 83.51507436213831}, 'per_category': {'dress': {'R@1': 18.06020066889632, 'R@10': 55.01672240802675, 'R@5': 41.80602006688963, 'R@50': 76.58862876254182}, 'shirt': {'R@1': 29.215358931552586, 'R@10': 59.26544240400668, 'R@5': 50.75125208681135, 'R@50': 84.47412353923205}, 'toptee': {'R@1': 24.707846410684475, 'R@10': 63.77295492487479, 'R@5': 48.24707846410685, 'R@50': 89.48247078464107}}, 'query_counts': {'dress': 598, 'shirt': 599, 'toptee': 599}, 'scope': 'development', 'unit': 'percent'}, 'B': {'Mean': 73.93895995369465, 'gallery_counts': {'dress': 598, 'shirt': 599, 'toptee': 599}, 'n_query': 1796, 'overall': {'R@1': 25.665220555254674, 'R@10': 62.24625583702306, 'R@5': 50.38600193931543, 'R@50': 85.63166407036626}, 'per_category': {'dress': {'R@1': 20.401337792642142, 'R@10': 56.52173913043478, 'R@5': 43.64548494983278, 'R@50': 80.10033444816054}, 'shirt': {'R@1': 30.217028380634392, 'R@10': 63.43906510851419, 'R@5': 53.756260434056756, 'R@50': 85.80968280467445}, 'toptee': {'R@1': 26.37729549248748, 'R@10': 66.7779632721202, 'R@5': 53.756260434056756, 'R@50': 90.98497495826378}}, 'query_counts': {'dress': 598, 'shirt': 599, 'toptee': 599}, 'scope': 'development', 'unit': 'percent'}}, 'tolerances': {'A_metric_reproduction_pp': 0.01, 'A_source_file': 'baseline_verification.json', 'adapter_gate_off_atol': 1e-06, 'B_rerun_tolerance_explicitly_defined_in_artifact': None, 'B_saved_prediction_metric_consistency_atol': 1e-10, 'B_note': 'No separate numerical rerun tolerance exists for B; source completion recomputes saved ranks with <1e-10 metric consistency. Do not invent relaxed B tolerance.'}, 'manifests': {'train': {'filename': 'development_train.csv', 'sha256': '465f758604249e2005cc58c0dfa79edf236081613e908c1e5942fd61abf17086', 'count': 16204}, 'validation': {'filename': 'development_val.csv', 'sha256': 'ace52e94935adacfd19aeaea7a99314602954bce9bff43458ceb564fdf05a8b5', 'count': 1796}, 'gallery': {'filename': 'development_gallery.csv', 'sha256': '83c08f14b606227c1a406b5074163ef70e42eb025144d19e2b1de4f38d234dc5', 'count': 1796}}, 'cache': {'root': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_development/FashionIQ/retrieval_only_dev_v1_base42_adapter42_checked_v1/cache', 'fingerprint': 'b6fbd67a1d203ced8626b25afd592cba578b196a2d9b9b6bff5e2ad0e875daff', 'index_sha256': 'c90c8f77acc6d48e7d3adc6df8deaa1cd56aec49b7b84f8ab2388a1bf881f841', 'index_drive_id': '1CkZxYsIOkA1RzPea8-MyGoNWXPMzLbq9', 'counts': {'eos': 18000, 'input_ids': 18000, 'mask': 18000, 'reference': 12525, 'reference_pooled': 12525, 'target_pooled': 17839, 'text': 18000}, 'contract': {'amp': False, 'attention_backend': 'eager', 'base_checkpoint_sha256': 'f62513032c086ce5c2e09a9774add5bc590e6ded81c5b9d92d0229cd526f080b', 'cache_shard_size': 128, 'caption_policy': 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1', 'dtype': 'float32', 'extractor_schema': 'trained-conventional-finalLN-final-image-v1', 'extractor_source_sha256': '82a37f5162dfd31ac5bbdd37bc903cf260ce8983c4788331399620274b1cac7b', 'feature_fingerprint': '1e86518f73f1640b6907474749b86e5d175b64c343b4de9f2f2e2e71af926037', 'gallery_sha256': '83c08f14b606227c1a406b5074163ef70e42eb025144d19e2b1de4f38d234dc5', 'image_extraction_contract': {'attention_backend': 'eager', 'autocast': False, 'batch_size': 32, 'caption_policy': 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1', 'cuda': '13.0', 'device': 'cuda', 'device_name': 'Tesla T4', 'dtype': 'float32', 'encoder_id': 'patrickjohncyh/fashion-clip', 'encoder_weight_sha256': '7b2e2766a4372167f9db95876b897dcecfe1ca5963f2bf620b3a8393f3ad6366', 'feature_shard_size': 256, 'implementation': 'fashioniq-preparation-v1.0.0', 'libraries': {'Pillow': '11.3.0', 'huggingface-hub': '0.36.0', 'numpy': '2.4.6', 'pandas': '2.2.3', 'tokenizers': '0.22.1', 'torch': '2.11.0+cu130', 'transformers': '4.57.1'}, 'processor': {'_processor_class': None, 'crop_size': {'height': 224, 'width': 224}, 'do_center_crop': True, 'do_convert_rgb': True, 'do_normalize': True, 'do_rescale': True, 'do_resize': True, 'image_mean': [0.48145466, 0.4578275, 0.40821073], 'image_processor_type': 'CLIPImageProcessor', 'image_std': [0.26862954, 0.26130258, 0.27577711], 'resample': 3, 'rescale_factor': 0.00392156862745098, 'size': {'shortest_edge': 224}}, 'reference': 'vision_model.last_hidden_state INCLUDING CLS; no projection or L2', 'repeat_acceptance': {'atol': 6.103515625e-05, 'note': 'Repeat same input and execution contract; not a historical feature acceptance threshold.', 'predeclared': True, 'rtol': 6.103515625e-05}, 'revision': '7e3ba62ce16b379a1ab479346b66f192e76f51b7', 'target': 'visual_projection(vision_model.pooler_output) then L2, [512]', 'text': 'text_model.last_hidden_state, [77,512], no projection/normalization + binary attention_mask', 'tf32': False, 'tokenizer': {'class': 'CLIPTokenizer', 'max_length': 77, 'special_tokens': {'bos_token': '<|startoftext|>', 'eos_token': '<|endoftext|>', 'pad_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, 'vocab_sha256': 'f9190aead40a965a09db1ba2e556dccac0eda8423cb4926bcd994722899ec68e'}, 'truncation_policy': 'error_if_over_77_preserve_all_source_rows'}, 'implementation': 'visfit-adapter-development-cache-v1', 'runtime': {'device': 'cuda', 'gpu': 'Tesla T4', 'packages': {'numpy': '2.4.6', 'torch': '2.14.1+cu130', 'transformers': '4.57.1'}}, 'source_image_index_sha256': {'reference': 'b0391ce423bb89b335aec95d9b2c752bd146d6e79bb6f4d1319ccd6898e5ad4b', 'target': '71a80c213d06a0605451c354386a9d41614d4a8f281e52aaddaa167bc046c1fb'}, 'text': 'strict trained conventional last stream -> final LN, padded outputs zero; frozen trained projection applied later', 'text_encode_batch_size': 16, 'tf32': False, 'train_sha256': '465f758604249e2005cc58c0dfa79edf236081613e908c1e5942fd61abf17086', 'validation_sha256': 'ace52e94935adacfd19aeaea7a99314602954bce9bff43458ceb564fdf05a8b5'}, 'shard_size': 128, 'role_shapes': {'eos': {'shape': [], 'dtype': 'int64'}, 'input_ids': {'shape': [77], 'dtype': 'int64'}, 'mask': {'shape': [77], 'dtype': 'int64'}, 'reference': {'shape': [50, 768], 'dtype': 'float32'}, 'reference_pooled': {'shape': [512], 'dtype': 'float32'}, 'target_pooled': {'shape': [512], 'dtype': 'float32'}, 'text': {'shape': [77, 512], 'dtype': 'float32'}}}, 'prediction_sha256': {'A': 'b0c154b47fd62610dfbebb5e04d440a33539882c280400c7a6d624b62db53881', 'B': '93864044d36e077cee4459f115d5dbf65405e53a26ef5ce6f501f1e8725248c9'}, 'actual_checkpoint_binary_verified': ['adapter'], 'checkpoint_binary_deferred_to_colab': ['encoder (~894MB)'], 'source_file_sha256': {'adapter_cache_audit.json': 'e5578b21ada12bab97966322929b0c3c0726cb1abe0150b85aff4be990ac7f2b', 'prep_feature_metadata.json': '48ee1866dc22f07fa588ea815318ddc9995cfa8734ffd0f2acc88c430f855342', 'development_full_cache_index.json': 'c90c8f77acc6d48e7d3adc6df8deaa1cd56aec49b7b84f8ab2388a1bf881f841', 'adapter_base_preflight.json': 'a0953b19940bcc5edc0aaec676a39277a7c168b5bc2c6713d5398168cec30368', 'adapter_best_result_verification.json': '5f2dd56bdd671c17db222c77966b93f7f0c39ff8d10af3dd3ccd7cd2c4809508', 'adapter_runtime_contract.json': '02696d2a36995ffd90791671203673b689bede949c5f05d92c20c4741e65ee2d', 'baseline_verification.json': '1ddd249b7d0674dae29e582a8383b7b7b96848994bc1b79be33213a8d2ab68a2', 'encoder_config.json': 'b3d48c8550a8f36ee1bbbc575878d10ac10e416509b8a599ba23019c7561935f', 'prep_protocol_config.json': '2fa2322dee2ebe9964e4ce63ebf0cd8dabad83392a39e288d0e43334e40faffb', 'encoder_best.json': '9ec6117b48e0df64e49247d9f41da1cc035c1a504e00551abc1aad654aadbca1', 'adapter_development_summary.json': '96da4b19138b4e2e687d1f8623a7411cfeedf0d6ba901a923639c2b829766139', 'adapter_locked_training_config.json': '4db8657b94ba13e5b0321af2b01986468f7e8949f7d2c539276cf8d84bf3ed3e', 'adapter_training_config.json': 'e8ccdc60d5e7c655a3a9009bb8b093a4e95d2d0074d18442a6ed3d77da5dfe5d', 'adapter_verified_trial_reference.json': 'dfaf7c892539869b9a001280e8379ac5c381635eddfa12b8e24a2cfc6a5cc833', 'full_development_seed42_result.json': '8300050811f23178f3e49562877452f94b23c8a2d01b8330d33d1afc98cbbbd1', 'adapter_full_cache_reuse.json': '3deb322a2d0679fb2ad5ca1bd2e87b0fe1b9c4c4e20baba920a1f01287e894b9', 'adapter_source_provenance.json': 'da564ddd55c8d82da1f1347c8ebd2948366ec050f797ec80bf228d0bdc02689f', 'encoder_metric_history.json': '253717bafb9c38d8dd1f0cf484d78b6165c2708233afec225bc7a905e3d8829e', 'adapter_experiment_config.json': '6a8a18859e4253cb67311d071caa15fe8c385bd48c8bffa9f9f9d68e7618901e', 'adapter_best.json': 'ee3be0b8e8ef22bac279a2de1584915314d3323088b5bca67d30ddc641c73e5c'}, 'warnings': ['Baseline verification records one rank difference, with identical aggregate metrics and 0.01pp configured source tolerance; do not require exact baseline rank equality without diagnosing that known evidence.', 'Trial2048 cache is not sufficient; checked_v1 full cache verified18000 query IDs and manifest alignment.', 'No official-val/final checkpoint is part of these source records.'], 'source_record': {'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/development/seed_43/result.json', 'sha256': 'cc0e5af0c2e9bfda54a5b40627d1ec422c383ddd396df036fc385946963fcb6b'}, 'prediction_paths': {'A': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/development/seed_43/encoder/per_query_results.csv', 'B': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/development/seed_43/adapter/per_query_results.csv'}, 'trained_text_cache': {'root': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/text_cache/13fc326ae07aee1689ee4cc8a445b206f3f407086e1ff2d6c1896e12f70e44f5', 'index_sha256': '4b9aa660aea5f125c6e5595fc493f927f81341c1bb16ab41752836f7f9cb9eb3', 'contract': {'cache_shard_size': 128, 'caption_policy': 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1', 'encode_batch_size': 16, 'encoder_checkpoint_sha256': '7fbac15a8d3b83ae88cdd6bedd50fcf05076f620e1df27d76aa4218ac9b0a33f', 'encoder_weights_sha256': '049523c1b55a00cc0d64295eae80af0f91010fed19c57005a740fa20e1e6409a', 'max_length': 77, 'phase': 'development', 'precision': 'float32', 'runtime': {'amp': False, 'attention': 'eager', 'cuda': '13.0', 'device': 'cuda', 'dtype': 'float32', 'gpu': 'Tesla T4', 'packages': {'Pillow': '11.3.0', 'huggingface-hub': '0.36.0', 'numpy': '2.4.6', 'pandas': '2.2.3', 'tokenizers': '0.22.1', 'torch': '2.14.1+cu130', 'transformers': '4.57.1'}, 'python': '3.13.16', 'tf32': False}, 'schema': 'trained-conventional-finalLN-padded-zero-v1', 'train_sha256': '465f758604249e2005cc58c0dfa79edf236081613e908c1e5942fd61abf17086', 'validation_sha256': 'ace52e94935adacfd19aeaea7a99314602954bce9bff43458ceb564fdf05a8b5'}, 'schema': 'native'}}, 44: {'source_verification': 'Actual Drive metadata and raw small artifacts read; no Drive mutations', 'encoder': {'encoder_checkpoint_sha256': None, 'epochs': 3, 'format': 'native', 'kind': 'encoder', 'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/development/seed_44/encoder/best.pt', 'phase': 'development', 'seed': 44, 'sha256': 'cee91e3ae1d586dfd1ef91a56f65c4b743e1e7ca673028a042f1afeb021709c5', 'signature': '36899841bd778cc25c7dd63e396adcb5c0075f8d3e2b9814b4aed15b41c342c8'}, 'adapter': {'encoder_checkpoint_sha256': 'cee91e3ae1d586dfd1ef91a56f65c4b743e1e7ca673028a042f1afeb021709c5', 'epochs': 9, 'format': 'native', 'kind': 'adapter', 'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/development/seed_44/adapter/best.pt', 'phase': 'development', 'seed': 44, 'sha256': '912b8c70b7b2008e56939e3f7fa7175125518bc8592e63c039775f7d39728668', 'signature': '93869666cf8e1d8a52826b68c6f2619908f65b2ab5be2a071faa51d0f693ae8a'}, 'expected_metrics': {'A': {'Mean': 72.49229019752354, 'gallery_counts': {'dress': 598, 'shirt': 599, 'toptee': 599}, 'n_query': 1796, 'overall': {'R@1': 24.66317887672319, 'R@10': 61.0225515212087, 'R@5': 49.94165303376307, 'R@50': 83.9620288738384}, 'per_category': {'dress': {'R@1': 19.732441471571907, 'R@10': 57.52508361204013, 'R@5': 45.1505016722408, 'R@50': 79.76588628762542}, 'shirt': {'R@1': 28.380634390651082, 'R@10': 61.43572621035058, 'R@5': 51.08514190317195, 'R@50': 83.63939899833055}, 'toptee': {'R@1': 25.87646076794658, 'R@10': 64.10684474123539, 'R@5': 53.58931552587646, 'R@50': 88.48080133555926}}, 'query_counts': {'dress': 598, 'shirt': 599, 'toptee': 599}, 'scope': 'development', 'unit': 'percent'}, 'B': {'Mean': 73.32659598029417, 'gallery_counts': {'dress': 598, 'shirt': 599, 'toptee': 599}, 'n_query': 1796, 'overall': {'R@1': 26.556244800419876, 'R@10': 61.96810738075164, 'R@5': 51.667308762467364, 'R@50': 84.6850845798367}, 'per_category': {'dress': {'R@1': 21.57190635451505, 'R@10': 56.68896321070234, 'R@5': 46.15384615384615, 'R@50': 79.09698996655518}, 'shirt': {'R@1': 30.05008347245409, 'R@10': 63.10517529215359, 'R@5': 54.09015025041736, 'R@50': 85.64273789649415}, 'toptee': {'R@1': 28.046744574290482, 'R@10': 66.110183639399, 'R@5': 54.757929883138566, 'R@50': 89.31552587646077}}, 'query_counts': {'dress': 598, 'shirt': 599, 'toptee': 599}, 'scope': 'development', 'unit': 'percent'}}, 'tolerances': {'A_metric_reproduction_pp': 0.01, 'A_source_file': 'baseline_verification.json', 'adapter_gate_off_atol': 1e-06, 'B_rerun_tolerance_explicitly_defined_in_artifact': None, 'B_saved_prediction_metric_consistency_atol': 1e-10, 'B_note': 'No separate numerical rerun tolerance exists for B; source completion recomputes saved ranks with <1e-10 metric consistency. Do not invent relaxed B tolerance.'}, 'manifests': {'train': {'filename': 'development_train.csv', 'sha256': '465f758604249e2005cc58c0dfa79edf236081613e908c1e5942fd61abf17086', 'count': 16204}, 'validation': {'filename': 'development_val.csv', 'sha256': 'ace52e94935adacfd19aeaea7a99314602954bce9bff43458ceb564fdf05a8b5', 'count': 1796}, 'gallery': {'filename': 'development_gallery.csv', 'sha256': '83c08f14b606227c1a406b5074163ef70e42eb025144d19e2b1de4f38d234dc5', 'count': 1796}}, 'cache': {'root': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_development/FashionIQ/retrieval_only_dev_v1_base42_adapter42_checked_v1/cache', 'fingerprint': 'b6fbd67a1d203ced8626b25afd592cba578b196a2d9b9b6bff5e2ad0e875daff', 'index_sha256': 'c90c8f77acc6d48e7d3adc6df8deaa1cd56aec49b7b84f8ab2388a1bf881f841', 'index_drive_id': '1CkZxYsIOkA1RzPea8-MyGoNWXPMzLbq9', 'counts': {'eos': 18000, 'input_ids': 18000, 'mask': 18000, 'reference': 12525, 'reference_pooled': 12525, 'target_pooled': 17839, 'text': 18000}, 'contract': {'amp': False, 'attention_backend': 'eager', 'base_checkpoint_sha256': 'f62513032c086ce5c2e09a9774add5bc590e6ded81c5b9d92d0229cd526f080b', 'cache_shard_size': 128, 'caption_policy': 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1', 'dtype': 'float32', 'extractor_schema': 'trained-conventional-finalLN-final-image-v1', 'extractor_source_sha256': '82a37f5162dfd31ac5bbdd37bc903cf260ce8983c4788331399620274b1cac7b', 'feature_fingerprint': '1e86518f73f1640b6907474749b86e5d175b64c343b4de9f2f2e2e71af926037', 'gallery_sha256': '83c08f14b606227c1a406b5074163ef70e42eb025144d19e2b1de4f38d234dc5', 'image_extraction_contract': {'attention_backend': 'eager', 'autocast': False, 'batch_size': 32, 'caption_policy': 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1', 'cuda': '13.0', 'device': 'cuda', 'device_name': 'Tesla T4', 'dtype': 'float32', 'encoder_id': 'patrickjohncyh/fashion-clip', 'encoder_weight_sha256': '7b2e2766a4372167f9db95876b897dcecfe1ca5963f2bf620b3a8393f3ad6366', 'feature_shard_size': 256, 'implementation': 'fashioniq-preparation-v1.0.0', 'libraries': {'Pillow': '11.3.0', 'huggingface-hub': '0.36.0', 'numpy': '2.4.6', 'pandas': '2.2.3', 'tokenizers': '0.22.1', 'torch': '2.11.0+cu130', 'transformers': '4.57.1'}, 'processor': {'_processor_class': None, 'crop_size': {'height': 224, 'width': 224}, 'do_center_crop': True, 'do_convert_rgb': True, 'do_normalize': True, 'do_rescale': True, 'do_resize': True, 'image_mean': [0.48145466, 0.4578275, 0.40821073], 'image_processor_type': 'CLIPImageProcessor', 'image_std': [0.26862954, 0.26130258, 0.27577711], 'resample': 3, 'rescale_factor': 0.00392156862745098, 'size': {'shortest_edge': 224}}, 'reference': 'vision_model.last_hidden_state INCLUDING CLS; no projection or L2', 'repeat_acceptance': {'atol': 6.103515625e-05, 'note': 'Repeat same input and execution contract; not a historical feature acceptance threshold.', 'predeclared': True, 'rtol': 6.103515625e-05}, 'revision': '7e3ba62ce16b379a1ab479346b66f192e76f51b7', 'target': 'visual_projection(vision_model.pooler_output) then L2, [512]', 'text': 'text_model.last_hidden_state, [77,512], no projection/normalization + binary attention_mask', 'tf32': False, 'tokenizer': {'class': 'CLIPTokenizer', 'max_length': 77, 'special_tokens': {'bos_token': '<|startoftext|>', 'eos_token': '<|endoftext|>', 'pad_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, 'vocab_sha256': 'f9190aead40a965a09db1ba2e556dccac0eda8423cb4926bcd994722899ec68e'}, 'truncation_policy': 'error_if_over_77_preserve_all_source_rows'}, 'implementation': 'visfit-adapter-development-cache-v1', 'runtime': {'device': 'cuda', 'gpu': 'Tesla T4', 'packages': {'numpy': '2.4.6', 'torch': '2.14.1+cu130', 'transformers': '4.57.1'}}, 'source_image_index_sha256': {'reference': 'b0391ce423bb89b335aec95d9b2c752bd146d6e79bb6f4d1319ccd6898e5ad4b', 'target': '71a80c213d06a0605451c354386a9d41614d4a8f281e52aaddaa167bc046c1fb'}, 'text': 'strict trained conventional last stream -> final LN, padded outputs zero; frozen trained projection applied later', 'text_encode_batch_size': 16, 'tf32': False, 'train_sha256': '465f758604249e2005cc58c0dfa79edf236081613e908c1e5942fd61abf17086', 'validation_sha256': 'ace52e94935adacfd19aeaea7a99314602954bce9bff43458ceb564fdf05a8b5'}, 'shard_size': 128, 'role_shapes': {'eos': {'shape': [], 'dtype': 'int64'}, 'input_ids': {'shape': [77], 'dtype': 'int64'}, 'mask': {'shape': [77], 'dtype': 'int64'}, 'reference': {'shape': [50, 768], 'dtype': 'float32'}, 'reference_pooled': {'shape': [512], 'dtype': 'float32'}, 'target_pooled': {'shape': [512], 'dtype': 'float32'}, 'text': {'shape': [77, 512], 'dtype': 'float32'}}}, 'prediction_sha256': {'A': 'a1ad7569437308e5caf561a2b67794d953179f0e2f179443565340219f1dc73d', 'B': '71ebc214cfda8416aef6087616fbaedb8fe2b53783eb2df53cf2f5962a5d7e48'}, 'actual_checkpoint_binary_verified': ['adapter'], 'checkpoint_binary_deferred_to_colab': ['encoder (~894MB)'], 'source_file_sha256': {'adapter_cache_audit.json': 'e5578b21ada12bab97966322929b0c3c0726cb1abe0150b85aff4be990ac7f2b', 'prep_feature_metadata.json': '48ee1866dc22f07fa588ea815318ddc9995cfa8734ffd0f2acc88c430f855342', 'development_full_cache_index.json': 'c90c8f77acc6d48e7d3adc6df8deaa1cd56aec49b7b84f8ab2388a1bf881f841', 'adapter_base_preflight.json': 'a0953b19940bcc5edc0aaec676a39277a7c168b5bc2c6713d5398168cec30368', 'adapter_best_result_verification.json': '5f2dd56bdd671c17db222c77966b93f7f0c39ff8d10af3dd3ccd7cd2c4809508', 'adapter_runtime_contract.json': '02696d2a36995ffd90791671203673b689bede949c5f05d92c20c4741e65ee2d', 'baseline_verification.json': '1ddd249b7d0674dae29e582a8383b7b7b96848994bc1b79be33213a8d2ab68a2', 'encoder_config.json': 'b3d48c8550a8f36ee1bbbc575878d10ac10e416509b8a599ba23019c7561935f', 'prep_protocol_config.json': '2fa2322dee2ebe9964e4ce63ebf0cd8dabad83392a39e288d0e43334e40faffb', 'encoder_best.json': '9ec6117b48e0df64e49247d9f41da1cc035c1a504e00551abc1aad654aadbca1', 'adapter_development_summary.json': '96da4b19138b4e2e687d1f8623a7411cfeedf0d6ba901a923639c2b829766139', 'adapter_locked_training_config.json': '4db8657b94ba13e5b0321af2b01986468f7e8949f7d2c539276cf8d84bf3ed3e', 'adapter_training_config.json': 'e8ccdc60d5e7c655a3a9009bb8b093a4e95d2d0074d18442a6ed3d77da5dfe5d', 'adapter_verified_trial_reference.json': 'dfaf7c892539869b9a001280e8379ac5c381635eddfa12b8e24a2cfc6a5cc833', 'full_development_seed42_result.json': '8300050811f23178f3e49562877452f94b23c8a2d01b8330d33d1afc98cbbbd1', 'adapter_full_cache_reuse.json': '3deb322a2d0679fb2ad5ca1bd2e87b0fe1b9c4c4e20baba920a1f01287e894b9', 'adapter_source_provenance.json': 'da564ddd55c8d82da1f1347c8ebd2948366ec050f797ec80bf228d0bdc02689f', 'encoder_metric_history.json': '253717bafb9c38d8dd1f0cf484d78b6165c2708233afec225bc7a905e3d8829e', 'adapter_experiment_config.json': '6a8a18859e4253cb67311d071caa15fe8c385bd48c8bffa9f9f9d68e7618901e', 'adapter_best.json': 'ee3be0b8e8ef22bac279a2de1584915314d3323088b5bca67d30ddc641c73e5c'}, 'warnings': ['Baseline verification records one rank difference, with identical aggregate metrics and 0.01pp configured source tolerance; do not require exact baseline rank equality without diagnosing that known evidence.', 'Trial2048 cache is not sufficient; checked_v1 full cache verified18000 query IDs and manifest alignment.', 'No official-val/final checkpoint is part of these source records.'], 'source_record': {'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/development/seed_44/result.json', 'sha256': '2b27cf9c0e3e5475f745730e8979cb4baffd2a739f5281b18ae4fc1f12ac5639'}, 'prediction_paths': {'A': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/development/seed_44/encoder/per_query_results.csv', 'B': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/development/seed_44/adapter/per_query_results.csv'}, 'trained_text_cache': {'root': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/text_cache/e43c962bdf59e96983326d143a7c0231812eaf138947a0e431c3229a996c3663', 'index_sha256': 'e1a95584d7b3f709514427fe0811b53b0260332769fc89d211189cb8a4ac11e7', 'contract': {'cache_shard_size': 128, 'caption_policy': 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1', 'encode_batch_size': 16, 'encoder_checkpoint_sha256': 'cee91e3ae1d586dfd1ef91a56f65c4b743e1e7ca673028a042f1afeb021709c5', 'encoder_weights_sha256': '0832743747747b89857ed111f008b2f54cbd5d326808008810c8c8460188ec04', 'max_length': 77, 'phase': 'development', 'precision': 'float32', 'runtime': {'amp': False, 'attention': 'eager', 'cuda': '13.0', 'device': 'cuda', 'dtype': 'float32', 'gpu': 'Tesla T4', 'packages': {'Pillow': '11.3.0', 'huggingface-hub': '0.36.0', 'numpy': '2.4.6', 'pandas': '2.2.3', 'tokenizers': '0.22.1', 'torch': '2.14.1+cu130', 'transformers': '4.57.1'}, 'python': '3.13.16', 'tf32': False}, 'schema': 'trained-conventional-finalLN-padded-zero-v1', 'train_sha256': '465f758604249e2005cc58c0dfa79edf236081613e908c1e5942fd61abf17086', 'validation_sha256': 'ace52e94935adacfd19aeaea7a99314602954bce9bff43458ceb564fdf05a8b5'}, 'schema': 'native'}}}



IMPLEMENTATION_SHA = 'a14c96979f29c29ca76b20855ffeba04e46e8cbb1b585c76227e479574aa9dce'


In [ ]:
import os,json,hashlib,random,time,copy,traceback,uuid,shutil,platform,tempfile,contextlib,importlib.metadata
from pathlib import Path
from collections import OrderedDict
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from transformers import CLIPModel,CLIPTokenizer,CLIPImageProcessor

def require(condition, message):
    if not condition: raise RuntimeError(message)

def canonical(x):
    return json.dumps(x, ensure_ascii=False, sort_keys=True, separators=(',', ':'), allow_nan=False).encode()

def fingerprint(x): return hashlib.sha256(canonical(x)).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(8*1024*1024), b''): h.update(block)
    return h.hexdigest()

def read_json(path): return json.loads(Path(path).read_text())

def write_bytes(path, raw, immutable=False):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    if immutable and path.exists():
        require(path.read_bytes() == raw, 'Existing immutable artifact differs: '+str(path)); return
    tmp = path.with_name(path.name+'.'+uuid.uuid4().hex+'.partial')
    tmp.write_bytes(raw); os.replace(tmp, path)

def write_json(path, value, immutable=False):
    write_bytes(path, canonical(value)+b'\n', immutable)

def write_csv(path, frame, immutable=False):
    write_bytes(path, frame.to_csv(index=False, lineterminator='\n').encode(), immutable)

def safe_child(root, relative):
    root = Path(root).resolve(); path = (root/relative).resolve()
    require(path != root and root in path.parents, 'Path escapes root'); return path

def tensor_hash(module):
    h = hashlib.sha256()
    for name, value in sorted(module.state_dict().items()):
        h.update(name.encode()); h.update(str(value.dtype).encode())
        h.update(value.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

def finite(x, name):
    require(bool(torch.isfinite(x).all()), 'NaN/Inf: '+name); return x

def unit(x):
    finite(x, 'unnormalized vector')
    require(bool((x.float().norm(dim=-1)>1e-8).all()), 'Zero retrieval vector')
    return F.normalize(x.float(), dim=-1)

def normalize_caption(x): return x.strip('.?, ')

def caption_text(a, b): return normalize_caption(a).capitalize()+' and '+normalize_caption(b)

def assess_captions(caps):
    if not isinstance(caps, list) or len(caps)!=2 or not all(isinstance(x,str) for x in caps):
        return 'schema_error', None
    content = [bool(normalize_caption(x)) and not normalize_caption(x).isspace() for x in caps]
    if not any(content): return 'both_empty_captions', None
    return None, 'single_empty_caption' if not all(content) else None

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def rng_state():
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
            'cuda':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}

def restore_rng(s):
    random.setstate(s['python']); np.random.set_state(s['numpy']); torch.set_rng_state(s['torch'])
    if s['cuda'] is not None:
        require(torch.cuda.is_available(), 'Resume CUDA checkpoint on CPU refused')
        torch.cuda.set_rng_state_all(s['cuda'])

def epoch_batches(frame,batch,seed,epoch):
    ids=np.asarray(sorted(frame.query_id),dtype=object);order=np.random.default_rng(seed+epoch).permutation(ids)
    require(len(ids)>=2 and batch>=2,'Contrastive batch too small')
    batches=[order[i:i+batch].tolist() for i in range(0,len(ids),batch)]
    if len(batches)>1 and len(batches[-1])==1:batches[-2].extend(batches.pop())
    return batches

def exact_ranks(query,gallery,target_positions):
    q=np.asarray(query,dtype=np.float32); g=np.asarray(gallery,dtype=np.float32)
    require(q.ndim==g.ndim==2 and q.shape[1]==g.shape[1] and len(q)>0 and len(g)>0,'Ranking shapes invalid')
    require(np.isfinite(q).all() and np.isfinite(g).all(),'Nonfinite ranking input')
    require((np.linalg.norm(q,axis=-1)>1e-8).all() and (np.linalg.norm(g,axis=-1)>1e-8).all(),'Zero ranking input')
    q=q/np.linalg.norm(q,axis=-1,keepdims=True);g=g/np.linalg.norm(g,axis=-1,keepdims=True)
    targets=np.asarray(target_positions,dtype=np.int64)
    require(targets.shape==(len(q),) and (targets>=0).all() and (targets<len(g)).all(),'Target gallery alignment invalid')
    scores=q@g.T; ts=scores[np.arange(len(q)),targets]
    ranks=1+(scores>ts[:,None]).sum(1)+((scores==ts[:,None])&(np.arange(len(g))[None,:]<targets[:,None])).sum(1)
    return ranks

def macro_metrics(per_category):
    require(set(per_category)==set(CATEGORIES),'Macro requires three categories')
    overall={f'R@{k}':float(np.mean([per_category[c][f'R@{k}'] for c in CATEGORIES])) for k in (1,5,10,50)}
    return {'overall':overall,'Mean':(overall['R@10']+overall['R@50'])/2}

@contextlib.contextmanager
def isolated_rng():
    saved=rng_state()
    try:yield
    finally:restore_rng(saved)

def same_rng(a,b):
    cpu=a['python']==b['python'] and np.array_equal(a['numpy'][1],b['numpy'][1]) and a['numpy'][2:]==b['numpy'][2:] and torch.equal(a['torch'],b['torch'])
    cuda=(a['cuda'] is None and b['cuda'] is None) or (a['cuda'] is not None and b['cuda'] is not None and len(a['cuda'])==len(b['cuda']) and all(torch.equal(x,y) for x,y in zip(a['cuda'],b['cuda'])))
    return cpu and cuda

def cpu_tree(x):
    if torch.is_tensor(x):return x.detach().cpu().clone()
    if isinstance(x,dict):return {k:cpu_tree(v) for k,v in x.items()}
    if isinstance(x,list):return [cpu_tree(v) for v in x]
    if isinstance(x,tuple):return tuple(cpu_tree(v) for v in x)
    return copy.deepcopy(x)

def frame_sha(frame):return hashlib.sha256(frame.to_csv(index=False,lineterminator='\n').encode()).hexdigest()

def csv_frame(path):return pd.read_csv(path,dtype=str,keep_default_na=False)

def physical_runtime():
    return {'packages':{k:importlib.metadata.version(k) for k in ('torch','transformers','tokenizers','huggingface-hub','numpy','pandas','Pillow')},'python':platform.python_version(),'cuda':torch.version.cuda,'device':str(DEVICE),'gpu':torch.cuda.get_device_name() if DEVICE.type=='cuda' else None,'dtype':'float32','attention':'eager','amp':False,'tf32':False}

def eos_positions(ids, mask, eos_token_id):
    require(ids.shape==mask.shape and bool(((mask==0)|(mask==1)).all()), 'Token/mask shape or values invalid')
    lengths=mask.sum(-1)
    require(bool((lengths>=2).all()), 'Empty text token sequence')
    expected=torch.arange(ids.shape[1],device=ids.device)[None,:]<lengths[:,None]
    require(torch.equal(expected,mask.bool()), 'Tokenizer must right-pad contiguous sequences')
    # pad_token == EOS for CLIP; intersect with attention_mask and select first real EOS.
    is_eos=(ids==eos_token_id)&mask.bool()
    require(bool(is_eos.any(-1).all()), 'EOS absent from unpadded tokens')
    pos=is_eos.long().argmax(-1)
    require(bool((pos==lengths-1).all()), 'EOS must close each untruncated modifier')
    return pos

def text_masks(mask, dtype):
    b,n=mask.shape; neg=torch.finfo(dtype).min
    padding=torch.zeros((b,1,n,n),device=mask.device,dtype=dtype).masked_fill(~mask[:,None,None,:].bool(),neg)
    causal=torch.full((n,n),neg,dtype=dtype,device=mask.device).triu(1)
    return padding,causal[None,None].expand(b,1,n,n)

class ResidualAdapter(nn.Module):
    def __init__(self,projection_weight,text_dim=512,vision_dim=768,width=128,heads=4):
        super().__init__();require(width%heads==0,'Adapter width/head mismatch')
        self.register_buffer('projection_weight',projection_weight.detach().clone())
        self.text_norm=nn.LayerNorm(text_dim);self.text_down=nn.Linear(text_dim,width)
        self.reference_norm=nn.LayerNorm(vision_dim);self.reference_down=nn.Linear(vision_dim,width)
        self.cross_attention=nn.MultiheadAttention(width,heads,dropout=0.,batch_first=True)
        self.ri_mlp=nn.Sequential(nn.Linear(width,width),nn.GELU(),nn.Linear(width,text_dim))
        self.ta_mlp=nn.Sequential(nn.Linear(text_dim,width),nn.GELU(),nn.Linear(width,text_dim))
        self.gate_ri=nn.Parameter(torch.zeros(()));self.gate_ta=nn.Parameter(torch.zeros(()))
    def forward(self,reference_features,text_features,mask,eos,return_aux=False,gates_off=False):
        require(text_features.ndim==3 and mask.shape==text_features.shape[:2] and eos.shape==(len(text_features),),'Adapter input shape')
        require(bool(((mask==0)|(mask==1)).all()) and bool((eos==mask.sum(-1)-1).all()),'Mask/EOS invalid')
        require(bool((eos>=1).all()) and bool((mask.sum(-1)<=mask.shape[1]).all()),'Invalid sequence length')
        require(torch.equal(mask.bool(),torch.arange(mask.shape[1],device=mask.device)[None,:]<mask.sum(-1)[:,None]),'Mask must be contiguous right padding')
        valid=mask.unsqueeze(-1).to(text_features.dtype)
        x=self.text_norm(text_features);q=self.text_down(x);kv=self.reference_down(self.reference_norm(reference_features['tokens']))
        ri=self.cross_attention(q,kv,kv,need_weights=False)[0];ri=self.ri_mlp(ri)*valid
        ta=self.ta_mlp(x)*valid
        gri=self.gate_ri.new_zeros(()) if gates_off else torch.tanh(self.gate_ri)
        gta=self.gate_ta.new_zeros(()) if gates_off else torch.tanh(self.gate_ta)
        enhanced=(text_features+gri*ri+gta*ta)*valid # no LN/MLP on the base stream
        index=torch.arange(len(eos),device=eos.device)
        text=unit(F.linear(enhanced[index,eos],self.projection_weight))
        query=unit(text+unit(reference_features['pooled'])) # reference composed exactly once
        if return_aux:
            student=F.linear((text_features+ta)[index,eos],self.projection_weight)
            return query,student,enhanced
        return query
    def encode_query(self,reference_features,text_features,mask,eos):
        return self(reference_features,text_features,mask,eos)
    def enhanced_text_sequence(self,reference_features,text_features,mask,eos):
        _,_,h=self(reference_features,text_features,mask,eos,return_aux=True)
        return h,mask,eos

def positive_mask(target_ids,device):
    require(target_ids and all(isinstance(v,str) and v for v in target_ids),'Invalid target IDs')
    return torch.tensor([[a==b for b in target_ids] for a in target_ids],dtype=torch.bool,device=device)

def retrieval_loss(query,target,target_ids,temperature):
    require(temperature>0 and query.shape==target.shape,'InfoNCE shape/temperature')
    logits=unit(query)@unit(target.detach()).T/temperature;p=positive_mask(target_ids,query.device)
    def direction(x,m):return -((F.log_softmax(x,dim=1)*m).sum(1)/m.sum(1)).mean()
    return (direction(logits,p)+direction(logits.T,p.T))/2

class ConventionalOnly(nn.Module):
    """Exactly the source conventional path; no RI/TA/full-depth modules are created."""
    def __init__(self,text,projection,eos_id):
        super().__init__();self.text=copy.deepcopy(text);self.projection=copy.deepcopy(projection);self.eos_id=int(eos_id)
        self.requires_grad_(True);self.text.embeddings.requires_grad_(False)
    def encode_text(self,modifier):
        ids,mask=modifier['input_ids'],modifier['attention_mask'];eos=eos_positions(ids,mask,self.eos_id)
        h=self.text.embeddings(input_ids=ids);padding,causal=text_masks(mask,h.dtype);valid=mask.unsqueeze(-1).to(h.dtype)
        for block in self.text.encoder.layers:h=block(h,attention_mask=padding,causal_attention_mask=causal)[0]*valid
        return self.text.final_layer_norm(h)*valid,mask,eos
    def encode_query(self,reference,modifier):
        h,mask,eos=self.encode_text(modifier);text=self.projection(h[torch.arange(len(h),device=h.device),eos])
        return unit(unit(text)+unit(reference['pooled']))
    def freeze(self):self.requires_grad_(False);self.eval();return self

class CachedConventional(nn.Module):
    def __init__(self,projection):super().__init__();self.register_buffer('projection_weight',projection.detach().clone())
    def forward(self,ref,text,mask,eos):
        return unit(unit(F.linear(text[torch.arange(len(eos),device=eos.device),eos],self.projection_weight))+unit(ref['pooled']))
    def encode_query(self,*args):return self(*args)

def load_pretrained(package):
    c=package.contract
    with isolated_rng():
        base=CLIPModel.from_pretrained(c['encoder_id'],revision=c['revision'],dtype=torch.float32,attn_implementation='eager')
        require(tensor_hash(base)==c['encoder_weight_sha256'],'Pinned pretrained weights differ')
        tokenizer=CLIPTokenizer.from_pretrained(c['encoder_id'],revision=c['revision']);processor=CLIPImageProcessor.from_pretrained(c['encoder_id'],revision=c['revision'])
        require(fingerprint(tokenizer.get_vocab())==c['tokenizer']['vocab_sha256'] and tokenizer.special_tokens_map==c['tokenizer']['special_tokens'] and tokenizer.padding_side=='right','Tokenizer contract differs')
        require(all(processor.to_dict().get(k)==v for k,v in c['processor'].items() if k not in ('_processor_class','image_processor_type')),'Preprocessing contract differs')
        base.float().to(DEVICE).requires_grad_(False).eval()
    return base,tokenizer,processor

def fresh_encoder(base,tokenizer,seed):
    with isolated_rng():
        seed_all(seed);model=ConventionalOnly(base.text_model,base.text_projection,tokenizer.eos_token_id).float().to(DEVICE)
    return model

def fresh_adapter(encoder,seed):
    with isolated_rng():
        seed_all(seed);model=ResidualAdapter(encoder.projection.weight,width=CFG['adapter']['width'],heads=CFG['adapter']['heads']).float().to(DEVICE)
    require(sum(p.numel() for p in model.parameters() if p.requires_grad)==446978,'Residual adapter architecture/parameter count differs')
    return model

class TrialStore:
    def __init__(self,root,local,contract,specs,shard_size):
        self.key=fingerprint(contract);self.root=Path(root)/self.key;self.local=Path(local)/self.key
        self.root.mkdir(parents=True,exist_ok=True);self.local.mkdir(parents=True,exist_ok=True)
        self.specs=specs;self.contract=contract;self.step=shard_size;self.maps={k:{v:i for i,v in enumerate(s['ids'])} for k,s in specs.items()}
        require(all(len(self.maps[k])==len(s['ids']) for k,s in specs.items()),'Duplicate cached IDs')
        self.index={'fingerprint':self.key,'contract':contract,'specs':specs,'shard_size':shard_size}
        write_json(self.root/'index.json',self.index,immutable=True);self.loaded=OrderedDict();self.paths={}
    def prepare(self,role,compute):
        spec=self.specs[role];ids=spec['ids'];dtype=np.dtype(spec['dtype']);tail=spec['shape']
        (self.root/role).mkdir(exist_ok=True);(self.local/role).mkdir(exist_ok=True)
        for start in range(0,len(ids),self.step):
            end=min(start+self.step,len(ids));name=f'{start:07d}_{end:07d}';p=self.root/role/(name+'.npy');marker=p.with_suffix('.json');pending=p.with_suffix('.pending.json')
            expected={'fingerprint':self.key,'role':role,'ids_sha256':fingerprint(ids[start:end]),'shape':[end-start,*tail],'dtype':spec['dtype']}
            if not marker.is_file() and p.is_file() and pending.is_file():
                m=read_json(pending);require(all(m.get(k)==v for k,v in expected.items()) and file_hash(p)==m['sha256'],'Pending cache corrupt');write_json(marker,m,immutable=True)
            local=self.local/role/p.name
            if marker.is_file():
                m=read_json(marker);require(all(m.get(k)==v for k,v in expected.items()),'Cache fingerprint/ID/shape differs')
                require(p.is_file() and file_hash(p)==m['sha256'],'Committed Drive cache corrupt')
                if not local.is_file() or file_hash(local)!=m['sha256']:
                    temp=local.with_suffix('.copying');stream_copy(p,temp);require(file_hash(temp)==m['sha256'],'Local cache copy corrupt');os.replace(temp,local)
            else:
                require(not p.exists(),'Uncommitted orphan cache; preserve and investigate')
                a=np.asarray(compute(ids[start:end]),dtype=dtype)
                require(list(a.shape)==expected['shape'] and np.isfinite(a).all(),'Computed cache shape/finite')
                temp=local.with_suffix('.partial');
                with temp.open('wb') as f:np.save(f,a,allow_pickle=False)
                m={**expected,'sha256':file_hash(temp)};write_json(pending,m)
                remote=p.with_suffix('.copying');stream_copy(temp,remote);require(file_hash(remote)==m['sha256'],'Cache sync corrupt')
                os.replace(remote,p);write_json(marker,m,immutable=True);os.replace(temp,local)
            a=np.load(local,mmap_mode='r',allow_pickle=False)
            require(list(a.shape)==expected['shape'] and a.dtype==dtype and np.isfinite(a).all(),'Local cache invalid')
            if role in ('reference_pooled','target_pooled'):require(np.allclose(np.linalg.norm(a,axis=-1),1,atol=1e-5,rtol=1e-5),'Cached embedding not unit')
            self.paths[role,start]=local
    def take(self,role,ids):
        # Training hot path: prevalidated local mmap, no Drive/JSON/checksum access.
        result=[]
        for image_id in ids:
            row=self.maps[role][image_id];start=row//self.step*self.step;key=(role,start)
            require(key in self.paths,'Run cache preflight before use')
            if key not in self.loaded:
                if len(self.loaded)>=16:self.loaded.popitem(last=False)
                self.loaded[key]=np.load(self.paths[key],mmap_mode='r',allow_pickle=False)
            self.loaded.move_to_end(key);result.append(np.asarray(self.loaded[key][row-start]).copy())
        return np.stack(result)

def checkpoint_save(path,state,signature):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    require(state['signature']==signature,'Checkpoint signature differs')
    with tempfile.NamedTemporaryFile(dir=path.parent,suffix='.pt',delete=False) as f:temp=Path(f.name)
    try:
        torch.save(state,temp);meta={'signature':signature,'sha256':file_hash(temp),'phase':state.get('phase'),'step':state.get('step')}
        write_json(path.with_suffix('.pending.json'),meta);os.replace(temp,path);write_json(path.with_suffix('.json'),meta)
    finally:
        if temp.exists():temp.unlink()

def checkpoint_load(path,signature):
    path=Path(path);marker=path.with_suffix('.json');pending=path.with_suffix('.pending.json');sha=file_hash(path)
    if pending.is_file():
        p=read_json(pending)
        if p.get('sha256')==sha:
            require(p['signature']==signature,'Pending checkpoint signature')
            if not marker.is_file() or read_json(marker)!=p:write_json(marker,p)
    m=read_json(marker);require(m['signature']==signature and m['sha256']==sha,'Checkpoint checksum/config mismatch')
    state=torch.load(path,map_location='cpu',weights_only=False)
    require(state['signature']==signature,'Checkpoint payload identity differs');return state

def sync_checkpoint(local,destination,signature):
    destination=Path(destination);destination.parent.mkdir(parents=True,exist_ok=True)
    marker=read_json(Path(local).with_suffix('.json'));require(marker['signature']==signature,'Checkpoint sync signature')
    write_json(destination.with_suffix('.pending.json'),marker)
    temp=destination.with_suffix('.copying');stream_copy(local,temp);require(file_hash(temp)==marker['sha256'],'Drive checkpoint sync corrupt')
    os.replace(temp,destination);write_json(destination.with_suffix('.json'),marker)

def committed_checkpoint_load(path,signature):
    # Protected/complete artifacts are read only. Never repair their pending/commit markers.
    path=Path(path);m=read_json(path.with_suffix('.json'))
    require(m['signature']==signature and m['sha256']==file_hash(path),'Committed source checkpoint checksum/signature mismatch')
    state=torch.load(path,map_location='cpu',weights_only=False)
    require(state['signature']==signature,'Committed source checkpoint payload signature differs');return state

class Preparation:
    def __init__(self,root):
        self.root=Path(root);self.frames={};self.hashes={};self.views={}
        for name,sha in PREPARATION_ANCHORS.items():
            require(file_hash(safe_child(self.root,name))==sha,'Preparation anchor changed: '+name)
        self.protocol=read_json(self.root/'protocol_config.json');self.contract=read_json(self.root/'extraction_contract.json');self.meta=read_json(self.root/'feature_metadata.json')
        audit=read_json(self.root/'data_audit.json')
        require(file_hash(self.root/'feature_audit.json')==self.meta['feature_audit_sha256'],'Preparation feature audit checksum differs')
        feature_audit=read_json(self.root/'feature_audit.json');require(feature_audit['passed'] and not feature_audit['partial'] and feature_audit['fingerprint']==self.meta['fingerprint'],'Incomplete feature extraction audit')
        require(self.contract==EXTRACTION_CONTRACT and self.meta['contract']==self.contract and self.meta['status']=='complete' and not self.meta['legacy_arrays_mixed'],'Preparation/extraction contract incomplete or differs')
        require(audit['passed'] and not audit['partial_mode'] and audit['schema_error_count']==audit['both_empty_captions_count']==audit['missing_or_corrupt_image_count']==0,'Preparation data/image audit blocked')
        require(self.protocol['caption_policy']==CAPTION_POLICY and self.protocol['reference_policy']=='retain in gallery (FashionIQ)' and self.protocol['actual_counts']==audit['counts'],'Locked annotation/protocol counts differ')
        self.binding={'anchors':PREPARATION_ANCHORS,'feature_fingerprint':self.meta['fingerprint'],'protocol_sha256':file_hash(self.root/'protocol_config.json'),'extraction_sha256':file_hash(self.root/'extraction_contract.json')}
        for name in ('development_train','development_val','development_gallery','official_train_queries'):self.read(name)
        train,val,full=[self.frames[x] for x in ('development_train','development_val','official_train_queries')]
        require(len(train)==16204 and len(val)==1796 and len(full)==18000,'Locked development/final training counts differ')
        require(set(train.query_id).isdisjoint(val.query_id) and set(train.target_image_id).isdisjoint(val.target_image_id),'Development target/query leakage')
        joined=pd.concat([train,val],ignore_index=True).sort_values('query_id').reset_index(drop=True)
        require(joined[full.columns].equals(full.sort_values('query_id').reset_index(drop=True)),'Full official-train != exact union of development partitions by query ID')
    def read(self,name):
        if name in self.frames:return self.frames[name]
        path=safe_child(self.root,name+'.csv');require(file_hash(path)==PREPARATION_ANCHORS[name+'.csv'],'Manifest SHA differs: '+name)
        f=csv_frame(path);view_spec=self.meta['feature_views'][name];view_path=safe_child(self.root,view_spec['path'])
        require(file_hash(view_path)==view_spec['sha256'],'Feature view SHA differs: '+name)
        view=csv_frame(view_path);require(len(f)==view_spec['rows'] and f.equals(view[f.columns]),'Feature view row/ID alignment differs')
        if 'query_id' in f:
            split='val' if name=='official_val_queries' else 'train'
            require(f.query_id.is_unique and set(f.official_split)=={split},'Query ID or official split invalid')
            warnings=[]
            for row in f.itertuples():
                require(row.category in CATEGORIES and row.source_file==f'cap.{row.category}.{split}.json' and row.query_id==f'{row.category}:{split}:{int(row.source_row):06d}','Annotation/source mapping invalid: '+row.query_id)
                for image_id in (row.reference_image_id,row.target_image_id):require(bool(image_id) and image_id==image_id.strip() and '/' not in image_id and '\\' not in image_id,'Invalid image ID')
                error,warning=assess_captions([row.caption_1,row.caption_2]);require(error is None,'Caption blocked: '+row.query_id)
                require(row.modification_text==caption_text(row.caption_1,row.caption_2) and row.triplet_key_sha256==fingerprint([row.category,row.reference_image_id,row.target_image_id,row.caption_1,row.caption_2]),'Caption policy/triplet changed')
                if warning:warnings.append({'query_id':row.query_id,'reason':warning})
            if name.startswith('official_'):
                counts=self.protocol['actual_counts']['official_val' if split=='val' else 'official_train']
                require(f.groupby('category').size().to_dict()==counts,'Official annotation counts differ')
                for c in CATEGORIES:require(sorted(f[f.category==c].source_row.astype(int))==list(range(counts[c])),'Official annotation rows missing/duplicated')
            write_json(ROOT/'data_audit'/(name+'_warnings.json'),warnings)
        else:
            f['gallery_index']=f.gallery_index.astype(int)
            for c in CATEGORIES:
                g=f[f.category==c];require(g.image_id.is_unique and g.gallery_index.tolist()==list(range(len(g))),'Gallery order/IDs differ')
                if name=='official_val_gallery':require(fingerprint(g.image_id.tolist())==self.protocol['id_order_sha256']['official_val_gallery'][c],'Official image_splits order differs')
        self.frames[name]=f;self.views[name]=view;self.hashes[name]=file_hash(path);return f
    def data(self,phase):
        require(phase=='development','Only development is allowed in this pilot')
        if phase=='development':names=('development_train','development_val','development_gallery')
        elif phase=='final':names=('official_train_queries',None,None)
        else:names=(None,'official_val_queries','official_val_gallery')
        train,val,gallery=[self.read(x) if x else None for x in names]
        for name in names:
            if name:require(frame_sha(self.frames[name])==PREPARATION_ANCHORS[name+'.csv'],'In-memory manifest changed after validation: '+name)
        if val is not None:
            for c in CATEGORIES:require(set(val[val.category==c].target_image_id)<=set(gallery[gallery.category==c].image_id),'Target coverage missing in category gallery')
        return {'phase':phase,'train':train,'validation':val,'gallery':gallery,'train_sha256':frame_sha(train) if train is not None else None,'validation_sha256':frame_sha(val) if val is not None else None,'gallery_sha256':frame_sha(gallery) if gallery is not None else None,'package':self.binding,'views':{role:self.views[name] for role,name in zip(('train','validation','gallery'),names) if name}}
    def image_source(self):
        root=self.root/'features'/self.meta['fingerprint']
        return {'prep':self.root,'meta':self.meta,'mapping_hashes':{k:file_hash(root/self.meta['mapping_files'][k]) for k in ('reference','target')}}

def query_forward(model,batch,kind):
    if kind=='encoder':return model.encode_query(batch['reference_features'],batch['modifier'])
    return model.encode_query(batch['reference_features'],batch['text_features'],batch['mask'],batch['eos'])

@torch.inference_mode()
def retrieval_evaluate(model,data,images,batch_fn,kind,top_k=50):
    require(data['validation'] is not None and data['gallery'] is not None,'Evaluation manifest required')
    model.eval();per={};queries=[];top=[]
    for c in CATEGORIES:
        f=data['validation'][data['validation'].category==c];g=data['gallery'][data['gallery'].category==c].sort_values('gallery_index',kind='stable')
        require(len(f)>0 and g.gallery_index.tolist()==list(range(len(g))) and g.image_id.is_unique,'Gallery/category alignment invalid')
        gids=g.image_id.tolist();lookup={v:i for i,v in enumerate(gids)};vectors=images.take('target_pooled',gids);vectors=vectors/np.linalg.norm(vectors,axis=-1,keepdims=True)
        ranks=[]
        for start in range(0,len(f),CFG['eval_batch_size']):
            rows=f.iloc[start:start+CFG['eval_batch_size']];b=batch_fn(rows,False);require('target' not in b,'Target entered query inference')
            q=query_forward(model,b,kind).cpu().numpy();q=q/np.linalg.norm(q,axis=-1,keepdims=True)
            positions=np.array([lookup[x] for x in rows.target_image_id]);scores=q@vectors.T;require(np.isfinite(scores).all(),'Nonfinite scores')
            ts=scores[np.arange(len(rows)),positions];r=1+(scores>ts[:,None]).sum(1)+((scores==ts[:,None])&(np.arange(len(gids))[None,:]<positions[:,None])).sum(1)
            ranks.extend(r.tolist());orders=np.argsort(-scores,axis=1,kind='stable')[:,:min(top_k,len(gids))]
            for i,row in enumerate(rows.itertuples()):
                queries.append({'query_id':row.query_id,'category':c,'reference_image_id':row.reference_image_id,'target_image_id':row.target_image_id,'target_rank':int(r[i])})
                top.extend({'query_id':row.query_id,'category':c,'rank':j+1,'image_id':gids[pos],'score':float(scores[i,pos])} for j,pos in enumerate(orders[i]))
        per[c]={f'R@{k}':float((np.asarray(ranks)<=k).mean()*100) for k in (1,5,10,50)}
    require(len(queries)==len(data['validation']) and len({q['query_id'] for q in queries})==len(queries),'Evaluation query coverage differs')
    return {'scope':data['phase'],'n_query':len(queries),'query_counts':data['validation'].groupby('category').size().to_dict(),'gallery_counts':data['gallery'].groupby('category').size().to_dict(),'unit':'percent','per_category':per,**macro_metrics(per)},pd.DataFrame(queries),pd.DataFrame(top)

def metrics_from_predictions(queries,data):
    f=data['validation'];g=data['gallery'];keys=['query_id','category','reference_image_id','target_image_id']
    require(len(queries)==len(f) and queries.query_id.is_unique and set(queries.query_id)==set(f.query_id),'Prediction query coverage')
    require(len(queries.merge(f[keys],on=keys,validate='one_to_one'))==len(f),'Prediction source ID mapping differs')
    per={}
    for c in CATEGORIES:
        r=queries[queries.category==c].target_rank.to_numpy(dtype=np.float64);ng=len(g[g.category==c])
        require(len(r)==int((f.category==c).sum()) and np.isfinite(r).all() and np.all(r==r.astype(int)) and np.all((r>=1)&(r<=ng)),'Saved ranks invalid')
        per[c]={f'R@{k}':float((r<=k).mean()*100) for k in (1,5,10,50)}
    return {'scope':data['phase'],'n_query':len(f),'query_counts':f.groupby('category').size().to_dict(),'gallery_counts':g.groupby('category').size().to_dict(),'unit':'percent','per_category':per,**macro_metrics(per)}

def metric_equal(a,b):
    require(a['n_query']==b['n_query'] and all(abs(a['per_category'][c][f'R@{k}']-b['per_category'][c][f'R@{k}'])<1e-10 for c in CATEGORIES for k in (1,5,10,50)) and abs(a['Mean']-b['Mean'])<1e-10,'Saved metric mismatch')
    for k in (1,5,10,50):require(abs(a['overall'][f'R@{k}']-b['overall'][f'R@{k}'])<1e-10,'Saved macro mismatch')

def verify_topk(top,queries,data,top_k):
    require(set(top.query_id)==set(queries.query_id) and not top.duplicated(['query_id','rank']).any() and not top.duplicated(['query_id','image_id']).any(),'TopK ID/duplicate coverage')
    require(len(top.merge(queries[['query_id','category']],on=['query_id','category'],validate='many_to_one'))==len(top),'TopK category mapping')
    require(np.isfinite(top.score).all(),'TopK NaN/Inf')
    for c in CATEGORIES:
        g=data['gallery'][data['gallery'].category==c];p=top[top.category==c];n=int((queries.category==c).sum());k=min(top_k,len(g))
        require(len(p)==n*k and set(p.image_id)<=set(g.image_id),'TopK gallery/count differs')
        require(p.groupby('query_id')['rank'].apply(lambda r:sorted(r)==list(range(1,k+1))).all(),'TopK rank coverage differs')
        ordered=p.sort_values(['query_id','rank'],kind='stable');same=ordered.query_id.eq(ordered.query_id.shift());previous=ordered.score.shift()
        require((ordered.loc[same,'score']<=previous[same]).all(),'TopK scores are not descending')
        gallery_order=dict(zip(g.image_id,g.gallery_index));indices=ordered.image_id.map(gallery_order);ties=same & ordered.score.eq(previous)
        require((indices[ties]>indices.shift()[ties]).all(),'TopK exact ties differ from locked gallery order')
        matched=p.merge(queries[['query_id','target_image_id','target_rank']],on='query_id',validate='many_to_one');hits=matched[matched.image_id==matched.target_image_id]
        expected=queries[(queries.category==c)&(queries.target_rank<=k)]
        require(set(hits.query_id)==set(expected.query_id) and (hits['rank']==hits.target_rank).all(),'TopK target positions disagree with saved ranks')

def export_evaluation(folder,model,data,images,batch_fn,kind):
    folder=Path(folder);metric,q,top=retrieval_evaluate(model,data,images,batch_fn,kind,CFG['export_top_k'])
    write_csv(folder/'per_query_results.csv',q);write_csv(folder/'top_k_results.csv',top)
    per=[{'category':c,**m,'Mean':(m['R@10']+m['R@50'])/2} for c,m in metric['per_category'].items()]+[{'category':'Overall macro',**metric['overall'],'Mean':metric['Mean']}]
    write_csv(folder/'per_category_metrics.csv',pd.DataFrame(per));write_json(folder/'metrics.json',metric)
    saved=pd.read_csv(folder/'per_query_results.csv',dtype={'query_id':str});savedtop=pd.read_csv(folder/'top_k_results.csv',dtype={'query_id':str,'image_id':str})
    metric_equal(metrics_from_predictions(saved,data),metric);verify_topk(savedtop,saved,data,CFG['export_top_k'])
    return metric,{n:file_hash(folder/n) for n in ('per_query_results.csv','top_k_results.csv','per_category_metrics.csv','metrics.json')}

def required_checkpoint(state,binding):
    keys={'signature','binding','phase','seed','kind','model','optimizer','scheduler','scaler','rng','epoch','cursor','step','history','metrics','finished','best_epoch','best_mean','bad_epochs','timing','initial_sha256'}
    require(keys<=state.keys(),'Incomplete checkpoint: '+','.join(sorted(keys-state.keys())))
    require(state['binding']==binding and state['signature']==fingerprint(binding),'Resume config/code/data/runtime mismatch')
    require(state['scheduler'] is None and state['scaler'] is None and {'state','param_groups'}<=state['optimizer'].keys() and {'python','numpy','torch','cuda'}<=state['rng'].keys(),'Missing optimizer/RNG or incompatible scheduler/scaler')
    require(len(state['history'])==state['step'] and state['phase']==binding['phase'] and state['seed']==binding['seed'] and state['kind']==binding['kind'],'Checkpoint progress/phase mismatch')
    require(state['epoch']>=0 and state['cursor']>=0 and state['step']>=0,'Negative checkpoint cursor')

def optimize_stage(model,frame,batch_fn,eval_fn,folder,local,binding,hp,budget,patience=None):
    """Shared production runner. Tests call this core with temporary models/frames/folders only."""
    folder,local=Path(folder),Path(local);folder.mkdir(parents=True,exist_ok=True);local.mkdir(parents=True,exist_ok=True)
    signature=fingerprint(binding);write_json(folder/'training_binding.json',{'signature':signature,'binding':binding},immutable=True)
    initial=tensor_hash(model);params=[p for p in model.parameters() if p.requires_grad];require(params,'No trainable parameters')
    opt=torch.optim.AdamW(params,lr=hp['learning_rate'],weight_decay=hp['weight_decay'],betas=(.9,.999),eps=1e-8)
    state={'signature':signature,'binding':binding,'phase':binding['phase'],'seed':binding['seed'],'kind':binding['kind'],'epoch':0,'cursor':0,'step':0,'history':[],'metrics':[],'finished':False,'stop_reason':None,'best_epoch':None,'best_mean':None,'bad_epochs':0,'initial_sha256':initial,'scheduler':None,'scaler':None,'timing':{'compute_seconds':0.,'training_wall_seconds':0.,'validation_seconds':0.,'checkpoint_seconds':0.},'best_snapshot':None}
    candidates=[]
    for path in (local/'last.pt',folder/'last.pt'):
        if path.exists():
            candidate=checkpoint_load(path,signature);required_checkpoint(candidate,binding);require(candidate['initial_sha256']==initial,'Resume initializer mismatch');candidates.append(candidate)
    if candidates:
        state=max(candidates,key=lambda x:(x['step'],x['epoch'],len(x['metrics']),x['finished']));model.load_state_dict(state['model'],strict=True);opt.load_state_dict(state['optimizer']);restore_rng(state['rng'])
    else:seed_all(binding['seed'])
    del candidates
    def persist(improved=False):
        started=time.perf_counter();state.update(model=cpu_tree(model.state_dict()),optimizer=cpu_tree(opt.state_dict()),rng=rng_state())
        if improved:state['best_snapshot']=cpu_tree({k:v for k,v in state.items() if k!='best_snapshot'})
        checkpoint_save(local/'last.pt',state,signature);sync_checkpoint(local/'last.pt',folder/'last.pt',signature)
        if state.get('best_snapshot') is not None:
            best=state['best_snapshot'];expected=read_json(folder/'best.json') if (folder/'best.json').exists() else None
            if expected is None or expected.get('step')!=best['step'] or not (folder/'best.pt').is_file():
                checkpoint_save(local/'best.pt',best,signature);sync_checkpoint(local/'best.pt',folder/'best.pt',signature)
        if state['finished'] and binding['phase']=='final':sync_checkpoint(local/'last.pt',folder/'final.pt',signature)
        state['timing']['checkpoint_seconds']+=time.perf_counter()-started
        write_json(folder/'status.json',{k:state[k] for k in ('phase','kind','seed','epoch','cursor','step','finished','stop_reason','best_epoch','best_mean','timing','initial_sha256')})
        write_csv(folder/'training_history.csv',pd.DataFrame(state['history']));write_json(folder/'metric_history.json',state['metrics'])
    indexed=frame.set_index('query_id',drop=False)
    try:
        for epoch in range(state['epoch'],budget):
            if state['finished']:break
            order=epoch_batches(frame,hp['batch_size'],binding['seed'],epoch);require(state['cursor']<=len(order),'Resume cursor beyond epoch')
            for cursor in range(state['cursor'],len(order)):
                wall=time.perf_counter();ids=order[cursor];rows=indexed.loc[ids];require(rows.query_id.tolist()==ids and set(rows.official_split)=={'train'},'Training query order/split differs')
                batch=batch_fn(rows,True);model.train();opt.zero_grad(set_to_none=True)
                if next(model.parameters()).is_cuda:torch.cuda.synchronize()
                t=time.perf_counter();query=query_forward(model,batch,binding['kind']);loss=retrieval_loss(query,batch['target'],batch['target_ids'],hp['retrieval_temperature']);finite(loss,'training loss');loss.backward()
                require(all(p.grad is not None for p in params),'Trainable branch missing gradient')
                grad=torch.nn.utils.clip_grad_norm_(params,hp['gradient_clip'],error_if_nonfinite=True);opt.step()
                require(bool(torch.stack([torch.isfinite(p).all() for p in params]).all()),'Optimizer produced nonfinite weight')
                if next(model.parameters()).is_cuda:torch.cuda.synchronize()
                compute=time.perf_counter()-t;elapsed=time.perf_counter()-wall
                values=torch.stack([loss.detach(),grad.detach()]).cpu().tolist();state['step']+=1;state['cursor']=cursor+1
                state['timing']['compute_seconds']+=compute;state['timing']['training_wall_seconds']+=elapsed
                state['history'].append({'epoch':epoch+1,'step':state['step'],'cursor':cursor+1,'batch_ids_sha256':fingerprint(ids),'loss_retrieval':values[0],'gradient_norm':values[1],'compute_seconds':compute,'training_wall_seconds':elapsed})
                if state['step']%hp['checkpoint_every_steps']==0:persist()
            improved=False
            if eval_fn is not None:
                model.eval();t=time.perf_counter();metric,pred=eval_fn(model);seconds=time.perf_counter()-t;state['timing']['validation_seconds']+=seconds
                require(metric['scope']=='development','Only internal development validation may choose epochs')
                improved=state['best_mean'] is None or metric['Mean']>state['best_mean']
                if improved:state.update(best_epoch=epoch+1,best_mean=metric['Mean'],bad_epochs=0)
                else:state['bad_epochs']+=1
                write_csv(folder/f'epoch_{epoch+1:02d}_per_query.csv',pred)
                state['metrics'].append({'epoch':epoch+1,'metrics':metric,'validation_seconds':seconds,'predictions_sha256':file_hash(folder/f'epoch_{epoch+1:02d}_per_query.csv')})
            require(binding['phase']=='development' or (eval_fn is None and patience is None),'Final training must not evaluate/early-stop on official-val')
            done=(epoch+1>=budget or (patience is not None and state['bad_epochs']>=patience))
            state.update(epoch=epoch+1,cursor=0,finished=done,stop_reason=('patience' if patience is not None and state['bad_epochs']>=patience else 'fixed_budget') if done else None)
            persist(improved);print(binding['phase'],binding['kind'],'seed',binding['seed'],'epoch',epoch+1,'step',state['step'],'Mean',state['best_mean'])
        # A lost best upload can be reconstructed from the authoritative last snapshot without optimizer updates.
        if state['finished'] and not (folder/('final.pt' if binding['phase']=='final' else 'best.pt')).exists():persist()
        return {k:state[k] for k in ('finished','epoch','step','best_epoch','best_mean','metrics','timing','initial_sha256','stop_reason')}
    except BaseException:
        write_bytes(folder/'training_traceback.txt',traceback.format_exc().encode());raise
    finally:opt.zero_grad(set_to_none=True)

def validate_asset(record):
    require(record['format'] in ('native','legacy_conventional_v1','legacy_adapter_v2'),'Unknown checkpoint schema')
    path=Path(record['path']);m=read_json(path.with_suffix('.json'));require(m['sha256']==record['sha256'] and m['signature']==record['signature'] and file_hash(path)==record['sha256'],'Checkpoint marker/content changed: '+str(path))
    return committed_checkpoint_load(path,record['signature'])

def load_encoder_asset(record,base,tokenizer):
    require(record['kind']=='encoder','An adapter checkpoint cannot be used as encoder')
    state=validate_asset(record)
    if record['format']=='legacy_conventional_v1':
        require(record['seed']==42 and record['phase']=='development' and state['phase']=='conventional' and state['config']['seed']==42 and state['epoch']==3 and state['finished'],'Legacy encoder phase/config differs')
        roots={k.split('.')[0] for k in state['model']};require(roots=={'text','projection','align','inject','merge','align_norm','align_projection'},'Unknown legacy encoder module roots')
        weights={k:v for k,v in state['model'].items() if k.startswith(('text.','projection.'))}
    else:
        required_checkpoint(state,state['binding']);require(state['kind']=='encoder' and state['phase']==record['phase'] and state['seed']==record['seed'] and state['binding']['origin']=='locked_pretrained_fresh','Encoder phase/seed/origin mismatch')
        require(state['binding']['initial_pretrained_sha256']==EXTRACTION_CONTRACT['encoder_weight_sha256'],'Final/development encoder did not start from locked pretrained')
        weights=state['model']
    model=fresh_encoder(base,tokenizer,record['seed']);model.load_state_dict(weights,strict=True);model.freeze()
    if record['format']=='legacy_conventional_v1':require(tensor_hash(model)==LEGACY['encoder_text_hash'],'Extracted legacy conventional weights differ')
    del state;return model

def load_adapter_asset(record,encoder):
    require(record['kind']=='adapter' and all(not p.requires_grad for p in encoder.parameters()),'Adapter expects frozen trained encoder')
    state=validate_asset(record)
    if record['format']=='legacy_adapter_v2':
        require(state['binding']['base_checkpoint']==LEGACY['encoder_best']['sha256'] and state['binding']['base_seed']==state['binding']['adapter_seed']==record['seed']==42,'Legacy adapter encoder/seed mismatch')
    else:
        required_checkpoint(state,state['binding']);require(state['phase']==record['phase'] and state['kind']=='adapter' and state['seed']==record['seed'] and state['binding']['encoder_asset']['sha256']==record['encoder_checkpoint_sha256'],'Adapter final/development provenance mismatch')
    model=fresh_adapter(encoder,record['seed']);model.load_state_dict(state['model'],strict=True)
    require(torch.equal(model.projection_weight,encoder.projection.weight.detach()),'Adapter frozen projection belongs to another encoder')
    model.eval();del state;return model

def verify_receipts(receipts):
    for name,sha in receipts.items():require(file_hash(Path(name))==sha,'Artifact changed or missing: '+name)
class ReadonlyCache:
    def __init__(self,root,role,ids,local_root,kind=False):
        self.root=Path(root);self.role=role;self.index=read_json(self.root/'index.json');self.key=self.index['fingerprint'];self.kind=kind
        if kind:
            require(self.index['kind']==role,'Feature kind mismatch');self.ids=self.index['ids'];self.tail=self.index['shape_tail'];self.dtype=self.index['dtype'];self.step=self.index['shard_size']
            require(read_json(self.root.parent/(role+'_id2idx.json'))=={v:i for i,v in enumerate(self.ids)},'Feature row mapping mismatch')
        else:
            require(fingerprint(self.index['contract'])==self.key and self.root.name==self.key,'Cache contract fingerprint mismatch')
            spec=self.index['specs'][role];self.ids=spec['ids'];self.tail=spec['shape'];self.dtype=spec['dtype'];self.step=self.index['shard_size']
        self.map={v:i for i,v in enumerate(self.ids)};require(len(self.map)==len(self.ids) and set(ids)<=set(self.map),'Missing/duplicate feature IDs')
        self.paths={};self.loaded=OrderedDict();self.local=Path(local_root)/self.key/role;self.local.mkdir(parents=True,exist_ok=True)
        needed=sorted({self.map[v]//self.step*self.step for v in ids})
        for start in needed:
            end=min(start+self.step,len(self.ids));name=f'{start:07d}_{end:07d}';folder=self.root if kind else self.root/role;p=folder/(name+'.npy');marker=read_json(p.with_suffix('.json'))
            expected={'fingerprint':self.key,'kind' if kind else 'role':role,'ids_sha256':fingerprint(self.ids[start:end]),'shape':[end-start,*self.tail],'dtype':self.dtype}
            require(all(marker.get(k)==v for k,v in expected.items()),'Committed shard identity mismatch: '+str(p))
            require(file_hash(p)==marker['sha256'],'Source shard corrupt: '+str(p));dest=self.local/p.name
            if not dest.exists() or file_hash(dest)!=marker['sha256']:
                tmp=dest.with_suffix('.partial')
                with p.open('rb') as src,tmp.open('wb') as dst:shutil.copyfileobj(src,dst,8*1024*1024)
                require(file_hash(tmp)==marker['sha256'],'Local copy corrupt');os.replace(tmp,dest)
            a=np.load(dest,mmap_mode='r',allow_pickle=False);require(list(a.shape)==expected['shape'] and str(a.dtype)==self.dtype and np.isfinite(a).all(),'Invalid shard values/header')
            if role in ('target','target_pooled','reference_pooled'):require(np.allclose(np.linalg.norm(a,axis=-1),1,atol=1e-5,rtol=1e-5),'Non-unit image embedding')
            if role=='reference':require((np.linalg.norm(a.reshape(len(a),-1),axis=-1)>1e-8).all(),'Zero reference feature')
            if role=='mask':require(np.isin(a,[0,1]).all(),'Invalid mask')
            self.paths[start]=dest
    def take(self,ids):
        values=[]
        for v in ids:
            row=self.map[v];start=row//self.step*self.step;require(start in self.paths,'ID not preflighted')
            if start not in self.loaded:
                if len(self.loaded)>=16:self.loaded.popitem(last=False)
                self.loaded[start]=np.load(self.paths[start],mmap_mode='r',allow_pickle=False)
            self.loaded.move_to_end(start);values.append(np.array(self.loaded[start][row-start],copy=True))
        return np.stack(values)

## Luồng tensor và biến thể MR
Nguồn: H[77,512] từ encoder Conventional đã train → Adapter RI/TA đóng băng (D) hoặc giữ H (C) → unit(projection(EOS)) + ảnh CLS pooled, rồi L2.

Pilot: frozen post-LN + visual projection ánh xạ50 reference token768→512, không L2 từng token. MR nguồn hidden256/alpha1 nhận masked mean H sau Adapter(D), trước Adapter(C). `delta = mean(MR(V,t) − V)`; `q = unit(text_embedding + unit(reference_pooled + delta))`. **Ảnh được cộng đúng một lần**. Tắt MR hoặc gate khởi tạo identity tái hiện A/B. Aggregation residual token-mean là phần thay đổi so với XAA/DVR nguồn; không thêm head học mới, chỉ262.913 tham số MR.

C/D có cùng init MR, batch order theo CFG[seed] và ngân sách3 epoch, batch32, FP32, AdamW LR1e-4/WD0.01/clip1, constant LR, symmetric multi-positive InfoNCE target-ID/T0.07. Đánh giá mỗi epoch; best Macro Mean, hòa giữ epoch sớm nhất. Giữ đủ mọi train row (batch cuối12), không dùng drop_last của DVR cũ.


In [ ]:
class ModifierGuidedRefinement(nn.Module):
    def __init__(self, dim=512, hidden=DVR_HIDDEN, alpha=DVR_ALPHA):
        super().__init__()
        self.alpha = float(alpha)
        self.v_proj = nn.Linear(dim, hidden)       # W_v
        self.t_proj = nn.Linear(dim, hidden)       # W_t
        self.act = nn.GELU()
        self.out = nn.Linear(hidden, 1)            # gate head
        nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)   # identity tại khởi tạo

    def forward(self, v, text_hidden, text_mask):
        # v: [B,N,D] | text_hidden: [B,L,D] | text_mask: [B,L] (1 = token thật)
        m = text_mask.unsqueeze(-1).to(text_hidden.dtype)
        t = (text_hidden * m).sum(1) / m.sum(1).clamp(min=1.0)                  # masked mean, bỏ padding   [B,D]
        h = self.act(self.v_proj(v) + self.t_proj(t).unsqueeze(1))              # [B,N,hidden]
        gate = 1.0 + self.alpha * torch.tanh(self.out(h).squeeze(-1))          # [B,N]  ∈ (1-α, 1+α)
        return v * gate.unsqueeze(-1), gate

def validate_frozen_text(text,mask,eos):
    require(text.ndim==3 and text.shape[-1]==512 and text.shape[1]==77,'Text must be [B,77,512]')
    require(text.dtype==torch.float32 and mask.dtype==eos.dtype==torch.int64,'FP32 text / int64 mask and EOS required')
    require(mask.shape==text.shape[:2] and eos.shape==(len(text),),'Mask/EOS shape mismatch')
    require(bool(((mask==0)|(mask==1)).all()),'Mask must be binary')
    lengths=mask.sum(-1)
    require(bool((lengths>=2).all()) and torch.equal(eos,lengths-1),'EOS must end untruncated text')
    require(torch.equal(mask.bool(),torch.arange(mask.shape[1],device=mask.device)[None,:]<lengths[:,None]),'Mask must right-pad contiguously')
    finite(text,'frozen text')
    require(torch.count_nonzero(text[mask==0]).item()==0,'Frozen text padding must be zero')
    return text,mask,eos

@torch.no_grad()
def reduce_frozen_text(text,mask,eos,projection_weight):
    """Reduced immutable input, not a new encoder. No target parameter is accepted."""
    validate_frozen_text(text,mask,eos)
    require(projection_weight.shape==(512,512) and projection_weight.dtype==torch.float32,'Frozen text projection schema')
    t=unit(F.linear(text[torch.arange(len(text),device=text.device),eos],projection_weight))
    m=mask.unsqueeze(-1).to(text.dtype);guidance=(text*m).sum(1)/m.sum(1).clamp(min=1.0)
    return {'text_embedding':t.detach(),'guidance':finite(guidance,'MR text guidance').detach()}

@torch.no_grad()
def project_reference_tokens(tokens,vision_post_layernorm,visual_projection):
    """Use frozen Fashion-CLIP modules from the verified preparation revision.
    Source pooled reference is unit(project(postLN(CLS))). Per-token vectors stay
    unnormalized: MR receives the frozen projected tokens; normalization occurs
    only after combining the reference residual with the existing CLS vector.
    """
    require(tokens.ndim==3 and tokens.shape[1:]==(50,768) and tokens.dtype==torch.float32,'Reference tokens must be FP32 [B,50,768]')
    require(not vision_post_layernorm.training and not visual_projection.training,'Frozen vision modules must be eval')
    require(all(not p.requires_grad for m in (vision_post_layernorm,visual_projection) for p in m.parameters()),'Frozen vision modules unexpectedly trainable')
    output=visual_projection(vision_post_layernorm(finite(tokens,'raw reference tokens')))
    require(output.shape==(len(tokens),50,512) and output.dtype==torch.float32,'Projected MR reference schema')
    return finite(output,'projected reference tokens').detach()

class ResidualMRCore(nn.Module):
    """Only MR is trainable. Inputs are checkpoint-bound immutable cache rows."""
    def __init__(self,hidden=256,alpha=1.0):
        super().__init__();self.mr=ModifierGuidedRefinement(512,hidden,alpha)
    def forward(self,projected_tokens,pooled,text_embedding,guidance,mr_off=False):
        b=len(projected_tokens)
        require(projected_tokens.shape==(b,50,512) and pooled.shape==text_embedding.shape==guidance.shape==(b,512),'Cached MR input shape mismatch')
        require(all(x.dtype==torch.float32 for x in (projected_tokens,pooled,text_embedding,guidance)),'Cached MR inputs must be FP32')
        require(len({x.device for x in (projected_tokens,pooled,text_embedding,guidance)})==1,'Cached MR input devices differ')
        for name,x in [('projected_tokens',projected_tokens),('pooled',pooled),('text_embedding',text_embedding),('guidance',guidance)]:finite(x,name)
        require(bool(torch.allclose(text_embedding.norm(dim=-1),torch.ones(b,device=text_embedding.device),rtol=1e-5,atol=1e-5)),'Cached text embedding not unit')
        v=projected_tokens.detach();base=pooled.detach();text=text_embedding.detach();guide=guidance.detach()
        if mr_off:delta=torch.zeros_like(base)
        else:
            gated,_=self.mr(v,guide[:,None,:],torch.ones((b,1),dtype=torch.int64,device=guide.device))
            delta=(gated-v).mean(1)
        return unit(text+unit(base+delta))
    def encode_query(self,reference_features,text_embedding,guidance,mr_off=False):
        return self(reference_features['mr_tokens'],reference_features['pooled'],text_embedding,guidance,mr_off)

class ResidualMRQuery(nn.Module):
    """Native full-sequence wrapper for extraction/preflight parity with cached core.
    C passes trained conventional hidden states. D passes enhanced hidden states
    from the already-trained frozen Adapter. No Adapter/encoder weights train.
    """
    def __init__(self,projection_weight,hidden=256,alpha=1.0):
        super().__init__();self.register_buffer('projection_weight',projection_weight.detach().clone())
        self.core=ResidualMRCore(hidden,alpha)
    def forward(self,reference_features,text,mask,eos,mr_off=False):
        reduced=reduce_frozen_text(text,mask,eos,self.projection_weight)
        return self.core(reference_features['mr_tokens'],reference_features['pooled'],**reduced,mr_off=mr_off)
    def encode_query(self,*args,**kwargs):return self(*args,**kwargs)

@torch.no_grad()
def frozen_adapter_text(adapter,reference_features,text,mask,eos):
    require(not adapter.training and all(not p.requires_grad for p in adapter.parameters()),'Adapter must be frozen/eval')
    validate_frozen_text(text,mask,eos)
    enhanced,returned_mask,returned_eos=adapter.enhanced_text_sequence(reference_features,text,mask,eos)
    require(torch.equal(returned_mask,mask) and torch.equal(returned_eos,eos),'Adapter changed mask/EOS')
    validate_frozen_text(enhanced,mask,eos)
    return enhanced.detach()

In [ ]:

def configure_seed(seed, sources):
    global SOURCE
    require(seed in (42,43,44) and seed in sources,'Unsupported development seed')
    SOURCE=copy.deepcopy(sources[seed])
    require(SOURCE['encoder']['seed']==SOURCE['adapter']['seed']==seed and SOURCE['encoder']['phase']==SOURCE['adapter']['phase']=='development','Source seed/phase mismatch')
    CFG.update(seed=seed,campaign_id=f'adapter_mr_pilot_v1_seed{seed}')

def active_source_locks():
    if 'source_record' in SOURCE:return {SOURCE['source_record']['path']:SOURCE['source_record']['sha256']}
    return SOURCE_FILE_LOCKS

# Development-only pilot. All writes are restricted to this new campaign/local cache.
PREFLIGHT = None
def stream_copy(source,destination):
    with Path(source).open('rb') as a,Path(destination).open('wb') as b:shutil.copyfileobj(a,b,8*1024*1024)

def science_config():
    return {k:v for k,v in CFG.items() if k not in ('RUN_PILOT',)}

def initialize_pilot():
    global ROOT,LOCAL,DEVICE,PREFLIGHT
    PREFLIGHT=None
    require(CFG['seed'] in (42,43,44) and SOURCE['encoder']['seed']==SOURCE['adapter']['seed']==CFG['seed'] and CFG['epochs']==3 and CFG['hp']==PILOT_HP,'Preserve the declared pilot recipe')
    require(CFG['eval_batch_size']==32 and CFG['adapter']=={'width':128,'heads':4},'Preserve source evaluator/Adapter architecture')
    require(torch.cuda.is_available(),'Colab GPU required for real-source preflight; no real training on local/CPU')
    DEVICE=torch.device('cuda');torch.set_default_dtype(torch.float32);torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False;torch.backends.cudnn.benchmark=False
    require(not torch.is_autocast_enabled(),'FP32 only; disable inherited autocast')
    ROOT=Path(CFG['output_root'])/CFG['campaign_id'];LOCAL=Path(CFG['local_root'])/CFG['campaign_id']
    protected=[Path(CFG[k]).resolve() for k in ('preparation_root','legacy_conventional_root','legacy_adapter_root')]+[Path(SOURCE['cache']['root']).resolve(),Path(SOURCE['encoder']['path']).parent.resolve(),Path(SOURCE['adapter']['path']).parent.resolve()]
    for destination in (ROOT.resolve(),LOCAL.resolve()):
        require(all(destination!=p and destination not in p.parents and p not in destination.parents for p in protected),'Pilot output/cache overlaps protected source')
    require(ROOT.resolve()!=LOCAL.resolve() and ROOT.resolve() not in LOCAL.resolve().parents and LOCAL.resolve() not in ROOT.resolve().parents,'Local cache and Drive output overlap')
    require(not (ROOT/'D/completion.json').exists() or CFG['seed']==44,'Completed historical seed is read-only; use aggregate_pilots() instead of rerunning it')
    ROOT.mkdir(parents=True,exist_ok=True);LOCAL.mkdir(parents=True,exist_ok=True)
    require(shutil.disk_usage(LOCAL).free>=CFG['min_local_free_gib']*1024**3,'Insufficient local disk; stop, do not reduce data/batch size')
    # Actual Drive quota, not the host filesystem size reported by DriveFS.
    from google.colab import auth
    import google.auth
    from google.auth.transport.requests import AuthorizedSession
    auth.authenticate_user()
    credentials,_=google.auth.default()
    response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',params={'fields':'storageQuota'},timeout=60)
    require(response.ok,'Cannot verify Drive quota; complete account authorization then rerun')
    quota=response.json()['storageQuota'];remaining=int(quota['limit'])-int(quota['usage']) if quota.get('limit') else None
    require(remaining is None or remaining>CFG['min_drive_free_gib']*1024**3,'Drive quota insufficient for pilot cache/checkpoints; preserve sources and stop')
    write_json(ROOT/'config.json',science_config(),immutable=True)
    write_json(ROOT/'source_provenance.json',SOURCE,immutable=True)
    write_json(ROOT/'implementation.json',{'implementation_sha256':IMPLEMENTATION_SHA,'source_notebooks':SOURCE_NOTEBOOKS},immutable=True)
    write_json(LOCAL/'resource_report.json',{'device':physical_runtime(),'local_free_bytes':shutil.disk_usage(LOCAL).free,'drive_storage_quota':quota})

class PilotPreparation(Preparation):
    def read(self,name):
        require(name in ('development_train','development_val','development_gallery','official_train_queries'),'Pilot prohibits official-val/final data access')
        return super().read(name)
    def data(self,phase):
        require(phase=='development','Pilot never switches to final/official-val')
        return super().data(phase)

class SourceInputs:
    def __init__(self,package,data,encoder,adapter,base,tokenizer):
        self.data=data;self.encoder=encoder;self.adapter=adapter;self.base=base;self.tokenizer=tokenizer
        root=Path(SOURCE['cache']['root'])/SOURCE['cache']['fingerprint']
        require(file_hash(root/'index.json')==SOURCE['cache']['index_sha256'],'Verified source cache index changed')
        index=read_json(root/'index.json');require(index['contract']==SOURCE['cache']['contract'],'Source cache extraction contract changed')
        ct=index['contract'];require(ct['image_extraction_contract']==package.contract,'Text/image encoder contract mismatch')
        for k in ('train_sha256','validation_sha256','gallery_sha256'):require(ct[k]==data[k],'Source cache manifests differ: '+k)
        frame=pd.concat([data['train'],data['validation']],ignore_index=True);qids=sorted(frame.query_id);refs=sorted(set(frame.reference_image_id));targets=sorted(set(data['train'].target_image_id)|set(data['gallery'].image_id))
        self.frame=frame;self.readers={}
        image_root,image_index=root,index
        text_spec=SOURCE['trained_text_cache'];text_root=Path(text_spec['root'])
        require(file_hash(text_root/'index.json')==text_spec['index_sha256'],'Trained text index changed')
        text_index=read_json(text_root/'index.json');text_contract=text_index['contract']
        require(text_contract==text_spec['contract'],'Trained text contract changed')
        if text_spec['schema']=='native':
            require(text_contract['encoder_checkpoint_sha256']==SOURCE['encoder']['sha256'] and text_contract['encoder_weights_sha256']==tensor_hash(encoder) and text_contract['phase']=='development','Wrong trained text encoder/phase')
        else:require(text_contract['base_checkpoint_sha256']==SOURCE['encoder']['sha256'],'Wrong legacy trained text encoder')
        for k in ('train_sha256','validation_sha256'):require(text_contract[k]==data[k],'Trained text manifest differs')
        require(text_contract['caption_policy']==CAPTION_POLICY,'Text caption policy differs')
        for role in ('reference','reference_pooled','target_pooled','text','mask','eos','input_ids'):
            root,index=(image_root,image_index) if role in ('reference','reference_pooled','target_pooled') else (text_root,text_index)
            ids=refs if role.startswith('reference') else targets if role=='target_pooled' else qids
            spec=index['specs'][role];require(spec['ids']==ids and spec['shape']==SOURCE['cache']['role_shapes'][role]['shape'] and spec['dtype']==SOURCE['cache']['role_shapes'][role]['dtype'],'Source full-cache ID/schema mismatch: '+role)
            self.readers[role]=ReadonlyCache(root,role,ids,LOCAL/'source_cache')
        # Complete tokenizer/mask/EOS/padding check across all18000 source rows.
        for start in range(0,len(frame),128):
            rows=frame.iloc[start:start+128];ids=rows.query_id.tolist();tok=tokenizer(rows.modification_text.tolist(),padding='max_length',max_length=77,truncation=False,return_tensors='np')
            i=self.take('input_ids',ids);m=self.take('mask',ids);e=self.take('eos',ids);h=self.take('text',ids)
            require(np.array_equal(i,tok['input_ids']) and np.array_equal(m,tok['attention_mask']),'Text/caption/token row mapping differs')
            require(np.array_equal(eos_positions(torch.from_numpy(i),torch.from_numpy(m),tokenizer.eos_token_id).numpy(),e),'Cached EOS differs')
            validate_frozen_text(torch.from_numpy(h),torch.from_numpy(m),torch.from_numpy(e))
        self.native_cache_diagnostic=[]
        # Samples check native reconstruction, not a claim of full native re-encoding.
        for category in CATEGORIES:
            rows=data['validation'][data['validation'].category==category].iloc[:16];b=self.batch(rows,False)
            modifier={k:torch.from_numpy(self.take(k,rows.query_id.tolist())).to(DEVICE) for k in ('input_ids','mask')};modifier['attention_mask']=modifier.pop('mask')
            with torch.no_grad():native=encoder.encode_text(modifier)[0]
            error=float((native-b['text_features']).abs().max().cpu());self.native_cache_diagnostic.append({'category':category,'max_abs':error})
            require(torch.allclose(native,b['text_features'],atol=NATIVE_ATOL,rtol=NATIVE_RTOL),'Native trained text vs source cache differs; diagnose encoder/masks/backend before pilot')
    def take(self,role,ids):return self.readers[role].take(ids)
    def batch(self,rows,training=False):
        ids=rows.query_id.tolist();refids=rows.reference_image_id.tolist()
        tensor=lambda role,ids:torch.from_numpy(self.take(role,ids)).to(DEVICE)
        b={'reference_features':{'tokens':tensor('reference',refids),'pooled':tensor('reference_pooled',refids)},'text_features':tensor('text',ids),'mask':tensor('mask',ids),'eos':tensor('eos',ids)}
        if training:b.update(target=tensor('target_pooled',rows.target_image_id.tolist()).detach(),target_ids=rows.target_image_id.tolist())
        return b

class GalleryInputs:
    def __init__(self,source):self.source=source
    def take(self,role,ids):
        require(role=='target_pooled','Unexpected gallery role');return self.source.take(role,ids)

class CachedPilotMR(ResidualMRCore):
    def encode_query(self,ref,text,mask=None,eos=None):
        return self(ref['mr_tokens'],ref['pooled'],text['text_embedding'],text['guidance'])

class ReducedInputs:
    def __init__(self,source,data,base,encoder,adapter):
        self.source=source;self.data=data;self.lookup=source.frame.set_index('query_id');qids=sorted(self.lookup.index);refs=sorted(set(self.lookup.reference_image_id))
        contract={'schema':'adapter-mr-frozen-reductions-v1','implementation_sha256':IMPLEMENTATION_SHA,'encoder_sha256':SOURCE['encoder']['sha256'],'adapter_sha256':SOURCE['adapter']['sha256'],'source_cache_index_sha256':SOURCE['cache']['index_sha256'],'source_cache_fingerprint':SOURCE['cache']['fingerprint'],'train_sha256':data['train_sha256'],'validation_sha256':data['validation_sha256'],'gallery_sha256':data['gallery_sha256'],'caption_policy':CAPTION_POLICY,'extraction_contract':EXTRACTION_CONTRACT,'runtime':physical_runtime(),'reduction_batch_policy':'split-category-original-source-order-batches32','no_target_in_text_reduction':True}
        specs={'mr_tokens':{'ids':refs,'shape':[50,512],'dtype':'float32'}}
        for branch in ('C','D'):
            for role in ('text_embedding','guidance'):specs[branch+'_'+role]={'ids':qids,'shape':[512],'dtype':'float32'}
        self.store=TrialStore(ROOT/'pilot_cache',LOCAL/'pilot_cache',contract,specs,128);self.key=self.store.key;self.reductions=None
        def projected(ids):
            a=[]
            for start in range(0,len(ids),16):
                raw=torch.from_numpy(source.take('reference',ids[start:start+16])).to(DEVICE)
                v=project_reference_tokens(raw,base.vision_model.post_layernorm,base.visual_projection)
                pooled=torch.from_numpy(source.take('reference_pooled',ids[start:start+16])).to(DEVICE)
                require(torch.allclose(unit(v[:,0]),pooled,atol=NATIVE_ATOL,rtol=NATIVE_RTOL),'Reference CLS/projection cache mismatch')
                a.append(v.cpu().numpy())
            return np.concatenate(a)
        self.store.prepare('mr_tokens',projected)
        def all_reductions():
            if self.reductions is not None:return
            self.reductions={role:np.empty((len(qids),512),np.float32) for role in specs if role!='mr_tokens'};positions={q:i for i,q in enumerate(qids)}
            # Match native evaluation batch contexts for full validation, including final short batch.
            for frame in (data['train'],data['validation']):
                for c in CATEGORIES:
                    f=frame[frame.category==c]
                    for start in range(0,len(f),32):
                        rows=f.iloc[start:start+32];b=source.batch(rows,False)
                        with torch.no_grad():
                            enhanced=frozen_adapter_text(adapter,b['reference_features'],b['text_features'],b['mask'],b['eos'])
                            for branch,h in (('C',b['text_features']),('D',enhanced)):
                                reduced=reduce_frozen_text(h,b['mask'],b['eos'],encoder.projection.weight)
                                for role,a in reduced.items():self.reductions[branch+'_'+role][[positions[q] for q in rows.query_id]]=a.cpu().numpy()
        for role in specs:
            if role=='mr_tokens':continue
            def compute(ids,r=role):
                all_reductions();position={q:i for i,q in enumerate(qids)};return self.reductions[r][[position[q] for q in ids]]
            self.store.prepare(role,compute)
        self.reductions=None
    def batch(self,rows,branch,training=False):
        qids=rows.query_id.tolist();refs=rows.reference_image_id.tolist();take=lambda role,ids:torch.from_numpy(self.store.take(role,ids)).to(DEVICE)
        b={'reference_features':{'mr_tokens':take('mr_tokens',refs),'pooled':torch.from_numpy(self.source.take('reference_pooled',refs)).to(DEVICE)},'text_features':{r:take(branch+'_'+r,qids) for r in ('text_embedding','guidance')},'mask':None,'eos':None}
        if training:b.update(target=torch.from_numpy(self.source.take('target_pooled',rows.target_image_id.tolist())).to(DEVICE).detach(),target_ids=rows.target_image_id.tolist())
        return b

@torch.inference_mode()
def check_gate_and_reduced(core,source,reduced,rows,encoder,adapter,branch):
    original=source.batch(rows,False);b=reduced.batch(rows,branch,False)
    expected=CachedConventional(encoder.projection.weight).to(DEVICE).encode_query(original['reference_features'],original['text_features'],original['mask'],original['eos']) if branch=='C' else adapter.encode_query(original['reference_features'],original['text_features'],original['mask'],original['eos'])
    ref,text=b['reference_features'],b['text_features'];off=core(ref['mr_tokens'],ref['pooled'],**text,mr_off=True)
    require(torch.allclose(off,expected,atol=GATE_ATOL,rtol=0),'MR-off source A/B embedding mismatch')
    enhanced=original['text_features'] if branch=='C' else frozen_adapter_text(adapter,original['reference_features'],original['text_features'],original['mask'],original['eos'])
    native=reduce_frozen_text(enhanced,original['mask'],original['eos'],encoder.projection.weight)
    native_q=core(ref['mr_tokens'],ref['pooled'],**native)
    require(torch.allclose(query_forward(core,b,'adapter'),native_q,atol=GATE_ATOL,rtol=0),'Reduced/native MR query mismatch')
    return {'mr_off_max_abs':float((off-expected).abs().max().cpu()),'cache_native_max_abs':float((native_q-query_forward(core,b,'adapter')).abs().max().cpu())}

def compare_control(metric,pred,branch,data,attempt):
    expected=SOURCE['expected_metrics'][branch];errors={c+'/'+str(k):abs(metric['per_category'][c][f'R@{k}']-expected['per_category'][c][f'R@{k}']) for c in CATEGORIES for k in (1,5,10,50)};errors['Mean']=abs(metric['Mean']-expected['Mean'])
    path=Path(SOURCE['prediction_paths'][branch])
    require(file_hash(path)==SOURCE['prediction_sha256'][branch],'Source per-query predictions changed')
    saved=pd.read_csv(path,dtype=str,keep_default_na=False);keys=['query_id','category','reference_image_id','target_image_id'];joined=pred.merge(saved[keys+['target_rank']],on=keys,validate='one_to_one',suffixes=('_new','_old'))
    require(len(joined)==1796 and set(pred.query_id)==set(data['validation'].query_id),'Control prediction coverage/mapping')
    changed=joined[joined.target_rank_new.astype(int)!=joined.target_rank_old.astype(int)];write_csv(attempt/(branch+'_rank_differences.csv'),changed)
    tol=SOURCE['tolerances']['A_metric_reproduction_pp'] if branch=='A' else SOURCE['tolerances']['B_saved_prediction_metric_consistency_atol']
    passed=max(errors.values())<=tol if branch=='A' else max(errors.values())<tol and len(changed)==0
    record={'passed':bool(passed),'metric_deviation_pp':errors,'rank_difference_count':len(changed),'metric_tolerance_pp':tol,'note':'A source audit already had one rank difference with identical metrics; B has no relaxed rerun tolerance'}
    write_json(attempt/(branch+'_reproduction.json'),record)
    require(passed,'Control '+branch+' reproduction differs; inspect rank differences/cache/projection/caption/checkpoint, do not relax tolerance')
    return record

def run_binding(branch,data,reduced,initial):
    return {'phase':'development','seed':CFG['seed'],'kind':'mr_residual_pilot','variant':branch,'implementation_sha256':IMPLEMENTATION_SHA,'config':science_config(),'runtime':physical_runtime(),'data':{k:data[k] for k in ('train_sha256','validation_sha256','gallery_sha256')},'source_encoder':SOURCE['encoder'],'source_adapter':SOURCE['adapter'],'frozen_input_cache':reduced.key,'mr_initial_sha256':initial,'origin':'fresh identical MR; frozen source encoder and Adapter'}

def preflight_pilot(seed=None, sources=None):
    global PACKAGE,DATA,BASE,ENCODER,ADAPTER,INPUTS,REDUCED,IMAGES,INITIAL_MR,PREFLIGHT,CONTROL_METRICS
    configure_seed(CFG['seed'] if seed is None else seed,SOURCE_REGISTRY if sources is None else sources)
    started=time.perf_counter();initialize_pilot();attempt=ROOT/'preflight_runs'/time.strftime('%Y%m%dT%H%M%S')/uuid.uuid4().hex[:8];attempt.mkdir(parents=True)
    try:
        PACKAGE=PilotPreparation(CFG['preparation_root']);DATA=PACKAGE.data('development')
        BASE,tokenizer,_=load_pretrained(PACKAGE)
        ENCODER=load_encoder_asset(SOURCE['encoder'],BASE,tokenizer).freeze();ADAPTER=load_adapter_asset(SOURCE['adapter'],ENCODER).eval().requires_grad_(False)
        # Additional pinned config/marker check before using source cache, never repairs source files.
        for path,sha in active_source_locks().items():require(file_hash(path)==sha,'Source metadata changed: '+path)
        frozen={'base':tensor_hash(BASE),'encoder':tensor_hash(ENCODER),'adapter':tensor_hash(ADAPTER)}
        INPUTS=SourceInputs(PACKAGE,DATA,ENCODER,ADAPTER,BASE,tokenizer);IMAGES=GalleryInputs(INPUTS)
        CONTROL_METRICS={};reproduction={}
        for label,model in [('A',CachedConventional(ENCODER.projection.weight).to(DEVICE)),('B',ADAPTER)]:
            folder=attempt/'controls'/label;metric,_=export_evaluation(folder,model,DATA,IMAGES,INPUTS.batch,'adapter');pred=pd.read_csv(folder/'per_query_results.csv',dtype=str,keep_default_na=False)
            reproduction[label]=compare_control(metric,pred,label,DATA,attempt);CONTROL_METRICS[label]=metric
        REDUCED=ReducedInputs(INPUTS,DATA,BASE,ENCODER,ADAPTER)
        with isolated_rng():
            seed_all(CFG['seed']);initial=CachedPilotMR().to(DEVICE);INITIAL_MR=cpu_tree(initial.state_dict());init_hash=tensor_hash(initial)
            parity={}
            for branch,control in [('C','A'),('D','B')]:
                rows=pd.concat([DATA['validation'][DATA['validation'].category==c].iloc[:32] for c in CATEGORIES])
                # Verify using separate source category batches, not an invented merged evaluation context.
                parity[branch]=[{'category':c,'batch32':check_gate_and_reduced(initial,INPUTS,REDUCED,rows[rows.category==c],ENCODER,ADAPTER,branch),'batch1':check_gate_and_reduced(initial,INPUTS,REDUCED,rows[rows.category==c].iloc[:1],ENCODER,ADAPTER,branch)} for c in CATEGORIES]
                metric,q,_=retrieval_evaluate(initial,DATA,IMAGES,lambda r,t=False:REDUCED.batch(r,branch,t),'adapter',50)
                metric_equal(metric,CONTROL_METRICS[control])
                source_q=pd.read_csv(attempt/'controls'/control/'per_query_results.csv',dtype=str).set_index('query_id')
                require((q.set_index('query_id').target_rank.astype(int)==source_q.loc[q.query_id].target_rank.astype(int)).all(),'Zero-MR full validation ranks differ from native '+control)
            smoke=smoke_pilot_runner(initial,DATA,REDUCED,IMAGES)
        require(frozen=={'base':tensor_hash(BASE),'encoder':tensor_hash(ENCODER),'adapter':tensor_hash(ADAPTER)},'Smoke changed frozen source weights')
        marker={'preflight_wall_seconds':time.perf_counter()-started,'passed':True,'config':science_config(),'implementation_sha256':IMPLEMENTATION_SHA,'cache':REDUCED.key,'initial_mr_sha256':init_hash,'runtime':physical_runtime(),'frozen_hashes':frozen,'attempt':str(attempt),'reproduction':reproduction,'parity':parity,'smoke':smoke,'native_cache_sample_checks':INPUTS.native_cache_diagnostic,'ready_for_pilot':True}
        write_json(attempt/'test_report.json',marker);write_json(ROOT/'preflight.json',marker);PREFLIGHT=marker
        summarize_pilot();print('Preflight đạt. A/B đã tái hiện; chỉ bật RUN_PILOT=True khi muốn chạy C/D tối đa3 epoch.')
        return marker
    except BaseException:
        write_json(attempt/'test_report.json',{'passed':False,'ready_for_pilot':False,'traceback':traceback.format_exc()});write_json(ROOT/'preflight.json',{'passed':False,'ready_for_pilot':False,'attempt':str(attempt)});raise

def run_pilot(seed=None):
    require(seed is None or seed==CFG['seed'],'Training seed differs from preflight')
    require(CFG['RUN_PILOT'] is True,'Training is opt-in: set CFG[RUN_PILOT]=True')
    require(PREFLIGHT and PREFLIGHT['passed'] and PREFLIGHT['config']==science_config() and PREFLIGHT['runtime']==physical_runtime(),'Run mandatory preflight with matching config/runtime before training')
    for role,key in [('train','train_sha256'),('validation','validation_sha256'),('gallery','gallery_sha256')]:require(frame_sha(DATA[role])==DATA[key],'In-memory development manifest changed after preflight')
    require(len(DATA['train'])==16204 and len(DATA['validation'])==1796 and len(DATA['gallery'])==1796,'Pilot full-data coverage changed')
    require(tensor_hash(BASE)==PREFLIGHT['frozen_hashes']['base'],'Frozen vision/base changed')
    require(PREFLIGHT['cache']==REDUCED.key and tensor_hash(ENCODER)==PREFLIGHT['frozen_hashes']['encoder'] and tensor_hash(ADAPTER)==PREFLIGHT['frozen_hashes']['adapter'],'Frozen source/cache changed')
    for branch in ('C','D'):
        with isolated_rng():
            seed_all(CFG['seed']);model=CachedPilotMR().to(DEVICE);model.load_state_dict(INITIAL_MR,strict=True)
            require(tensor_hash(model)==PREFLIGHT['initial_mr_sha256'],'C/D MR initialization differs')
            binding=run_binding(branch,DATA,REDUCED,tensor_hash(model));folder=ROOT/branch
            batch=lambda rows,training=False,b=branch:REDUCED.batch(rows,b,training)
            def evaluate(m):
                metrics,queries,_=retrieval_evaluate(m,DATA,IMAGES,batch,'adapter',50);return metrics,queries
            result=optimize_stage(model,DATA['train'],batch,evaluate,folder,LOCAL/branch,binding,CFG['hp'],3,None)
            require(result['finished'] and result['epoch']==3,'Pilot incomplete; preserve last checkpoint and resume same config')
            best=checkpoint_load(folder/'best.pt',fingerprint(binding));required_checkpoint(best,binding);model.load_state_dict(best['model'],strict=True)
            metrics,receipts=export_evaluation(folder/'best_evaluation',model,DATA,IMAGES,batch,'adapter')
            require(abs(metrics['Mean']-best['best_mean'])<1e-10 and metrics['n_query']==1796,'Best metric/coverage incomplete')
            verify_receipts({str(folder/'best_evaluation'/name):sha for name,sha in receipts.items()})
            completion={'passed':True,'completed':True,'phase':'development-pilot','seed':CFG['seed'],'config':science_config(),'binding':binding,'best_epoch':best['epoch'],'metrics':metrics,'result':result,'file_receipts':receipts}
            write_json(folder/'timing.json',result['timing']);write_json(folder/'completion.json',completion)
        require(tensor_hash(BASE)==PREFLIGHT['frozen_hashes']['base'] and tensor_hash(ENCODER)==PREFLIGHT['frozen_hashes']['encoder'] and tensor_hash(ADAPTER)==PREFLIGHT['frozen_hashes']['adapter'],'Pilot changed frozen vision/encoder/Adapter weights')
        summarize_pilot();print(branch,'complete | best epoch',best['epoch'],'| Mean',metrics['Mean'])

def summarize_pilot():
    rows=[];metrics={}
    for label in ('A','B','C','D'):
        metric=CONTROL_METRICS.get(label);seconds=0. if label in ('A','B') else None;epoch=None;status='evaluated' if metric else 'not_run'
        if label in ('C','D') and (ROOT/label/'completion.json').is_file():
            r=read_json(ROOT/label/'completion.json');require(r['passed'] and r['completed'] and r['config']==science_config(),'Completion config differs')
            expected_binding=run_binding(label,DATA,REDUCED,PREFLIGHT['initial_mr_sha256']);require(r['binding']==expected_binding,'Completion code/runtime/cache/source differs')
            last=checkpoint_load(ROOT/label/'last.pt',fingerprint(expected_binding));required_checkpoint(last,expected_binding)
            best=checkpoint_load(ROOT/label/'best.pt',fingerprint(expected_binding));required_checkpoint(best,expected_binding)
            require(last['finished'] and last['epoch']==3 and best['epoch']==r['best_epoch'] and best['best_mean']==r['metrics']['Mean'],'Completion checkpoint/metric differs')
            for name,sha in r['file_receipts'].items():require(file_hash(ROOT/label/'best_evaluation'/name)==sha,'Completed output corrupt')
            metric=r['metrics'];seconds=r['result']['timing']['training_wall_seconds'];epoch=r['best_epoch'];status='evaluated'
        if metric:metrics[label]=metric
        rows.append({'variant':label,'status':status,'R@10':metric['overall']['R@10'] if metric else None,'R@50':metric['overall']['R@50'] if metric else None,'Mean':metric['Mean'] if metric else None,'best_epoch':epoch,'training_wall_seconds':seconds,'trainable_parameters':262913 if label in ('C','D') else 0})
    write_csv(ROOT/'comparison.csv',pd.DataFrame(rows));deltas={}
    for tag,left,right in [('C_minus_A','C','A'),('D_minus_B','D','B'),('D_minus_C','D','C')]:
        deltas[tag]={k:metrics[left]['overall'][k]-metrics[right]['overall'][k] for k in ('R@10','R@50')}|{'Mean':metrics[left]['Mean']-metrics[right]['Mean']} if left in metrics and right in metrics else None
    write_json(ROOT/'paired_deltas.json',deltas)
    text='# Adapter + MR pilot\n\nSeed'+str(CFG['seed'])+', development only; A/B frozen controls, C/D fresh identical MR. '+('C/D complete.\n' if all(k in metrics for k in ('C','D')) else 'C/D chưa đủ kết quả; metric chưa chạy để trống.\n')
    text+='\nMR là biến thể residual token-mean quanh source gate; không thêm head. Không dùng official-val để chọn epoch.\n\n'+pd.DataFrame(rows).to_csv(index=False)+'\nDeltas (điểm %): '+json.dumps(deltas,ensure_ascii=False)+'\n\nMột seed/ngân sách pilot mới; không suy luận ý nghĩa thống kê, SOTA hoặc độ ổn định ba seed. A/B train cost0 ở pilot, không phải chi phí huấn luyện lịch sử.\n'
    write_bytes(ROOT/'summary.md',text.encode())
    from IPython.display import display
    display(pd.DataFrame(rows))

def verify_historical_seed(seed, data, sources):
    """Read-only semantic compatibility, NOT permission to resume old implementation."""
    require(seed in (42,43,44),'Unexpected seed')
    root=Path(CFG['output_root'])/f'adapter_mr_pilot_v1_seed{seed}'
    if not (root/'preflight.json').is_file():return None
    marker=read_json(root/'preflight.json');require(marker['passed'],'Historical preflight incomplete')
    saved_config=read_json(root/'config.json')
    require(saved_config['seed']==seed and saved_config['epochs']==3 and saved_config['hp']==PILOT_HP and saved_config['adapter']=={'width':128,'heads':4} and saved_config['eval_batch_size']==32,'Historical scientific recipe differs')
    actual_source=sources[seed];metrics={};rows=[]
    for label in ('A','B'):
        folder=Path(marker['attempt'])/'controls'/label
        metric=read_json(folder/'metrics.json');pred=pd.read_csv(folder/'per_query_results.csv',dtype=str)
        metric_equal(metrics_from_predictions(pred,data),metric)
        expected=actual_source['expected_metrics'][label]
        deviation=max(abs(metric['per_category'][c][f'R@{k}']-expected['per_category'][c][f'R@{k}']) for c in CATEGORIES for k in (1,5,10,50))
        require(deviation<=.01 if label=='A' else deviation<1e-10,'Historical control differs from its own source seed')
        metrics[label]=metric;rows.append({'seed':seed,'variant':label,'R@10':metric['overall']['R@10'],'R@50':metric['overall']['R@50'],'Mean':metric['Mean'],'best_epoch':None,'additional_training_seconds':0.,'cost_scope':'no retraining in MR pilot; excludes historical encoder/adapter cost'})
    checks={}
    for label in ('C','D'):
        path=root/label/'completion.json'
        if not path.is_file():continue
        done=read_json(path);binding=done['binding']
        require(done['passed'] and done['completed'] and binding['seed']==seed and binding['variant']==label and binding['phase']=='development','Historical completion seed/phase differs')
        require(binding['config']['hp']==PILOT_HP and binding['config']['epochs']==3,'Historical pilot recipe differs')
        for key in ('encoder','adapter'):require(binding['source_'+key]['sha256']==actual_source[key]['sha256'],'Historical model uses wrong encoder/adapter')
        for key in ('train_sha256','validation_sha256','gallery_sha256'):require(binding['data'][key]==data[key],'Historical data mapping differs')
        # These loaders do not repair or modify source markers, hashes or checkpoints.
        last=committed_checkpoint_load(root/label/'last.pt',fingerprint(binding));required_checkpoint(last,binding)
        best=committed_checkpoint_load(root/label/'best.pt',fingerprint(binding));required_checkpoint(best,binding)
        require(last['initial_sha256']==binding['mr_initial_sha256'],'Historical initializer fingerprint differs')
        require(all(v.dtype==torch.float32 and bool(torch.isfinite(v).all()) for v in best['model'].values()),'Historical MR checkpoint dtype/finite mismatch')
        require(last['finished'] and last['epoch']==3 and last['step']==1521 and len(last['history'])==1521,'Historical training budget/cursor incomplete')
        require(best['epoch']==done['best_epoch'] and best['best_mean']==done['metrics']['Mean'],'Historical best checkpoint/metric mismatch')
        with isolated_rng():
            compatible_model=CachedPilotMR().cpu();compatible_model.load_state_dict(best['model'],strict=True)
            require(sum(p.numel() for p in compatible_model.parameters())==262913,'Historical MR schema differs')
        # Earliest exact tie is independently recovered from actual epoch metrics.
        curve=last['metrics'];best_mean=max(e['metrics']['Mean'] for e in curve);selected=min(e['epoch'] for e in curve if e['metrics']['Mean']==best_mean)
        require(len(curve)==3 and selected==done['best_epoch'],'Historical best epoch selection differs')
        for name,sha in done['file_receipts'].items():require(file_hash(root/label/'best_evaluation'/name)==sha,'Historical output receipt mismatch')
        pred=pd.read_csv(root/label/'best_evaluation/per_query_results.csv',dtype=str);metric=read_json(root/label/'best_evaluation/metrics.json')
        metric_equal(metrics_from_predictions(pred,data),metric);metric_equal(metric,done['metrics'])
        metrics[label]=metric
        rows.append({'seed':seed,'variant':label,'R@10':metric['overall']['R@10'],'R@50':metric['overall']['R@50'],'Mean':metric['Mean'],'best_epoch':selected,'additional_training_seconds':done['result']['timing']['training_wall_seconds'],'cost_scope':'MR pilot training only; excludes frozen source training and cache/evaluation'})
        checks[label]={'semantic_compatibility':'read-only compatible recipe/data/source/model schema/ranking','original_implementation_sha256':binding['implementation_sha256'],'original_signature':last['signature'],'resume_across_refactor':'not attempted or permitted','initial_sha256_preserved':last['initial_sha256']}
    if 'C' in checks and 'D' in checks:
        c=committed_checkpoint_load(root/'C/last.pt',fingerprint(read_json(root/'C/completion.json')['binding']))
        d=committed_checkpoint_load(root/'D/last.pt',fingerprint(read_json(root/'D/completion.json')['binding']))
        require(c['initial_sha256']==d['initial_sha256'],'C/D historical MR initializers differ')
        require([r['batch_ids_sha256'] for r in c['history']]==[r['batch_ids_sha256'] for r in d['history']],'C/D historical batch order differs')
    return {'seed':seed,'metrics':metrics,'rows':rows,'compatibility':checks,'historical_read_only':seed!=CFG['seed']}

def aggregate_pilots(data=None,sources=None):
    data=DATA if data is None else data;sources=SOURCE_REGISTRY if sources is None else sources
    results=[verify_historical_seed(seed,data,sources) for seed in (42,43,44)];results=[r for r in results if r]
    per_seed=pd.DataFrame([row for r in results for row in r['rows']]);paired=[]
    for r in results:
        m=r['metrics']
        if 'D' not in m:continue
        d=next(x for x in r['rows'] if x['variant']=='D')
        paired.append({'seed':r['seed'],'Adapter_Mean':m['B']['Mean'],'Adapter_MR_Mean':m['D']['Mean'],'D_minus_B_pp':m['D']['Mean']-m['B']['Mean'],'best_epoch_D':d['best_epoch'],'additional_training_seconds':d['additional_training_seconds']})
    paired=pd.DataFrame(paired);stats=[]
    if len(paired)==3:
        for col in ('Adapter_Mean','Adapter_MR_Mean','D_minus_B_pp','additional_training_seconds'):
            stats.append({'measure':col,'mean':float(paired[col].mean()),'sample_std':float(paired[col].std(ddof=1)),'n':3,'scope':'development pilot; best epoch selected separately per seed, not final benchmark'})
    write_csv(ROOT/'all_seed_abcd.csv',per_seed);write_csv(ROOT/'adapter_mr_paired_seeds.csv',paired);write_csv(ROOT/'three_seed_statistics.csv',pd.DataFrame(stats))
    write_json(ROOT/'historical_compatibility.json',[{'seed':r['seed'],'checks':r['compatibility'],'read_only':r['historical_read_only']} for r in results])
    write_json(ROOT/'three_seed_statistics.json',{'scope':'development pilot, per-seed best epochs, not final benchmark or significance evidence','complete_three_seeds':len(paired)==3,'statistics':stats})
    summary='# Adapter + MR development pilot\n\n'+per_seed.to_csv(index=False)+'\n'+paired.to_csv(index=False)+'\n'+pd.DataFrame(stats).to_csv(index=False)+'\nBest epoch được chọn riêng từng seed. Không suy luận ý nghĩa thống kê/SOTA. Chi phí thêm chỉ tính MR pilot, không coi frozen encoder/Adapter là miễn phí cho toàn pipeline.\n'
    write_bytes(ROOT/'summary.md',summary.encode())
    from IPython.display import display
    display(per_seed);display(paired);display(pd.DataFrame(stats))
    return {'per_seed':per_seed,'paired':paired,'statistics':stats}


In [ ]:
def same_tree(a,b):
    if isinstance(a,torch.Tensor):return isinstance(b,torch.Tensor) and torch.equal(a,b)
    if isinstance(a,np.ndarray):return isinstance(b,np.ndarray) and np.array_equal(a,b)
    if isinstance(a,dict):return isinstance(b,dict) and a.keys()==b.keys() and all(same_tree(a[k],b[k]) for k in a)
    if isinstance(a,(list,tuple)):return type(a)==type(b) and len(a)==len(b) and all(same_tree(x,y) for x,y in zip(a,b))
    return a==b

def pilot_fixture_checks():
    q=np.array([[1,0],[0,1]],np.float32);g=np.array([[1,0],[1,0],[0,1]],np.float32)
    require(exact_ranks(q,g,[1,2]).tolist()==[2,1],'Stable tie/ID ranking fixture')
    per={'dress':{'R@1':0.,'R@5':0.,'R@10':20.,'R@50':60.},'shirt':{'R@1':0.,'R@5':0.,'R@10':40.,'R@50':80.},'toptee':{'R@1':0.,'R@5':0.,'R@10':60.,'R@50':100.}}
    m=macro_metrics(per);require(m['overall']['R@10']==40. and m['overall']['R@50']==80. and m['Mean']==60.,'Macro/Mean fixture')
    with isolated_rng():
        x=unit(torch.randn(4,512,device=DEVICE));target=unit(torch.randn(4,512,device=DEVICE));ids=['same','same','other','last'];p=positive_mask(ids,DEVICE);require(int(p.sum())==6,'Duplicate positives missing')
        logits=unit(x)@unit(target.detach()).T/.07
        def direction(values,positives):return -((F.log_softmax(values,dim=1)*positives).sum(1)/positives.sum(1)).mean()
        expected=(direction(logits,p)+direction(logits.T,p.T))/2
        require(torch.allclose(retrieval_loss(x,target,ids,.07),expected,atol=0,rtol=0),'Source symmetric multi-positive loss fixture')
    return {'ranking_ties':'passed','macro_mean':'passed','duplicate_target_loss':'passed','metrics_are_fixture_only':True}

def smoke_pilot_runner(main,data,reduced,images):
    original_rng=rng_state();main_hash=tensor_hash(main);initial=cpu_tree(main.state_dict());reports={};all_orders={}
    try:
        with isolated_rng(),tempfile.TemporaryDirectory(prefix='mr_pilot_smoke_',dir=LOCAL) as temporary:
            tmp=Path(temporary);frame=pd.concat([data['train'][data['train'].category==c].iloc[:32] for c in CATEGORIES]);require(len(frame)==96,'Smoke needs96 real train rows')
            hp={**CFG['hp'],'checkpoint_every_steps':1}
            for branch in ('C','D'):
                binding={'phase':'development','seed':CFG['seed'],'kind':'mr_residual_pilot','variant':branch,'fixture_scope':'temporary real-data smoke, never main training','implementation_sha256':IMPLEMENTATION_SHA,'ids_sha256':fingerprint(frame.query_id.tolist()),'cache':reduced.key}
                def fresh():
                    m=CachedPilotMR().to(DEVICE);m.load_state_dict(initial,strict=True);return m
                reference=fresh();params=dict(reference.named_parameters());require(set(params) and all(n.startswith('mr.') for n in params),'Optimizer contains a frozen source parameter')
                require(sum(p.numel() for p in params.values())==262913,'MR parameter count changed')
                b=reduced.batch(frame.iloc[:32],branch,True)
                with torch.no_grad():
                    a=query_forward(reference,b,'adapter');changed={**b,'target':b['target'].flip(0)}
                    require(torch.equal(a,query_forward(reference,changed,'adapter')),'Query depends on target')
                    one=reduced.batch(frame.iloc[:1],branch,False);require(query_forward(reference,one,'adapter').shape==(1,512),'Batch1 shape')
                gradient_stages={n:[] for n in params}
                handles=[p.register_hook(lambda grad,n=n:gradient_stages[n].append(float(grad.detach().square().sum().cpu()))) for n,p in params.items()]
                teachers=[]
                def batch(rows,training=True):
                    v=reduced.batch(rows,branch,training)
                    if training:
                        v['target']=v['target'].detach().clone().requires_grad_();teachers.append(v['target'])
                    return v
                def evaluate(m):
                    metric,queries,_=retrieval_evaluate(m,data,images,lambda r,t=False:reduced.batch(r,branch,t),'adapter',50);return metric,queries
                full=optimize_stage(reference,frame,batch,evaluate,tmp/branch/'full',tmp/branch/'full_local',binding,hp,1,None)
                expected=checkpoint_load(tmp/branch/'full_local/last.pt',fingerprint(binding));require(full['step']==3 and full['finished'],'Smoke runner did not execute3 optimizer steps')
                for h in handles:h.remove()
                require(all(t.grad is None for t in teachers),'Target teacher gradient leaked')
                hidden=[n for n in params if not n.startswith('mr.out.')];heads=[n for n in params if n.startswith('mr.out.')]
                require(all(len(v)==3 and np.isfinite(v).all() for v in gradient_stages.values()),'MR gradient absent/nonfinite')
                require(sum(gradient_stages[n][0] for n in hidden)==0 and sum(gradient_stages[n][0] for n in heads)>0,'MR zero-init gradient flow unexpected')
                require(sum(gradient_stages[n][1] for n in hidden)>0,'MR hidden layers remain disconnected after gate update')
                count=0
                def interrupt(rows,training=True):
                    nonlocal count
                    count+=1
                    if count==3:raise RuntimeError('expected temporary smoke interruption before batch3')
                    return batch(rows,training)
                try:optimize_stage(fresh(),frame,interrupt,evaluate,tmp/branch/'resume',tmp/branch/'resume_local',binding,hp,1,None)
                except RuntimeError as e:require('expected temporary smoke interruption' in str(e),'Unexpected smoke runner error: '+str(e))
                else:raise RuntimeError('Smoke interruption was not exercised')
                saved=checkpoint_load(tmp/branch/'resume_local/last.pt',fingerprint(binding));require((saved['epoch'],saved['cursor'],saved['step'])==(0,2,2),'Mid-epoch checkpoint cursor differs')
                resumed=fresh();optimize_stage(resumed,frame,batch,evaluate,tmp/branch/'resume',tmp/branch/'resume_local',binding,hp,1,None)
                actual=checkpoint_load(tmp/branch/'resume_local/last.pt',fingerprint(binding))
                require(same_tree(expected['model'],actual['model']) and same_tree(expected['optimizer'],actual['optimizer']) and same_rng(expected['rng'],actual['rng']),'Model/optimizer/RNG resume differs')
                for field in ('batch_ids_sha256','loss_retrieval','gradient_norm'):require([x[field] for x in expected['history']]==[x[field] for x in actual['history']],'Resume order/loss/grad differs')
                all_orders[branch]=[x['batch_ids_sha256'] for x in expected['history']]
                best=checkpoint_load(tmp/branch/'resume/best.pt',fingerprint(binding));require(best['epoch']==1 and best['best_mean']==actual['best_mean'],'Best save/load differs')
                resumed.load_state_dict(best['model'],strict=True)
                metric,_=export_evaluation(tmp/branch/'evaluation',resumed,data,images,lambda r,t=False:reduced.batch(r,branch,t),'adapter')
                require(metric['n_query']==1796,'Smoke evaluator lost validation queries')
                reports[branch]={'passed':True,'scope':'temporary real-data production-runner smoke','train_rows':96,'validation_rows':1796,'optimizer_steps_per_pass':3,'interruption_cursor':2,'resume_exact_model_optimizer_rng_order':True,'source_model_and_rng_unchanged':True,'gradient_initial_out_then_hidden':gradient_stages,'batch1':True,'query_target_independent':True,'teacher_detached':True,'evaluator_csv_json_readback':True}
            require(all_orders['C']==all_orders['D'],'C/D smoke batch order differs')
        reports['fixtures']=pilot_fixture_checks()
        return reports
    finally:
        restore_rng(original_rng);require(tensor_hash(main)==main_hash and same_rng(original_rng,rng_state()),'Smoke changed main model/RNG')


## Preflight bắt buộc
A/B được inference đầy đủ1796 query và đối chiếu đúng artifact của seed đã chọn. MR-off khớp đối chứng; kiểm toàn bộ ID/cache/mask/EOS, gradient, teacher detach, frozen hashes và production-runner resume bằng model/RNG tạm. Không tăng tolerance. B không có tolerance rerun rộng hơn, giữ metric<1e-10/rank parity; A giữ tolerance0.01 điểm% như pilot nguồn.


In [ ]:
# Historical seed42 preflight output only; not generated by the refactored code.


In [ ]:
# Historical seed42 pilot output only; not generated by the refactored code.


## Kết quả lịch sử42/43
Output ở ô10/11 và13/14 là lịch sử đã xác minh, không phải kết quả chạy lại bằng code refactor. Artifact nguồn và checkpoint42/43 không được viết lại; aggregate_pilots() kiểm tra checksum, binding, recipe, schema, per-query/ranking trước khi reuse. Việc đọc lịch sử không được coi là resume training xuyên implementation.


In [ ]:
# Historical seed43 preflight output only; not generated by the refactored code.


In [ ]:
# Historical seed43 pilot output only; not generated by the refactored code.


In [ ]:
# Historical completed pilot output retained; not rerun by final phases.
if RUN_PILOT_VALIDATION:
    PREFLIGHT = preflight_pilot(CFG['seed'], SOURCE_REGISTRY)
else:
    print('Historical pilot validation/reporting is disabled by default.')


In [ ]:
# Saved output below is the completed authorized seed44 run with RUN_PILOT=True.
# The default is restored to False; this code has not rerun the historical output.
RUN_PILOT = False  # Safe default for Run all. Set True only after mandatory preflight.
CFG['RUN_PILOT']=RUN_PILOT
if RUN_PILOT:
    run_pilot(CFG['seed'])
else:
    print('Preflight only: no MR training started.')


In [ ]:
# Historical completed pilot output retained; not rerun by final phases.
if RUN_PILOT_VALIDATION:
    RUN_PILOT = False
    CFG['RUN_PILOT'] = False
    PILOT_RESULTS = aggregate_pilots(DATA, SOURCE_REGISTRY)
else:
    print('Historical pilot validation/reporting is disabled by default.')


## Hoàn thành development pilot seed44

Đã chạy thực tế trên Colab GPU T4: preflight, tái hiện A/B, MR-off, gradient, frozen hashes, save/load/resume bằng trạng thái tạm, rồi C/D đủ ba epoch (1.521 step mỗi cấu hình), đánh giá đủ1.796 query và gallery đã khóa. Cell đọc lại cuối đã đạt. Không training encoder/Adapter, không final/official-val.

| Seed | Adapter B Mean (%) | Adapter + MR D Mean (%) | D−B (đpt) | Best epoch D | MR D train (giây) |
|---|---:|---:|---:|---:|---:|
|42|74.885586|74.969198|+0.083612|3|80.863082|
|43|73.938960|74.272896|+0.333936|1|88.428241|
|44|73.326596|73.521505|+0.194909|3|76.809870|

Seed44: A=72.492290%; C=72.186038%, best epoch1, thấp hơn A0.306252đpt; C train83.619822 giây. Output42/43 là lịch sử đã đối chiếu checkpoint/receipt chỉ đọc, không train lại dưới code refactor.

Ba seed: B=74.050381±0.785445%; D=74.254533±0.724021%; D−B=+0.204152±0.125418đpt (sample std, ddof=1). Development pilot với best epoch riêng từng seed; không suy luận ý nghĩa thống kê/SOTA hoặc benchmark final. Thời gian trên chỉ là training MR bổ sung, không coi chi phí train frozen encoder/Adapter, preflight/cache/evaluation là bằng0.

Artifact mới: `MyDrive/ĐATN/data/processed/adapter_mr_pilot/FashionIQ/adapter_mr_pilot_v1_seed44`. Xem `all_seed_abcd.csv`, `adapter_mr_paired_seeds.csv`, `three_seed_statistics.json`, `abcd_statistics.json`, `historical_compatibility.json`, `final_verification.json`; chi tiết preflight ở đường dẫn attempt trong `preflight.json`. Artifact42/43 giữ nguyên.

Mặc định `RUN_PILOT=False` đã khôi phục. Output training44 lưu ở cell trên là receipt của lượt đã chủ động bật True, không phải chạy dưới giá trị False hiện tại. Không cần train lại. Nếu runtime mất: chạy setup/định nghĩa và preflight trước khi đọc/tổng hợp; chỉ resume cùng cấu hình khi có checkpoint hợp lệ.


In [ ]:
# Historical completed pilot output retained; not rerun by final phases.
if RUN_PILOT_VALIDATION:
    # Artifact readback and reporting only; no training or inference.
    RUN_PILOT = False
    CFG['RUN_PILOT'] = False
    saved_rows = pd.read_csv(ROOT/'all_seed_abcd.csv')
    saved_pairs = pd.read_csv(ROOT/'adapter_mr_paired_seeds.csv')
    saved_stats = read_json(ROOT/'three_seed_statistics.json')
    require(len(saved_rows)==12 and not saved_rows.duplicated(['seed','variant']).any(), 'Expected twelve unique seed/variant results')
    require(set(saved_rows.seed)=={42,43,44} and set(saved_rows.variant)=={'A','B','C','D'}, 'Incomplete seed/variant coverage')
    require(saved_stats['complete_three_seeds'] and len(saved_pairs)==3 and set(saved_pairs.seed)=={42,43,44}, 'Incomplete paired three-seed statistics')
    require(np.isfinite(saved_rows[['R@10','R@50','Mean']].to_numpy()).all(), 'Invalid saved metric')
    require(np.allclose(saved_rows.Mean,(saved_rows['R@10']+saved_rows['R@50'])/2,atol=1e-10,rtol=0), 'Saved Mean does not follow protocol')
    require(PREFLIGHT['passed'] and PREFLIGHT['ready_for_pilot'] and PREFLIGHT['config']==science_config(), 'Preflight binding mismatch')
    require(PREFLIGHT['frozen_hashes']=={'base':tensor_hash(BASE),'encoder':tensor_hash(ENCODER),'adapter':tensor_hash(ADAPTER)}, 'Frozen source weights changed after pipeline')
    verified_budget = {}
    for branch in ('C','D'):
        completion = read_json(ROOT/branch/'completion.json')
        require(completion['passed'] and completion['completed'] and completion['seed']==44 and completion['config']==science_config(), 'Seed44 completion mismatch')
        result = completion['result']
        require(result['finished'] and result['epoch']==3 and result['step']==1521 and completion['metrics']['n_query']==1796, 'Incomplete pilot budget or evaluator coverage')
        verify_receipts({str(ROOT/branch/'best_evaluation'/name):sha for name,sha in completion['file_receipts'].items()})
        verified_budget[branch] = {'epochs':3,'steps':1521,'best_epoch':completion['best_epoch'],'Mean':completion['metrics']['Mean'],'training_wall_seconds':result['timing']['training_wall_seconds'],'receipt_readback':True}
    ABCD_STATS = saved_rows.groupby('variant').Mean.agg(mean='mean',sample_std=lambda x:x.std(ddof=1),n='count').reset_index()
    require((ABCD_STATS.n==3).all(), 'Statistics require three real seeds for every variant')
    write_csv(ROOT/'abcd_statistics.csv',ABCD_STATS)
    write_json(ROOT/'abcd_statistics.json',{'scope':'development pilot; independently selected best epoch per seed; sample std ddof=1; not final benchmark','statistics':ABCD_STATS.to_dict('records')})
    FINAL_VERIFICATION = {'passed':True,'seed':44,'phase':'development-pilot','training_enabled':False,'train_rows':len(DATA['train']),'validation_rows':len(DATA['validation']),'gallery_rows':len(DATA['gallery']),'preflight_attempt':PREFLIGHT['attempt'],'seed44':verified_budget,'frozen_source_weights_unchanged':True,'historical_42_43_readonly_compatibility':read_json(ROOT/'historical_compatibility.json'),'three_seed_statistics':saved_stats,'source_training_cost_included_in_MR_seconds':False,'scope':'MR-only incremental training cost; excludes frozen encoder/Adapter training and preflight/cache/evaluation costs'}
    write_json(ROOT/'final_verification.json',FINAL_VERIFICATION)
    require(read_json(ROOT/'final_verification.json')['passed'], 'Final report readback failed')
    from IPython.display import display
    display(ABCD_STATS)
    print('Verified: three seeds, twelve A/B/C/D results; C/D44 each 3 epochs /1521 steps; full1796 validation; frozen source hashes unchanged. Training disabled.')
    print('Artifacts:', ROOT)
else:
    print('Historical pilot validation/reporting is disabled by default.')


## Final MR và benchmark official-val

Phase riêng: kiểm chứng pilot → khóa epoch chung từ metric từng epoch thật → fresh MR trên18.000 official-train, frozen FINAL encoder/Adapter cùng seed → chỉ sau đủ ba seed, benchmark ghép cặp official-val. Artifact pilot giữ nguyên. Cờ mặc định tắt; output thực thi được gắn nhãn theo lượt được cho phép.


In [ ]:
# Final MR is opt-in; this extension never calls pilot training.
FINAL_CFG = {'phase': 'preflight',
 'enable_lock': False,
 'enable_training': False,
 'enable_benchmark': False,
 'seeds': [42, 43, 44],
 'campaign_id': 'adapter_mr_final_v1',
 'output_root': '/content/drive/MyDrive/ĐATN/data/processed/adapter_mr_final/FashionIQ',
 'local_root': '/content/visfit_adapter_mr_final',
 'preparation_root': '/content/drive/MyDrive/ĐATN/data/processed/benchmark_preparation/FashionIQ/fashioniq_prep_v1_20261103',
 'pilot_root': '/content/drive/MyDrive/ĐATN/data/processed/adapter_mr_pilot/FashionIQ',
 'source_full_root': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1',
 'min_local_free_gib': 15,
 'min_drive_free_gib': 4,
 'io_workers': 8,
 'cache_shard_size': 128,
 'eval_batch_size': 32,
 'export_top_k': 50}
FINAL_SOURCES = {42: {'encoder': {'encoder_checkpoint_sha256': None,
                  'epochs': 3,
                  'format': 'native',
                  'kind': 'encoder',
                  'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/final/seed_42/encoder/final.pt',
                  'phase': 'final',
                  'seed': 42,
                  'sha256': 'b63517597e4adfd0e863c75795fb675dcedd80bd64f59131a367d0d704c0e780',
                  'signature': '2e4215e5c984b0ea50330759e1da760adf50cace67e86de46715a73a983010fe'},
      'adapter': {'encoder_checkpoint_sha256': 'b63517597e4adfd0e863c75795fb675dcedd80bd64f59131a367d0d704c0e780',
                  'epochs': 9,
                  'format': 'native',
                  'kind': 'adapter',
                  'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/final/seed_42/adapter/final.pt',
                  'phase': 'final',
                  'seed': 42,
                  'sha256': '03dec75cee1045f0c65fafeb880b148f85ee0769728704136142c8001a308221',
                  'signature': 'f290d9066e6fcde04fd6a0d076a3d76fb78229a4a06b5e4315a8c73c8f3abfd6'},
      'complete': {'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/final/seed_42/training_complete.json',
                   'sha256': '836667318be265dcff1ae32479c2456e87413c819bba027f7aa57bc735cfc32a'},
      'final_lock_fingerprint': '44489b7638dbb8c05c78e49840c4153115b8753305dde4be32a641f26ae00fc4',
      'encoder_record': {'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/final/seed_42/encoder/result.json',
                         'sha256': 'b09617ce89a479c6ade47ca40f8948e1165d9be938a18adaa08f2f4b15ad86d9'},
      'adapter_record': {'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/final/seed_42/adapter/result.json',
                         'sha256': 'd3fe4352683e21927a4a132e1ed2a3431e80999ede8aab3818fbf27d80a2f561'},
      'final_text': {'root': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/text_cache/8286a7e9afe3bf86d2a06b8d43f735db5979d62af3a8e5a301bcb3445cfb42dc',
                     'fingerprint': '8286a7e9afe3bf86d2a06b8d43f735db5979d62af3a8e5a301bcb3445cfb42dc',
                     'index_sha256': '80f316a034fc49b9540e653e1dc6761fce34ae5de68217d42f6cdc184d2c5db9',
                     'contract': {'cache_shard_size': 128,
                                  'caption_policy': 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1',
                                  'encode_batch_size': 16,
                                  'encoder_checkpoint_sha256': 'b63517597e4adfd0e863c75795fb675dcedd80bd64f59131a367d0d704c0e780',
                                  'encoder_weights_sha256': '5fd50e11caa9e8328584790750dc8a0c287dde2e7225542fe72f2d97dd92f2e8',
                                  'max_length': 77,
                                  'phase': 'final',
                                  'precision': 'float32',
                                  'runtime': {'amp': False,
                                              'attention': 'eager',
                                              'cuda': '13.0',
                                              'device': 'cuda',
                                              'dtype': 'float32',
                                              'gpu': 'Tesla T4',
                                              'packages': {'Pillow': '11.3.0',
                                                           'huggingface-hub': '0.36.0',
                                                           'numpy': '2.4.6',
                                                           'pandas': '2.2.3',
                                                           'tokenizers': '0.22.1',
                                                           'torch': '2.14.1+cu130',
                                                           'transformers': '4.57.1'},
                                              'python': '3.13.16',
                                              'tf32': False},
                                  'schema': 'trained-conventional-finalLN-padded-zero-v1',
                                  'train_sha256': '539c53d61e9b3d01bf010d089ffa16cef2cd437229feee4a475884586188fd11',
                                  'validation_sha256': None}},
      'benchmark_text': {'root': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/text_cache/4f55adcf14c2eb13711f2776c6b57aedb12717ab977e093379c614f631b8ab0d',
                         'fingerprint': '4f55adcf14c2eb13711f2776c6b57aedb12717ab977e093379c614f631b8ab0d',
                         'index_sha256': '47fb662c206efda9059d8592f2f417259262f133191c913cdc054924bfbda62f',
                         'contract': {'cache_shard_size': 128,
                                      'caption_policy': 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1',
                                      'encode_batch_size': 16,
                                      'encoder_checkpoint_sha256': 'b63517597e4adfd0e863c75795fb675dcedd80bd64f59131a367d0d704c0e780',
                                      'encoder_weights_sha256': '5fd50e11caa9e8328584790750dc8a0c287dde2e7225542fe72f2d97dd92f2e8',
                                      'max_length': 77,
                                      'phase': 'benchmark',
                                      'precision': 'float32',
                                      'runtime': {'amp': False,
                                                  'attention': 'eager',
                                                  'cuda': '13.0',
                                                  'device': 'cuda',
                                                  'dtype': 'float32',
                                                  'gpu': 'Tesla T4',
                                                  'packages': {'Pillow': '11.3.0',
                                                               'huggingface-hub': '0.36.0',
                                                               'numpy': '2.4.6',
                                                               'pandas': '2.2.3',
                                                               'tokenizers': '0.22.1',
                                                               'torch': '2.14.1+cu130',
                                                               'transformers': '4.57.1'},
                                                  'python': '3.13.16',
                                                  'tf32': False},
                                      'schema': 'trained-conventional-finalLN-padded-zero-v1',
                                      'train_sha256': None,
                                      'validation_sha256': '79ea1887f7d5cfd85174754833e4fe2bb6bb74c61c5dd49563cb03b58d9b327c'}},
      'benchmark_source': {'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/benchmark/seed_42/adapter/result.json',
                           'sha256': '18b9a8736cfedff7a997919ec70824cdc7c1a607eeba62700e60e71f70fdc3fe',
                           'metrics': {'Mean': 37.12998074384155,
                                       'gallery_counts': {'dress': 3817, 'shirt': 6346, 'toptee': 5373},
                                       'n_query': 6016,
                                       'overall': {'R@1': 1.5962062509706552,
                                                   'R@10': 25.5683119048207,
                                                   'R@5': 17.50886660058536,
                                                   'R@50': 48.6916495828624},
                                       'per_category': {'dress': {'R@1': 1.7848289538919186,
                                                                  'R@10': 25.037183936539414,
                                                                  'R@5': 17.00545364402578,
                                                                  'R@50': 49.18195339613287},
                                                        'shirt': {'R@1': 1.422963689892051,
                                                                  'R@10': 23.11089303238469,
                                                                  'R@5': 16.143277723258095,
                                                                  'R@50': 44.062806673209025},
                                                        'toptee': {'R@1': 1.580826109127996,
                                                                   'R@10': 28.556858745537987,
                                                                   'R@5': 19.37786843447221,
                                                                   'R@50': 52.83018867924528}},
                                       'query_counts': {'dress': 2017, 'shirt': 2038, 'toptee': 1961},
                                       'scope': 'benchmark',
                                       'unit': 'percent'},
                           'prediction_path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/benchmark/seed_42/adapter/per_query_results.csv',
                           'prediction_sha256': '3ca6ba53520ce039c6f0aeb88b7c2d2cba80f3bead6a864a3430fc811b0162ff',
                           'comparison_policy': 'exact source per-query ranks and original metric_equal atol<1e-10; no relaxed rerun tolerance '
                                                'exists'}},
 43: {'encoder': {'encoder_checkpoint_sha256': None,
                  'epochs': 3,
                  'format': 'native',
                  'kind': 'encoder',
                  'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/final/seed_43/encoder/final.pt',
                  'phase': 'final',
                  'seed': 43,
                  'sha256': '3ec938e0c6056cb9f074a4d4d431d2f22417d4435973eb9684351e07d2adc643',
                  'signature': '15d25f0898cbf42cd4b0820b8fcc91547c81a37940af5cee00ddb9b11c762e56'},
      'adapter': {'encoder_checkpoint_sha256': '3ec938e0c6056cb9f074a4d4d431d2f22417d4435973eb9684351e07d2adc643',
                  'epochs': 9,
                  'format': 'native',
                  'kind': 'adapter',
                  'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/final/seed_43/adapter/final.pt',
                  'phase': 'final',
                  'seed': 43,
                  'sha256': '518854f70ecf4fc6f3761519100b2b93e6e100d5e46e8b4ab445c2a1d35bc860',
                  'signature': '90c6153152d4f24a2aa793376f997c49837223cb4b03bd8b52c0e2988ac1ca6a'},
      'complete': {'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/final/seed_43/training_complete.json',
                   'sha256': '66089f5926365a0ac75e76e1f5e6c0805a3ae50d89fe56f889880bcecbe0bc43'},
      'final_lock_fingerprint': '44489b7638dbb8c05c78e49840c4153115b8753305dde4be32a641f26ae00fc4',
      'encoder_record': {'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/final/seed_43/encoder/result.json',
                         'sha256': 'c5b3eb7293875f8f2d78d51bc14b0dd637ff7fd1a251dd228b7388c9c91c49e5'},
      'adapter_record': {'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/final/seed_43/adapter/result.json',
                         'sha256': '74bb194f943d24ad73f09b33be3f75b9ada7f75021bef496a6a40b70d3f56497'},
      'final_text': {'root': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/text_cache/f24933ddf6faf6e0ffe4674c29b5d8d17610db720d8ec1524389c9de62456b7d',
                     'fingerprint': 'f24933ddf6faf6e0ffe4674c29b5d8d17610db720d8ec1524389c9de62456b7d',
                     'index_sha256': '58252b4b42050fb373a8f6cbbdda08e283d771f809a2bf3986fc584d148845de',
                     'contract': {'cache_shard_size': 128,
                                  'caption_policy': 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1',
                                  'encode_batch_size': 16,
                                  'encoder_checkpoint_sha256': '3ec938e0c6056cb9f074a4d4d431d2f22417d4435973eb9684351e07d2adc643',
                                  'encoder_weights_sha256': 'ef7544ef66b844b911bb002ff2586b2ac9e4cdc526b8bc34530fa33fb9ff64ed',
                                  'max_length': 77,
                                  'phase': 'final',
                                  'precision': 'float32',
                                  'runtime': {'amp': False,
                                              'attention': 'eager',
                                              'cuda': '13.0',
                                              'device': 'cuda',
                                              'dtype': 'float32',
                                              'gpu': 'Tesla T4',
                                              'packages': {'Pillow': '11.3.0',
                                                           'huggingface-hub': '0.36.0',
                                                           'numpy': '2.4.6',
                                                           'pandas': '2.2.3',
                                                           'tokenizers': '0.22.1',
                                                           'torch': '2.14.1+cu130',
                                                           'transformers': '4.57.1'},
                                              'python': '3.13.16',
                                              'tf32': False},
                                  'schema': 'trained-conventional-finalLN-padded-zero-v1',
                                  'train_sha256': '539c53d61e9b3d01bf010d089ffa16cef2cd437229feee4a475884586188fd11',
                                  'validation_sha256': None}},
      'benchmark_text': {'root': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/text_cache/d1abe30c129240ec0d0a642b90c717a180cf1c39c49df9e2a6d6e6a80f7bf670',
                         'fingerprint': 'd1abe30c129240ec0d0a642b90c717a180cf1c39c49df9e2a6d6e6a80f7bf670',
                         'index_sha256': '9805ef1da9867c8625572b6b8e6c3f43b3a2c7f3096320bce6c08a4c820ec34a',
                         'contract': {'cache_shard_size': 128,
                                      'caption_policy': 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1',
                                      'encode_batch_size': 16,
                                      'encoder_checkpoint_sha256': '3ec938e0c6056cb9f074a4d4d431d2f22417d4435973eb9684351e07d2adc643',
                                      'encoder_weights_sha256': 'ef7544ef66b844b911bb002ff2586b2ac9e4cdc526b8bc34530fa33fb9ff64ed',
                                      'max_length': 77,
                                      'phase': 'benchmark',
                                      'precision': 'float32',
                                      'runtime': {'amp': False,
                                                  'attention': 'eager',
                                                  'cuda': '13.0',
                                                  'device': 'cuda',
                                                  'dtype': 'float32',
                                                  'gpu': 'Tesla T4',
                                                  'packages': {'Pillow': '11.3.0',
                                                               'huggingface-hub': '0.36.0',
                                                               'numpy': '2.4.6',
                                                               'pandas': '2.2.3',
                                                               'tokenizers': '0.22.1',
                                                               'torch': '2.14.1+cu130',
                                                               'transformers': '4.57.1'},
                                                  'python': '3.13.16',
                                                  'tf32': False},
                                      'schema': 'trained-conventional-finalLN-padded-zero-v1',
                                      'train_sha256': None,
                                      'validation_sha256': '79ea1887f7d5cfd85174754833e4fe2bb6bb74c61c5dd49563cb03b58d9b327c'}},
      'benchmark_source': {'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/benchmark/seed_43/adapter/result.json',
                           'sha256': 'a04c608d234828f1387c2c1c0f9c03a2f278d4755711cb05964c42f7978f8d97',
                           'metrics': {'Mean': 36.74160651922693,
                                       'gallery_counts': {'dress': 3817, 'shirt': 6346, 'toptee': 5373},
                                       'n_query': 6016,
                                       'overall': {'R@1': 1.4323990517797967,
                                                   'R@10': 25.267155962421793,
                                                   'R@5': 17.73956671854325,
                                                   'R@50': 48.216057076032065},
                                       'per_category': {'dress': {'R@1': 1.338621715418939,
                                                                  'R@10': 24.293505205751114,
                                                                  'R@5': 16.757560733763015,
                                                                  'R@50': 48.53743182944968},
                                                        'shirt': {'R@1': 1.2757605495583906,
                                                                  'R@10': 24.582924435721296,
                                                                  'R@5': 16.8792934249264,
                                                                  'R@50': 44.50441609421001},
                                                        'toptee': {'R@1': 1.6828148903620603,
                                                                   'R@10': 26.925038245792965,
                                                                   'R@5': 19.581845996940338,
                                                                   'R@50': 51.60632330443651}},
                                       'query_counts': {'dress': 2017, 'shirt': 2038, 'toptee': 1961},
                                       'scope': 'benchmark',
                                       'unit': 'percent'},
                           'prediction_path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/benchmark/seed_43/adapter/per_query_results.csv',
                           'prediction_sha256': 'c228d2a724fc98610190c2dd1f815d3f411c95f963b5c38d9f0c3903e65e6a6c',
                           'comparison_policy': 'exact source per-query ranks and original metric_equal atol<1e-10; no relaxed rerun tolerance '
                                                'exists'}},
 44: {'encoder': {'encoder_checkpoint_sha256': None,
                  'epochs': 3,
                  'format': 'native',
                  'kind': 'encoder',
                  'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/final/seed_44/encoder/final.pt',
                  'phase': 'final',
                  'seed': 44,
                  'sha256': '4d0dd01483f5c76e2250a5b03e0c814e05502318fc9fbf28ef286318ae4ccb96',
                  'signature': '7c3a5dc28224f5dbf2cfeb3a0e373f8d3a3f2d0d5c5e5b880c5e2b2b7483955d'},
      'adapter': {'encoder_checkpoint_sha256': '4d0dd01483f5c76e2250a5b03e0c814e05502318fc9fbf28ef286318ae4ccb96',
                  'epochs': 9,
                  'format': 'native',
                  'kind': 'adapter',
                  'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/final/seed_44/adapter/final.pt',
                  'phase': 'final',
                  'seed': 44,
                  'sha256': 'e878ba94685120b7d3dc024c8ac1918ab590f13c47eb730b472b6a6e9d93a5b3',
                  'signature': '5c77c353ca0aa0f462494da5127a27de771660533a2f3cb52de984edb810e2cb'},
      'complete': {'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/final/seed_44/training_complete.json',
                   'sha256': 'bc86e5f7fd8ca3ddf69a9db4aa5d29780998207342bbcb6faedcfaa73a6ed244'},
      'final_lock_fingerprint': '44489b7638dbb8c05c78e49840c4153115b8753305dde4be32a641f26ae00fc4',
      'encoder_record': {'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/final/seed_44/encoder/result.json',
                         'sha256': '87003d3d175729d320fd6d0f8b3f8e7ea7f3904c992eb4b3341632db0058f96f'},
      'adapter_record': {'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/final/seed_44/adapter/result.json',
                         'sha256': '8912a27b5e41356057db997a46b10c36e979fe1a8540f712d407251e6e3d5c75'},
      'final_text': {'root': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/text_cache/9d502bae98bd1cfccc7ae8bc393d7dfd0dae1130479d32aae184b1f8db57b63f',
                     'fingerprint': '9d502bae98bd1cfccc7ae8bc393d7dfd0dae1130479d32aae184b1f8db57b63f',
                     'index_sha256': 'b5bcd9c516c9aec40d81622fca1e4bd4d6527764cc294a6d45228c856d858666',
                     'contract': {'cache_shard_size': 128,
                                  'caption_policy': 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1',
                                  'encode_batch_size': 16,
                                  'encoder_checkpoint_sha256': '4d0dd01483f5c76e2250a5b03e0c814e05502318fc9fbf28ef286318ae4ccb96',
                                  'encoder_weights_sha256': '2be66729dea4f07e23729ce2fe43c348d15ffe151aabce3730e074cae082b40f',
                                  'max_length': 77,
                                  'phase': 'final',
                                  'precision': 'float32',
                                  'runtime': {'amp': False,
                                              'attention': 'eager',
                                              'cuda': '13.0',
                                              'device': 'cuda',
                                              'dtype': 'float32',
                                              'gpu': 'Tesla T4',
                                              'packages': {'Pillow': '11.3.0',
                                                           'huggingface-hub': '0.36.0',
                                                           'numpy': '2.4.6',
                                                           'pandas': '2.2.3',
                                                           'tokenizers': '0.22.1',
                                                           'torch': '2.14.1+cu130',
                                                           'transformers': '4.57.1'},
                                              'python': '3.13.16',
                                              'tf32': False},
                                  'schema': 'trained-conventional-finalLN-padded-zero-v1',
                                  'train_sha256': '539c53d61e9b3d01bf010d089ffa16cef2cd437229feee4a475884586188fd11',
                                  'validation_sha256': None}},
      'benchmark_text': {'root': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/text_cache/6aa15254d5cfe9b1ac51aabeca7743b746dc9c0f2aa76ad492b9e929450e65db',
                         'fingerprint': '6aa15254d5cfe9b1ac51aabeca7743b746dc9c0f2aa76ad492b9e929450e65db',
                         'index_sha256': 'eb4218a1ecb486369f8116347ca6b8751ba065b0792f2c85859e3d62deb73a8f',
                         'contract': {'cache_shard_size': 128,
                                      'caption_policy': 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1',
                                      'encode_batch_size': 16,
                                      'encoder_checkpoint_sha256': '4d0dd01483f5c76e2250a5b03e0c814e05502318fc9fbf28ef286318ae4ccb96',
                                      'encoder_weights_sha256': '2be66729dea4f07e23729ce2fe43c348d15ffe151aabce3730e074cae082b40f',
                                      'max_length': 77,
                                      'phase': 'benchmark',
                                      'precision': 'float32',
                                      'runtime': {'amp': False,
                                                  'attention': 'eager',
                                                  'cuda': '13.0',
                                                  'device': 'cuda',
                                                  'dtype': 'float32',
                                                  'gpu': 'Tesla T4',
                                                  'packages': {'Pillow': '11.3.0',
                                                               'huggingface-hub': '0.36.0',
                                                               'numpy': '2.4.6',
                                                               'pandas': '2.2.3',
                                                               'tokenizers': '0.22.1',
                                                               'torch': '2.14.1+cu130',
                                                               'transformers': '4.57.1'},
                                                  'python': '3.13.16',
                                                  'tf32': False},
                                      'schema': 'trained-conventional-finalLN-padded-zero-v1',
                                      'train_sha256': None,
                                      'validation_sha256': '79ea1887f7d5cfd85174754833e4fe2bb6bb74c61c5dd49563cb03b58d9b327c'}},
      'benchmark_source': {'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/benchmark/seed_44/adapter/result.json',
                           'sha256': 'f55ac7f142bf71769b2f961598a61000d6b1b7b4a7bae3e3af588542339a6f61',
                           'metrics': {'Mean': 37.35287444848488,
                                       'gallery_counts': {'dress': 3817, 'shirt': 6346, 'toptee': 5373},
                                       'n_query': 6016,
                                       'overall': {'R@1': 1.6138466069004922,
                                                   'R@10': 26.242100557327092,
                                                   'R@5': 18.113944026979834,
                                                   'R@50': 48.463648339642674},
                                       'per_category': {'dress': {'R@1': 1.7848289538919186,
                                                                  'R@10': 25.780862667327714,
                                                                  'R@5': 17.253346554288548,
                                                                  'R@50': 49.380267724343085},
                                                        'shirt': {'R@1': 1.3738959764474974,
                                                                  'R@10': 24.33758586849853,
                                                                  'R@5': 16.48675171736997,
                                                                  'R@50': 44.45534838076546},
                                                        'toptee': {'R@1': 1.6828148903620603,
                                                                   'R@10': 28.607853136155022,
                                                                   'R@5': 20.60173380928098,
                                                                   'R@50': 51.55532891381947}},
                                       'query_counts': {'dress': 2017, 'shirt': 2038, 'toptee': 1961},
                                       'scope': 'benchmark',
                                       'unit': 'percent'},
                           'prediction_path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/benchmark/seed_44/adapter/per_query_results.csv',
                           'prediction_sha256': 'd06d75ce39ccdaa85f2b66d4c3907edcc98305af5a664bb12d65fc4efdf1bcd4',
                           'comparison_policy': 'exact source per-query ranks and original metric_equal atol<1e-10; no relaxed rerun tolerance '
                                                'exists'}}}
SOURCE_FINAL_LOCK = {'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/final_training_lock.json', 'sha256': '527b5234888d80a71fe0b4d313c2e1668665e7c65d5cd833f501b906b4db268b', 'fingerprint': '44489b7638dbb8c05c78e49840c4153115b8753305dde4be32a641f26ae00fc4'}
PILOT_MODEL_SHA256 = '6f91b9db044a476182267d3055e74dda9d0eb20fe48c514fb53acb427f30d668'
PILOT_RUNNER_SHA256 = '001f14211d046d048b33d2977279918c5d3f71ad942eb998118b2203eaccebe3'
PILOT_CFG_SNAPSHOT = copy.deepcopy(CFG)

RUN_FINAL_DIAGNOSTICS = False


In [ ]:
FINAL_MR_DATA_EXTENSION_SHA256 = 'cc8a0f5237633986d40fa81813fcfc9649636e2ed6595b03c45f441989860037'
FINAL_PREPARATION_ANCHORS = {'data_audit.json': 'fbaa7eae7b99c5e5aeeddcb0bc6027d3870e8c4f9be5658559caa95662321e95',
 'development_gallery.csv': '83c08f14b606227c1a406b5074163ef70e42eb025144d19e2b1de4f38d234dc5',
 'development_train.csv': '465f758604249e2005cc58c0dfa79edf236081613e908c1e5942fd61abf17086',
 'development_val.csv': 'ace52e94935adacfd19aeaea7a99314602954bce9bff43458ceb564fdf05a8b5',
 'extraction_contract.json': 'bbd53b94172aa5c07a358a0697163c620a823b297c6fec6209167bdb59d1f0a1',
 'feature_metadata.json': '48ee1866dc22f07fa588ea815318ddc9995cfa8734ffd0f2acc88c430f855342',
 'feature_views/development_gallery.csv': '283199425c5fc88fb8e063ea28be629e611e44513689e97f22ee3d0ec5a33927',
 'feature_views/development_train.csv': '6f7a5e6ab8792dd8fd08d6fea8c472a9d9b56b59af38ba146467d797f8a98ff0',
 'feature_views/development_val.csv': 'cbd8d7b72fa7fb4e3c38d64f3bb8ceeeba73be946e9918c5a77c856a42217619',
 'official_train_queries.csv': '539c53d61e9b3d01bf010d089ffa16cef2cd437229feee4a475884586188fd11',
 'official_val_gallery.csv': '58e274df22b52ed5eb3dbc3428ad2966f56b2fb6202cabc0c194f53e2bb8a3d9',
 'official_val_queries.csv': '79ea1887f7d5cfd85174754833e4fe2bb6bb74c61c5dd49563cb03b58d9b327c',
 'protocol_config.json': '2fa2322dee2ebe9964e4ce63ebf0cd8dabad83392a39e288d0e43334e40faffb',
 'training_plan.json': 'e5496feae33ab0e9dc832e29a1a9097b2d7fed6c2683bd9d4314d6e4973bdc21'}
# Final extension: separate roots, immutable lock, no pilot writes.
from concurrent.futures import ThreadPoolExecutor
import gc

def fm_root():
    return Path(FINAL_CFG['output_root'])/FINAL_CFG['campaign_id']

def fm_recipe():
    return {'architecture':'pilot ResidualMRCore/ModifierGuidedRefinement unchanged','hidden':256,'alpha':1.0,'trainable_parameters':262913,'query':'unit(text_embedding + unit(reference_pool + mean(MR(V,guidance)-V)))','loss':'pilot symmetric multi-positive InfoNCE by target image ID','hp':copy.deepcopy(PILOT_HP),'optimizer':{'name':'AdamW','betas':[0.9,0.999],'eps':1e-8},'precision':'float32','scheduler':None,'scaler':None,'model_definition_sha256':PILOT_MODEL_SHA256,'runner_definition_sha256':PILOT_RUNNER_SHA256,'caption_policy':CAPTION_POLICY,'reference_policy':'retain in gallery (FashionIQ)','seed_scope':[42,43,44]}

@contextlib.contextmanager
def fm_scope(seed=42):
    global ROOT,LOCAL,DEVICE,CFG
    previous={k:globals().get(k) for k in ('ROOT','LOCAL','DEVICE','CFG')}
    ROOT=fm_root();LOCAL=Path(FINAL_CFG['local_root'])/FINAL_CFG['campaign_id'];DEVICE=torch.device('cuda')
    CFG=copy.deepcopy(PILOT_CFG_SNAPSHOT);CFG.update(seed=seed,eval_batch_size=32,export_top_k=50)
    try:
        require(FINAL_CFG['seeds']==[42,43,44] and CFG['hp']==PILOT_HP,'Final recipe/seed scope changed')
        ROOT.mkdir(parents=True,exist_ok=True);LOCAL.mkdir(parents=True,exist_ok=True)
        yield
    finally:
        ROOT,LOCAL,DEVICE,CFG=[previous[k] for k in ('ROOT','LOCAL','DEVICE','CFG')]

class FinalMRPreparation(Preparation):
    def read(self,name):
        if name not in ('official_val_queries','official_val_gallery'):return super().read(name)
        if name in self.frames:return self.frames[name]
        path=safe_child(self.root,name+'.csv');require(file_hash(path)==FINAL_PREPARATION_ANCHORS[name+'.csv'],'Official-val manifest SHA differs: '+name)
        f=csv_frame(path);view_spec=self.meta['feature_views'][name];view_path=safe_child(self.root,view_spec['path'])
        require(file_hash(view_path)==view_spec['sha256'],'Official-val feature view SHA differs')
        view=csv_frame(view_path);require(len(f)==view_spec['rows'] and f.equals(view[f.columns]),'Official-val feature view ID/row alignment differs')
        if name=='official_val_queries':
            require(f.query_id.is_unique and set(f.official_split)=={'val'},'Official-val query IDs/split invalid');warnings=[]
            for row in f.itertuples():
                require(row.category in CATEGORIES and row.source_file==f'cap.{row.category}.val.json' and row.query_id==f'{row.category}:val:{int(row.source_row):06d}','Official annotation/source mapping invalid')
                for image_id in (row.reference_image_id,row.target_image_id):require(bool(image_id) and image_id==image_id.strip() and '/' not in image_id and chr(92) not in image_id,'Invalid official-val image ID')
                error,warning=assess_captions([row.caption_1,row.caption_2]);require(error is None,'Official caption blocked: '+row.query_id)
                require(row.modification_text==caption_text(row.caption_1,row.caption_2) and row.triplet_key_sha256==fingerprint([row.category,row.reference_image_id,row.target_image_id,row.caption_1,row.caption_2]),'Official caption policy/triplet changed')
                if warning:warnings.append({'query_id':row.query_id,'reason':warning})
            counts=self.protocol['actual_counts']['official_val'];require(f.groupby('category').size().to_dict()==counts,'Official annotation counts differ')
            for c in CATEGORIES:require(sorted(f[f.category==c].source_row.astype(int))==list(range(counts[c])),'Official annotation rows missing/duplicated')
            write_json(ROOT/'data_audit'/(name+'_warnings.json'),warnings)
        else:
            f['gallery_index']=f.gallery_index.astype(int)
            for c in CATEGORIES:
                g=f[f.category==c];require(g.image_id.is_unique and g.gallery_index.tolist()==list(range(len(g))),'Official gallery order/IDs differ')
                require(fingerprint(g.image_id.tolist())==self.protocol['id_order_sha256']['official_val_gallery'][c],'Official image_splits order differs')
        self.frames[name]=f;self.views[name]=view;self.hashes[name]=file_hash(path);return f

    def data(self,phase):
        require(phase in ('development','final','benchmark'),'Unknown final extension phase')
        names={'development':('development_train','development_val','development_gallery'),'final':('official_train_queries',None,None),'benchmark':(None,'official_val_queries','official_val_gallery')}[phase]
        frames=[self.read(name) if name else None for name in names]
        for name in names:
            if name:require(frame_sha(self.frames[name])==FINAL_PREPARATION_ANCHORS[name+'.csv'],'Manifest changed: '+name)
        train,val,gallery=frames
        if phase=='final':require(len(train)==18000 and set(train.official_split)=={'train'},'Final must use18000 official-train only')
        if phase=='benchmark':
            require(len(val)==6016 and gallery.groupby('category').size().to_dict()=={'dress':3817,'shirt':6346,'toptee':5373},'Incomplete official-val/gallery')
            require(set(val.query_id).isdisjoint(self.frames['official_train_queries'].query_id),'Official query leakage')
        if val is not None:
            for c in CATEGORIES:require(set(val[val.category==c].target_image_id)<=set(gallery[gallery.category==c].image_id),'Missing target coverage')
        binding=copy.deepcopy(self.binding)
        if phase=='benchmark':binding['anchors']=FINAL_PREPARATION_ANCHORS
        return {'phase':phase,'train':train,'validation':val,'gallery':gallery,'train_sha256':frame_sha(train) if train is not None else None,'validation_sha256':frame_sha(val) if val is not None else None,'gallery_sha256':frame_sha(gallery) if gallery is not None else None,'package':binding,'views':{role:self.views[name] for role,name in zip(('train','validation','gallery'),names) if name}}

def fm_data_id(data):return {k:data[k] for k in ('phase','train_sha256','validation_sha256','gallery_sha256','package')}

def fm_resources():
    require(torch.cuda.is_available(),'GPU required; no real training on local/CPU')
    torch.set_default_dtype(torch.float32);torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False;torch.backends.cudnn.benchmark=False
    require(not torch.is_autocast_enabled(),'FP32 required')
    root=fm_root().resolve()
    for source in (Path(FINAL_CFG['pilot_root']).resolve(),Path(FINAL_CFG['source_full_root']).resolve(),Path(FINAL_CFG['preparation_root']).resolve()):
        require(root!=source and root not in source.parents and source not in root.parents,'Final output overlaps protected source')
    require(shutil.disk_usage(LOCAL).free>=FINAL_CFG['min_local_free_gib']*1024**3,'Local disk insufficient; do not change scientific config')
    from google.colab import auth
    import google.auth
    from google.auth.transport.requests import AuthorizedSession
    auth.authenticate_user();credentials,_=google.auth.default()
    response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',params={'fields':'storageQuota'},timeout=60)
    require(response.ok,'Cannot verify Drive quota; complete account authorization')
    quota=response.json()['storageQuota'];remaining=int(quota['limit'])-int(quota['usage']) if quota.get('limit') else None
    require(remaining is None or remaining>=FINAL_CFG['min_drive_free_gib']*1024**3,'Drive quota insufficient; preserve artifacts and stop')
    write_json(ROOT/'resource_report.json',{'runtime':physical_runtime(),'local_free_bytes':shutil.disk_usage(LOCAL).free,'drive_storage_quota':quota})
    return {'passed':True,'runtime':physical_runtime()}

def fm_pinned(spec):
    require(file_hash(spec['path'])==spec['sha256'],'Source metadata changed: '+spec['path'])
    return read_json(spec['path'])

def fm_source_metadata(seed,package):
    s=FINAL_SOURCES[seed];lock=fm_pinned(SOURCE_FINAL_LOCK)
    require(fingerprint(lock)==SOURCE_FINAL_LOCK['fingerprint'] and not lock['official_val_metric_used'],'Source final lock differs or used official-val')
    require(lock['seeds']==[42,43,44] and lock['encoder_epochs']==3 and lock['adapter_epochs']==9,'Source final training budget/scope differs')
    require(lock['final_train_sha256']==package.hashes['official_train_queries'],'Source final data differs')
    complete=fm_pinned(s['complete']);require(complete['seed']==seed and complete['final_lock_sha256']==fingerprint(lock),'Source final completion seed/lock differs')
    for role in ('encoder','adapter'):
        asset=s[role];record=fm_pinned(s[role+'_record'])
        require(asset==complete[role] and asset==record['asset'] and asset['phase']=='final' and asset['seed']==seed and asset['kind']==role,'Wrong final checkpoint/seed')
        require(record['phase']=='final' and record['seed']==seed and record['lock_sha256']==fingerprint(lock) and record['origin']==('locked_pretrained_fresh' if role=='encoder' else 'fresh_zero_gate_adapter'),'Source origin/lock differs')
        require(record['data']['train_sha256']==lock['final_train_sha256'] and record['data']['validation_sha256'] is None and record['history']==[] and record['metrics'] is None,'Source final training used evaluation or wrong data')
        require(record['finished_epochs']==record['budget_epochs']==lock[role+'_epochs'] and record['recipe']==lock[role+'_recipe'],'Source final recipe/budget differs')
        verify_receipts(record['file_receipts'])
    require(s['adapter']['encoder_checkpoint_sha256']==s['encoder']['sha256'],'Adapter bound to another encoder')
    return s,lock

def fm_lock(package):
    require(FINAL_CFG['enable_lock'] is True,'Explicit lock opt-in required')
    data=package.data('development');proofs={};curves={}
    for seed in (42,43,44):
        validated=verify_historical_seed(seed,data,SOURCE_REGISTRY)
        require(validated is not None and set(validated['metrics'])=={'A','B','C','D'},'Pilot C/D incomplete for seed'+str(seed))
        root=Path(FINAL_CFG['pilot_root'])/f'adapter_mr_pilot_v1_seed{seed}';done=read_json(root/'D/completion.json');binding=done['binding']
        last=committed_checkpoint_load(root/'D/last.pt',fingerprint(binding));required_checkpoint(last,binding)
        history=read_json(root/'D/metric_history.json');require(history==last['metrics'] and len(history)==3,'Epoch history differs from authoritative checkpoint')
        entries={};files={}
        for item in history:
            epoch=item['epoch'];require(epoch not in entries and epoch in (1,2,3),'Duplicate/invalid epoch')
            path=root/'D'/f'epoch_{epoch:02d}_per_query.csv';sha=file_hash(path);require(sha==item['predictions_sha256'],'Epoch per-query receipt mismatch')
            pred=pd.read_csv(path,dtype=str,keep_default_na=False);metric=metrics_from_predictions(pred,data);metric_equal(metric,item['metrics'])
            entries[epoch]=metric['Mean'];files[str(path)]=sha
        require(set(entries)=={1,2,3},'Missing actual epoch metric')
        for path in (root/'D/last.pt',root/'D/last.json',root/'D/completion.json',root/'D/metric_history.json',root/'config.json',root/'source_provenance.json',root/'pilot_cache'/binding['frozen_input_cache']/'index.json'):files[str(path)]=file_hash(path)
        curves[seed]=entries;proofs[seed]={'source_encoder':binding['source_encoder'],'source_adapter':binding['source_adapter'],'signature':last['signature'],'cache_fingerprint':binding['frozen_input_cache'],'files':files,'C_D_readonly_validation':validated['compatibility']}
    common=set.intersection(*(set(x) for x in curves.values()));require(common,'No common real epoch')
    means={epoch:float(np.mean([curves[seed][epoch] for seed in (42,43,44)])) for epoch in sorted(common)}
    chosen=min(means,key=lambda e:(-means[e],e))
    sources={}
    for seed in (42,43,44):
        source,_=fm_source_metadata(seed,package);spec=source['final_text'];index=Path(spec['root'])/'index.json'
        require(file_hash(index)==spec['index_sha256'],'Final text source index changed')
        sources[seed]={'encoder':source['encoder'],'adapter':source['adapter'],'complete':source['complete'],'final_text':spec}
    payload={'schema':'visfit-final-mr-lock-v1','mr_epochs':chosen,'recipe':fm_recipe(),'protocol':package.protocol,'development_data':fm_data_id(data),'development_sources':proofs,'selection':{'policy':'maximum mean actual epoch Dev Mean over all three seeds; earliest exact tie','epoch_means':means,'per_seed_epoch_mean':curves,'common_epochs':sorted(common)},'final_source_lock':SOURCE_FINAL_LOCK,'final_sources':sources,'final_train_sha256':package.hashes['official_train_queries'],'official_val_used_for_selection':False}
    write_json(ROOT/'final_mr_lock.json',payload,immutable=True)
    require(read_json(ROOT/'final_mr_lock.json')==json.loads(json.dumps(payload)),'MR lock readback differs')
    print('MR lock verified: epoch',chosen,'| actual development epoch means',means)
    return read_json(ROOT/'final_mr_lock.json')

def fm_read_lock(package):
    path=ROOT/'final_mr_lock.json';require(path.is_file(),'Final MR lock missing')
    lock=read_json(path);require(lock['schema']=='visfit-final-mr-lock-v1' and lock['recipe']==fm_recipe() and not lock['official_val_used_for_selection'],'MR lock recipe/selection changed')
    require(lock['final_train_sha256']==package.hashes['official_train_queries'] and lock['protocol']==package.protocol,'MR lock protocol/data changed')
    selection=lock['selection'];curves=selection['per_seed_epoch_mean'];common=set.intersection(*(set(c) for c in curves.values()))
    means={int(e):float(np.mean([c[e] for c in curves.values()])) for e in common};require(lock['mr_epochs']==min(means,key=lambda e:(-means[e],e)),'MR epoch lock selection invalid')
    for proof in lock['development_sources'].values():verify_receipts(proof['files'])
    return lock


In [ ]:
# Historical authorized lock output below; safe default restored, not rerun.
RUN_FINAL_LOCK = False
if RUN_FINAL_LOCK:
    with fm_scope(42):
        FM_RESOURCES = fm_resources()
        FM_PACKAGE = FinalMRPreparation(FINAL_CFG['preparation_root'])
        FINAL_CFG['enable_lock'] = True
        try:
            FM_LOCK = fm_lock(FM_PACKAGE)
        finally:
            FINAL_CFG['enable_lock'] = False
else:
    print('Final MR lock is opt-in; no training started.')


In [ ]:
# SHA256 of cache/train/benchmark definitions, excluding this assignment.
FINAL_MR_CODE_SHA256 = '790364add06c0068ee77820ea1a33959b6f811a3b50a0625e5509f0d715795d0'
class FinalReadOnlyCache(ReadonlyCache):
    """Same committed-shard checks as pilot; parallel I/O only, ordered ID mapping unchanged."""
    def __init__(self,root,role,ids,local_root,kind=False):
        self.root=Path(root);self.role=role;self.index=read_json(self.root/'index.json');self.key=self.index['fingerprint'];self.kind=kind
        if kind:
            require(self.index['kind']==role,'Feature kind mismatch');self.ids=self.index['ids'];self.tail=self.index['shape_tail'];self.dtype=self.index['dtype'];self.step=self.index['shard_size']
            require(read_json(self.root.parent/(role+'_id2idx.json'))=={v:i for i,v in enumerate(self.ids)},'Feature row mapping mismatch')
        else:
            require(fingerprint(self.index['contract'])==self.key and self.root.name==self.key,'Cache contract fingerprint mismatch')
            spec=self.index['specs'][role];self.ids=spec['ids'];self.tail=spec['shape'];self.dtype=spec['dtype'];self.step=self.index['shard_size']
        self.map={v:i for i,v in enumerate(self.ids)};require(len(self.map)==len(self.ids) and set(ids)<=set(self.map),'Missing/duplicate feature IDs')
        self.paths={};self.loaded=OrderedDict();self.local=Path(local_root)/self.key/role;self.local.mkdir(parents=True,exist_ok=True)
        def verify(start):
            end=min(start+self.step,len(self.ids));name=f'{start:07d}_{end:07d}';folder=self.root if kind else self.root/role;p=folder/(name+'.npy');marker=read_json(p.with_suffix('.json'))
            expected={'fingerprint':self.key,'kind' if kind else 'role':role,'ids_sha256':fingerprint(self.ids[start:end]),'shape':[end-start,*self.tail],'dtype':self.dtype}
            require(all(marker.get(k)==v for k,v in expected.items()),'Committed shard identity mismatch: '+str(p))
            require(file_hash(p)==marker['sha256'],'Source shard corrupt: '+str(p));dest=self.local/p.name
            if not dest.exists() or file_hash(dest)!=marker['sha256']:
                tmp=dest.with_suffix('.partial.'+uuid.uuid4().hex)
                stream_copy(p,tmp);require(file_hash(tmp)==marker['sha256'],'Local copy corrupt');os.replace(tmp,dest)
            a=np.load(dest,mmap_mode='r',allow_pickle=False);require(list(a.shape)==expected['shape'] and str(a.dtype)==self.dtype and np.isfinite(a).all(),'Invalid shard values/header')
            if role in ('target','target_pooled','reference_pooled'):require(np.allclose(np.linalg.norm(a,axis=-1),1,atol=1e-5,rtol=1e-5),'Non-unit image embedding')
            if role=='reference':require((np.linalg.norm(a.reshape(len(a),-1),axis=-1)>1e-8).all(),'Zero reference feature')
            if role=='mask':require(np.isin(a,[0,1]).all(),'Invalid mask')
            return start,dest
        needed=sorted({self.map[v]//self.step*self.step for v in ids})
        with ThreadPoolExecutor(max_workers=FINAL_CFG['io_workers']) as pool:self.paths=dict(pool.map(verify,needed))
        print('Verified cache',role,'|',len(ids),'IDs |',len(needed),'shards')

_FM_IMAGE_BANKS={}
_FM_BASE=None
_FM_TOKENIZER=None

def fm_base(package):
    global _FM_BASE,_FM_TOKENIZER
    if _FM_BASE is None:_FM_BASE,_FM_TOKENIZER,_=load_pretrained(package)
    require(tensor_hash(_FM_BASE)==EXTRACTION_CONTRACT['encoder_weight_sha256'],'Pinned base weights changed')
    return _FM_BASE,_FM_TOKENIZER

class FinalImageBank:
    def __init__(self,package,base,data):
        self.base=base;self.data=data;self.phase=data['phase'];frame=data['train'] if data['train'] is not None else data['validation']
        refs=sorted(set(frame.reference_image_id));targets=sorted(set(data['train'].target_image_id) if data['train'] is not None else set(data['gallery'].image_id))
        src=package.image_source();root=Path(src['prep'])/'features'/src['meta']['fingerprint'];self.readers={}
        for role,ids,tail in [('reference',refs,[50,768]),('target',targets,[512])]:
            path=root/role/'index.json';require(file_hash(path)==src['meta']['store_index_sha256'][role],'Preparation image index changed')
            index=read_json(path);require(index['fingerprint']==src['meta']['fingerprint'] and index['shape_tail']==tail and index['dtype']=='float32','Image extraction schema mismatch')
            self.readers[role]=FinalReadOnlyCache(root/role,role,ids,LOCAL/'preparation_images',True)
        for role,view in data['views'].items():
            if 'query_id' in view:
                for row in view.itertuples():require(int(row.reference_feature_index)==self.readers['reference'].map[row.reference_image_id] and int(row.target_feature_index)==self.readers['target'].map[row.target_image_id],'Query feature-view ID mapping differs')
            else:
                for row in view.itertuples():require(int(row.target_feature_index)==self.readers['target'].map[row.image_id],'Gallery feature-view ID mapping differs')
        self.refs=set(refs);self.targets=set(targets)
        contract={'schema':'MR-reference-tokens-and-original-CLS-pool-v1','phase':self.phase,'reference_ids_sha256':fingerprint(refs),'extraction_contract':package.contract,'source_reference_index_sha256':src['meta']['store_index_sha256']['reference'],'base_weights':EXTRACTION_CONTRACT['encoder_weight_sha256'],'runtime':physical_runtime(),'projection_batch_size':16,'precision':'float32'}
        specs={'mr_tokens':{'ids':refs,'shape':[50,512],'dtype':'float32'},'reference_pooled':{'ids':refs,'shape':[512],'dtype':'float32'}}
        self.store=TrialStore(ROOT/'cache/vision',LOCAL/'cache/vision',contract,specs,128);self.key=self.store.key
        def compute(ids,role):
            values=[]
            for start in range(0,len(ids),16):
                raw=torch.from_numpy(self.readers['reference'].take(ids[start:start+16])).to(DEVICE)
                with torch.no_grad():
                    v=project_reference_tokens(raw,base.vision_model.post_layernorm,base.visual_projection) if role=='mr_tokens' else unit(base.visual_projection(base.vision_model.post_layernorm(raw[:,0])))
                values.append(v.cpu().numpy())
            return np.concatenate(values)
        for role in specs:self.store.prepare(role,lambda ids,r=role:compute(ids,r))
        write_json(ROOT/'cache'/self.phase/'image_audit.json',{'passed':True,'references':len(refs),'targets_gallery':len(targets),'all_used_shards_verified':True,'image_contract':package.contract,'vision_cache':self.key})
    def take(self,role,ids):
        if role=='reference':require(set(ids)<=self.refs,'Reference outside preflight coverage');return self.readers['reference'].take(ids)
        if role in ('reference_pooled','mr_tokens'):require(set(ids)<=self.refs,'Unprepared reference');return self.store.take(role,ids)
        require(role=='target_pooled' and set(ids)<=self.targets,'Target/gallery outside preflight coverage');return self.readers['target'].take(ids)

def fm_images(package,base,data):
    key=fingerprint(fm_data_id(data))
    if key not in _FM_IMAGE_BANKS:_FM_IMAGE_BANKS[key]=FinalImageBank(package,base,data)
    return _FM_IMAGE_BANKS[key]

class FinalFrozenInputs:
    def __init__(self,seed,phase,data,encoder,adapter,images,tokenizer):
        self.seed=seed;self.data=data;self.encoder=encoder;self.adapter=adapter;self.images=images;self.tokenizer=tokenizer
        self.frame=data['train'] if data['train'] is not None else data['validation'];ids=sorted(self.frame.query_id);source=FINAL_SOURCES[seed];spec=source[phase+'_text'];root=Path(spec['root'])
        require(file_hash(root/'index.json')==spec['index_sha256'],'Pinned FINAL text cache index changed')
        index=read_json(root/'index.json');ct=index['contract'];require(ct==spec['contract'] and fingerprint(ct)==spec['fingerprint'],'Text contract/fingerprint differs')
        require(ct['phase']==phase and ct['encoder_checkpoint_sha256']==source['encoder']['sha256'] and ct['encoder_weights_sha256']==tensor_hash(encoder),'Wrong final text encoder/phase')
        require(ct['caption_policy']==CAPTION_POLICY and ct['precision']=='float32' and ct['train_sha256']==data['train_sha256'] and ct['validation_sha256']==data['validation_sha256'],'Final text caption/data/precision mismatch')
        self.readers={};self.key=spec['fingerprint']
        for role,tail,dtype in [('text',[77,512],'float32'),('mask',[77],'int64'),('eos',[],'int64'),('input_ids',[77],'int64')]:
            entry=index['specs'][role];require(entry['ids']==ids and entry['shape']==tail and entry['dtype']==dtype,'Text cache coverage/schema/IDs differ')
            self.readers[role]=FinalReadOnlyCache(root,role,ids,LOCAL/'source_final_text')
        for start in range(0,len(self.frame),128):
            rows=self.frame.iloc[start:start+128];qids=rows.query_id.tolist();expected=tokenizer(rows.modification_text.tolist(),padding='max_length',max_length=77,truncation=False,return_tensors='np')
            i=self.take('input_ids',qids);m=self.take('mask',qids);e=self.take('eos',qids);h=self.take('text',qids)
            require(np.array_equal(i,expected['input_ids']) and np.array_equal(m,expected['attention_mask']),'Caption/token/mask row mapping differs')
            require(np.array_equal(eos_positions(torch.from_numpy(i),torch.from_numpy(m),tokenizer.eos_token_id).numpy(),e),'Text EOS mapping differs')
            validate_frozen_text(torch.from_numpy(h),torch.from_numpy(m),torch.from_numpy(e))
        native_checks=[]
        for category in CATEGORIES:
            rows=self.frame[self.frame.category==category].iloc[:16];qids=rows.query_id.tolist();mod={k:torch.from_numpy(self.take(role,qids)).to(DEVICE) for k,role in [('input_ids','input_ids'),('attention_mask','mask')]}
            with torch.no_grad():h=encoder.encode_text(mod)[0]
            cached=torch.from_numpy(self.take('text',qids)).to(DEVICE);error=float((h-cached).abs().max().cpu())
            require(torch.allclose(h,cached,atol=NATIVE_ATOL,rtol=NATIVE_RTOL),'Native final text/cache mismatch; diagnose before training')
            native_checks.append({'category':category,'queries':len(rows),'max_abs':error})
        self.audit={'passed':True,'seed':seed,'phase':phase,'query_count':len(ids),'text_fingerprint':self.key,'encoder':source['encoder'],'adapter':source['adapter'],'full_ID_shape_finite_token_mask_EOS_check':True,'native_reencoding_scope':'16 queries/category, not full reencoding','native_checks':native_checks}
    def take(self,role,ids):return self.readers[role].take(ids)
    def batch(self,rows,training=False):
        ids=rows.query_id.tolist();refs=rows.reference_image_id.tolist();tensor=lambda role,ids:torch.from_numpy(self.take(role,ids)).to(DEVICE)
        b={'reference_features':{'tokens':torch.from_numpy(self.images.take('reference',refs)).to(DEVICE),'pooled':torch.from_numpy(self.images.take('reference_pooled',refs)).to(DEVICE)},'text_features':tensor('text',ids),'mask':tensor('mask',ids),'eos':tensor('eos',ids)}
        if training:b.update(target=torch.from_numpy(self.images.take('target_pooled',rows.target_image_id.tolist())).to(DEVICE).detach(),target_ids=rows.target_image_id.tolist())
        return b

class FinalReducedInputs:
    def __init__(self,seed,data,source,images,encoder,adapter):
        self.source=source;self.images=images;frame=source.frame;ids=sorted(frame.query_id)
        asset=FINAL_SOURCES[seed];contract={'schema':'pilot-identical-D-reductions-final-v1','implementation_sha256':FINAL_MR_CODE_SHA256,'phase':data['phase'],'seed':seed,'encoder_sha256':asset['encoder']['sha256'],'adapter_sha256':asset['adapter']['sha256'],'text_index_sha256':asset[data['phase']+'_text']['index_sha256'],'vision_cache':images.key,'data':fm_data_id(data),'recipe':fm_recipe(),'runtime':physical_runtime(),'batch_policy':'original per-category order; FP32 batches32','target_used_in_reduction':False}
        specs={r:{'ids':ids,'shape':[512],'dtype':'float32'} for r in ('D_text_embedding','D_guidance')}
        self.store=TrialStore(ROOT/'cache'/data['phase']/f'seed_{seed}',LOCAL/'cache'/data['phase']/f'seed_{seed}',contract,specs,128);self.key=self.store.key;computed={}
        def compute(qids,role):
            if not computed:
                arrays={r:np.empty((len(ids),512),np.float32) for r in specs};positions={q:i for i,q in enumerate(ids)}
                for c in CATEGORIES:
                    f=frame[frame.category==c]
                    for start in range(0,len(f),32):
                        rows=f.iloc[start:start+32];b=source.batch(rows,False)
                        with torch.no_grad():
                            enhanced=frozen_adapter_text(adapter,b['reference_features'],b['text_features'],b['mask'],b['eos']);reduced=reduce_frozen_text(enhanced,b['mask'],b['eos'],encoder.projection.weight)
                        for r,value in reduced.items():arrays['D_'+r][[positions[q] for q in rows.query_id]]=value.cpu().numpy()
                computed.update(arrays)
            return computed[role][[self.store.maps[role][q] for q in qids]]
        for role in specs:self.store.prepare(role,lambda q,r=role:compute(q,r))
        computed.clear()
    def batch(self,rows,branch='D',training=False):
        require(branch=='D','Only Adapter + MR final is in scope');qids=rows.query_id.tolist();refs=rows.reference_image_id.tolist();take=lambda role:torch.from_numpy(self.store.take('D_'+role,qids)).to(DEVICE)
        b={'reference_features':{'mr_tokens':torch.from_numpy(self.images.take('mr_tokens',refs)).to(DEVICE),'pooled':torch.from_numpy(self.images.take('reference_pooled',refs)).to(DEVICE)},'text_features':{r:take(r) for r in ('text_embedding','guidance')},'mask':None,'eos':None}
        if training:b.update(target=torch.from_numpy(self.images.take('target_pooled',rows.target_image_id.tolist())).to(DEVICE).detach(),target_ids=rows.target_image_id.tolist())
        return b


In [ ]:
def fm_frozen(base,encoder,adapter):
    require(all(not p.requires_grad and p.grad is None for module in (base,encoder,adapter) for p in module.parameters()),'Frozen source is trainable or has gradients')
    return {'base':tensor_hash(base),'encoder':tensor_hash(encoder),'adapter':tensor_hash(adapter)}

def fm_load_models(seed,package):
    source,lock=fm_source_metadata(seed,package)
    progress_proof={}
    for role in ('encoder','adapter'):
        state=validate_asset(source[role]);required_checkpoint(state,state['binding'])
        source_batch=lock[role+'_recipe']['batch_size'];expected_steps=((18000+source_batch-1)//source_batch)*lock[role+'_epochs']
        require(state['finished'] and state['epoch']==lock[role+'_epochs'] and state['cursor']==0 and state['step']==expected_steps,'Source final training progress differs from its own locked batch size/budget')
        require(state['binding']['recipe']==lock[role+'_recipe'],'Checkpoint source recipe differs from final source lock')
        progress_proof[role]={'batch_size':source_batch,'epochs':state['epoch'],'step':state['step'],'expected_steps':expected_steps,'finished':state['finished']}
        require(state['binding']['phase']=='final' and state['binding']['seed']==seed and state['binding']['final_lock_sha256']==fingerprint(lock),'Source checkpoint final lock/seed differs')
        require(state['binding']['data']['train_sha256']==lock['final_train_sha256'] and state['metrics']==[] and state['best_mean'] is None,'Source used wrong training data/validation selection')
        del state
    write_json(ROOT/'final'/f'seed_{seed}'/'source_progress_audit.json',progress_proof)
    base,tokenizer=fm_base(package);encoder=load_encoder_asset(source['encoder'],base,tokenizer).freeze();adapter=load_adapter_asset(source['adapter'],encoder).eval().requires_grad_(False)
    require(torch.equal(adapter.projection_weight,encoder.projection.weight.detach()),'Final projection mismatch')
    fm_frozen(base,encoder,adapter)
    return base,tokenizer,encoder,adapter

def fm_fresh(seed):
    with isolated_rng():
        seed_all(seed);model=CachedPilotMR().to(DEVICE)
    require(sum(p.numel() for p in model.parameters())==262913 and all(n.startswith('mr.') for n,_ in model.named_parameters()),'MR architecture/optimizer scope changed')
    return model

def fm_smoke_final(main,seed,data,reduced):
    original_rng=rng_state();original_hash=tensor_hash(main);initial=cpu_tree(main.state_dict());reports={}
    try:
        with isolated_rng(),tempfile.TemporaryDirectory(prefix='mr_final_smoke_',dir=LOCAL) as temporary:
            tmp=Path(temporary);frame=pd.concat([data['train'][data['train'].category==c].iloc[:32] for c in CATEGORIES]);require(len(frame)==96,'Smoke needs96 official-train rows')
            hp={**PILOT_HP,'checkpoint_every_steps':1};binding={'phase':'final','seed':seed,'kind':'mr_residual_final','fixture_scope':'temporary96 official-train rows; two epochs solely to test epoch/cursor resume, no validation/selection','cache':reduced.key,'ids':fingerprint(frame.query_id.tolist()),'implementation_sha256':FINAL_MR_CODE_SHA256}
            def fresh():
                m=CachedPilotMR().to(DEVICE);m.load_state_dict(initial,strict=True);return m
            reference=fresh();params=dict(reference.named_parameters());gradients={n:[] for n in params};teachers=[]
            hooks=[p.register_hook(lambda g,n=n:gradients[n].append(float(g.detach().square().sum().cpu()))) for n,p in params.items()]
            first=reduced.batch(frame.iloc[:32],'D',True)
            with torch.no_grad():
                q=query_forward(reference,first,'adapter');changed={**first,'target':first['target'].flip(0)}
                require(torch.equal(q,query_forward(reference,changed,'adapter')),'Target enters final query')
                require(query_forward(reference,reduced.batch(frame.iloc[:1],'D',False),'adapter').shape==(1,512),'Batch1 shape')
            def batch(rows,training=True):
                b=reduced.batch(rows,'D',training)
                if training:b['target']=b['target'].detach().clone().requires_grad_();teachers.append(b['target'])
                return b
            full=optimize_stage(reference,frame,batch,None,tmp/'full',tmp/'full_local',binding,hp,2,None)
            expected=checkpoint_load(tmp/'full_local/last.pt',fingerprint(binding));require(full['finished'] and full['epoch']==2 and full['step']==6 and expected['metrics']==[] and expected['best_mean'] is None,'Final smoke budget/selection differs')
            for h in hooks:h.remove()
            require(all(t.grad is None for t in teachers),'Teacher target not detached')
            hidden=[n for n in params if not n.startswith('mr.out.')];heads=[n for n in params if n.startswith('mr.out.')]
            require(all(len(v)==6 and np.isfinite(v).all() for v in gradients.values()),'MR gradient missing/nonfinite')
            require(sum(gradients[n][0] for n in hidden)==0 and sum(gradients[n][0] for n in heads)>0 and sum(gradients[n][1] for n in hidden)>0,'Zero-init gate/hidden gradient flow differs')
            count=0
            def interrupt(rows,training=True):
                nonlocal count
                count+=1
                if count==3:raise RuntimeError('expected temporary final smoke interruption')
                return batch(rows,training)
            try:optimize_stage(fresh(),frame,interrupt,None,tmp/'resume',tmp/'resume_local',binding,hp,2,None)
            except RuntimeError as e:require('expected temporary final smoke interruption' in str(e),'Unexpected final smoke error')
            else:raise RuntimeError('Resume interruption not exercised')
            saved=checkpoint_load(tmp/'resume_local/last.pt',fingerprint(binding));require((saved['epoch'],saved['cursor'],saved['step'])==(0,2,2),'Final smoke cursor mismatch')
            resumed=fresh();optimize_stage(resumed,frame,batch,None,tmp/'resume',tmp/'resume_local',binding,hp,2,None)
            actual=checkpoint_load(tmp/'resume_local/last.pt',fingerprint(binding));required_checkpoint(actual,binding)
            require(same_tree(expected['model'],actual['model']) and same_tree(expected['optimizer'],actual['optimizer']) and same_rng(expected['rng'],actual['rng']),'Final resume model/optimizer/RNG mismatch')
            for name in ('batch_ids_sha256','loss_retrieval','gradient_norm'):require([r[name] for r in expected['history']]==[r[name] for r in actual['history']],'Final resume order/loss/gradient mismatch')
            final=committed_checkpoint_load(tmp/'resume/final.pt',fingerprint(binding));require(same_tree(final['model'],actual['model']) and not (tmp/'resume/best.pt').exists(),'Final checkpoint roles differ')
            reports={'passed':True,'temporary_training_rows':96,'optimizer_steps':6,'resume_epoch_step_cursor_model_optimizer_RNG_exact':True,'teacher_detached':True,'query_target_independent':True,'MR_gradient_stages':gradients,'batch1':True,'official_val_accessed':False,'fixtures':pilot_fixture_checks()}
        return reports
    finally:
        restore_rng(original_rng);require(tensor_hash(main)==original_hash and same_rng(original_rng,rng_state()),'Smoke changed main model/RNG')

def fm_binding(seed,data,reduced,model,lock,source):
    return {'phase':'final','seed':seed,'kind':'mr_residual_final','variant':'D','origin':'fresh MR by seed; no development MR warm-start','implementation_sha256':FINAL_MR_CODE_SHA256,'recipe':fm_recipe(),'data':fm_data_id(data),'mr_epochs':lock['mr_epochs'],'final_mr_lock_sha256':fingerprint(lock),'source_full_final_lock_sha256':source['final_lock_fingerprint'],'source_encoder':source['encoder'],'source_adapter':source['adapter'],'cache_fingerprint':reduced.key,'mr_initial_sha256':tensor_hash(model),'runtime':physical_runtime()}

def fm_verify_training(seed,package,lock):
    folder=ROOT/'final'/f'seed_{seed}';path=folder/'completion.json'
    if not path.is_file():return None
    done=read_json(path);source,_=fm_source_metadata(seed,package);data=package.data('final');binding=done['binding']
    require(done['passed'] and done['completed'] and done['seed']==seed and binding['phase']=='final' and binding['variant']=='D' and binding['kind']=='mr_residual_final','Final MR completion seed/phase differs')
    require(binding['recipe']==fm_recipe() and binding['implementation_sha256']==FINAL_MR_CODE_SHA256 and binding['data']==fm_data_id(data) and binding['final_mr_lock_sha256']==fingerprint(lock),'Final MR completion config/data/code/lock differs')
    require(binding['source_encoder']==source['encoder'] and binding['source_adapter']==source['adapter'] and binding['mr_epochs']==lock['mr_epochs'],'Final MR source/budget differs')
    verify_receipts(done['file_receipts']);last=committed_checkpoint_load(folder/'last.pt',fingerprint(binding));final=committed_checkpoint_load(folder/'final.pt',fingerprint(binding));required_checkpoint(last,binding);required_checkpoint(final,binding)
    budget=lock['mr_epochs'];require(last['finished'] and last['epoch']==budget and last['cursor']==0 and last['step']==budget*563 and len(last['history'])==budget*563,'Final MR training incomplete')
    require(last['metrics']==[] and last['best_mean'] is None and last['best_epoch'] is None and not (folder/'best.pt').exists(),'Final MR used validation selection')
    require(same_tree(last['model'],final['model']),'Last/final weights differ')
    with isolated_rng():
        fresh=fm_fresh(seed);require(tensor_hash(fresh)==last['initial_sha256']==binding['mr_initial_sha256'],'Final MR did not initialize fresh by seed');fresh.load_state_dict(final['model'],strict=True)
        require(all(p.dtype==torch.float32 and bool(torch.isfinite(p).all()) for p in fresh.parameters()),'Final MR weights invalid');del fresh
    expected=[fingerprint(ids) for epoch in range(budget) for ids in epoch_batches(data['train'],32,seed,epoch)]
    require([r['batch_ids_sha256'] for r in last['history']]==expected,'Final MR batch order/coverage differs')
    require(all(np.isfinite(r['loss_retrieval']) and np.isfinite(r['gradient_norm']) for r in last['history']),'Final loss/gradient invalid')
    return done

def fm_train_seed(seed,package,lock):
    require(FINAL_CFG['enable_training'] is True,'Final MR training disabled')
    existing=fm_verify_training(seed,package,lock)
    if existing:print('Verified completed final MR seed',seed,'; reuse');return existing
    source,_=fm_source_metadata(seed,package);data=package.data('final');folder=ROOT/'final'/f'seed_{seed}';local=LOCAL/'final'/f'seed_{seed}'
    folder.mkdir(parents=True,exist_ok=True);local.mkdir(parents=True,exist_ok=True)
    previous=folder/'training_binding.json'
    if previous.exists():
        b=read_json(previous)['binding'];require(b['implementation_sha256']==FINAL_MR_CODE_SHA256 and b['recipe']==fm_recipe() and b['runtime']==physical_runtime() and b['data']==fm_data_id(data) and b['final_mr_lock_sha256']==fingerprint(lock),'Active final MR resume incompatible; preserve checkpoint')
    base,tokenizer,encoder,adapter=fm_load_models(seed,package);frozen=fm_frozen(base,encoder,adapter);images=fm_images(package,base,data)
    inputs=FinalFrozenInputs(seed,'final',data,encoder,adapter,images,tokenizer);reduced=FinalReducedInputs(seed,data,inputs,images,encoder,adapter)
    model=fm_fresh(seed);initial=tensor_hash(model);attempt=folder/'preflight_runs'/uuid.uuid4().hex;attempt.mkdir(parents=True)
    with isolated_rng():
        parity={c:{'batch32':check_gate_and_reduced(model,inputs,reduced,data['train'][data['train'].category==c].iloc[:32],encoder,adapter,'D'),'batch1':check_gate_and_reduced(model,inputs,reduced,data['train'][data['train'].category==c].iloc[:1],encoder,adapter,'D')} for c in CATEGORIES}
        smoke=fm_smoke_final(model,seed,data,reduced)
    require(initial==tensor_hash(model) and frozen==fm_frozen(base,encoder,adapter),'Preflight changed main/frozen weights')
    test={'passed':True,'seed':seed,'phase':'final-preflight','source_cache':inputs.audit,'MR_off_native_parity':parity,'smoke':smoke,'initial_sha256':initial,'frozen_hashes':frozen,'official_val_used':False}
    write_json(attempt/'test_report.json',test);write_json(folder/'test_report.json',test)
    binding=fm_binding(seed,data,reduced,model,lock,source)
    with isolated_rng():
        result=optimize_stage(model,data['train'],lambda rows,training=True,r=reduced:r.batch(rows,'D',training),None,folder,local,binding,PILOT_HP,lock['mr_epochs'],None)
    require(result['finished'] and result['epoch']==lock['mr_epochs'] and result['metrics']==[] and result['best_mean'] is None,'Final MR budget incomplete/selected on validation')
    require(frozen==fm_frozen(base,encoder,adapter),'Final MR changed frozen source weights')
    receipts={str(folder/name):file_hash(folder/name) for name in ('final.pt','final.json','last.pt','last.json','training_history.csv','metric_history.json','training_binding.json','test_report.json')}
    done={'schema':'visfit-final-mr-completion-v1','passed':True,'completed':True,'seed':seed,'phase':'final','binding':binding,'result':result,'file_receipts':receipts,'frozen_hashes':frozen,'train_rows':18000,'official_val_evaluated_during_training':False,'selected_checkpoint':'final.pt at locked budget; no validation best checkpoint'}
    write_json(folder/'completion.json',done,immutable=True);verified=fm_verify_training(seed,package,lock)
    require(verified is not None,'Final MR completion readback failed')
    print('Final MR verified seed',seed,'| epochs',lock['mr_epochs'],'| steps',result['step'])
    del model,reduced,inputs,encoder,adapter;gc.collect()
    return verified


In [ ]:
class FinalMROff(nn.Module):
    def __init__(self,core):super().__init__();self.core=core
    def encode_query(self,reference_features,text_features,mask,eos):
        return self.core(reference_features['mr_tokens'],reference_features['pooled'],**text_features,mr_off=True)

def fm_compare_source(seed,metric,pred,data,folder):
    source=FINAL_SOURCES[seed]['benchmark_source'];old=fm_pinned(source)
    require(old['phase']=='benchmark' and old['label']=='adapter' and old['seed']==seed and old['encoder_asset']==FINAL_SOURCES[seed]['encoder'] and old['adapter_asset']==FINAL_SOURCES[seed]['adapter'],'Source benchmark provenance differs')
    require(old['data']==fm_data_id(data) and old['lock_sha256']==SOURCE_FINAL_LOCK['fingerprint'] and not old['selected_by_official_val'],'Source benchmark lock/protocol differs')
    verify_receipts(old['file_receipts']);require(file_hash(source['prediction_path'])==source['prediction_sha256'],'Source final per-query ranks changed')
    saved=pd.read_csv(source['prediction_path'],dtype=str,keep_default_na=False);metric_equal(metrics_from_predictions(saved,data),source['metrics'])
    keys=['query_id','category','reference_image_id','target_image_id'];joined=pred.merge(saved[keys+['target_rank']],on=keys,suffixes=('_new','_source'),validate='one_to_one')
    require(len(joined)==6016 and set(pred.query_id)==set(data['validation'].query_id),'Source comparison mapping/coverage differs')
    changed=joined[joined.target_rank_new.astype(int)!=joined.target_rank_source.astype(int)];write_csv(Path(folder)/'MR_off_rank_differences.csv',changed)
    deviations={c+'/'+str(k):abs(metric['per_category'][c][f'R@{k}']-source['metrics']['per_category'][c][f'R@{k}']) for c in CATEGORIES for k in (1,5,10,50)}
    passed=max(deviations.values())<1e-10 and len(changed)==0
    report={'passed':passed,'rank_difference_count':len(changed),'metric_deviation_pp':deviations,'policy':source['comparison_policy'],'source_result_sha256':source['sha256'],'reference_mean':source['metrics']['Mean'],'new_mean':metric['Mean']}
    write_json(Path(folder)/'MR_off_reproduction.json',report)
    require(passed,'Final Adapter MR-off reproduction differs; inspect differences/cache/projection before accepting benchmark, never relax tolerance')
    metric_equal(metric,source['metrics']);return report

def fm_verify_benchmark(seed,label,data,lock,final_done):
    folder=ROOT/'benchmark'/f'seed_{seed}'/label;path=folder/'result.json'
    if not path.is_file():return None
    result=read_json(path);binding=final_done['binding']
    require(result['passed'] and result['phase']=='benchmark' and result['seed']==seed and result['label']==label and result['data']==fm_data_id(data),'Benchmark result mapping/scope differs')
    require(result['final_mr_lock_sha256']==fingerprint(lock) and result['source_encoder']==binding['source_encoder'] and result['source_adapter']==binding['source_adapter'] and result['mr_final_sha256']==final_done['file_receipts'][str(ROOT/'final'/f'seed_{seed}'/'final.pt')],'Benchmark final checkpoint/lock mismatch')
    require(not result['selected_by_official_val'] and result['metrics']['n_query']==6016,'Invalid benchmark selection/coverage')
    verify_receipts(result['file_receipts']);pred=pd.read_csv(folder/'per_query_results.csv',dtype=str,keep_default_na=False)
    metric_equal(metrics_from_predictions(pred,data),result['metrics']);verify_topk(pd.read_csv(folder/'top_k_results.csv',dtype={'query_id':str,'image_id':str}),pred,data,50)
    if label=='adapter':fm_compare_source(seed,result['metrics'],pred,data,folder)
    return result

def fm_benchmark_seed(seed,package,lock,final_done):
    require(FINAL_CFG['enable_benchmark'] is True,'Benchmark opt-in required')
    # No official-val inference before every seed has completed fixed-budget final MR.
    for s in (42,43,44):require(fm_verify_training(s,package,lock) is not None,'All three final MR trainings must complete before benchmark')
    data=package.data('benchmark');existing={label:fm_verify_benchmark(seed,label,data,lock,final_done) for label in ('adapter','adapter_mr')}
    if all(existing.values()):print('Verified completed official-val seed',seed,'; reuse');return existing
    base,tokenizer,encoder,adapter=fm_load_models(seed,package);frozen=fm_frozen(base,encoder,adapter);images=fm_images(package,base,data)
    inputs=FinalFrozenInputs(seed,'benchmark',data,encoder,adapter,images,tokenizer);reduced=FinalReducedInputs(seed,data,inputs,images,encoder,adapter)
    folder=ROOT/'final'/f'seed_{seed}';state=committed_checkpoint_load(folder/'final.pt',fingerprint(final_done['binding']));required_checkpoint(state,final_done['binding'])
    mr=fm_fresh(seed);mr.load_state_dict(state['model'],strict=True);mr.eval().requires_grad_(False);before=tensor_hash(mr)
    batch=lambda rows,training=False,r=reduced:r.batch(rows,'D',False)
    for label,model in [('adapter',FinalMROff(mr)),('adapter_mr',mr)]:
        if existing[label]:continue
        out=ROOT/'benchmark'/f'seed_{seed}'/label;started=time.perf_counter();metric,files=export_evaluation(out,model,data,images,batch,'adapter')
        pred=pd.read_csv(out/'per_query_results.csv',dtype=str,keep_default_na=False)
        if label=='adapter':fm_compare_source(seed,metric,pred,data,out)
        require(before==tensor_hash(mr) and frozen==fm_frozen(base,encoder,adapter),'Benchmark mutated weights')
        result={'schema':'visfit-final-mr-official-val-v1','passed':True,'phase':'benchmark','seed':seed,'label':label,'data':fm_data_id(data),'source_encoder':FINAL_SOURCES[seed]['encoder'],'source_adapter':FINAL_SOURCES[seed]['adapter'],'mr_final_sha256':final_done['file_receipts'][str(folder/'final.pt')],'final_mr_lock_sha256':fingerprint(lock),'metrics':metric,'file_receipts':{str(out/name):sha for name,sha in files.items()},'paired_same_encoder_adapter_gallery':True,'selected_by_official_val':False,'evaluation_seconds':time.perf_counter()-started,'cache':{'text':inputs.key,'reduced':reduced.key,'vision':images.key},'input_audit':inputs.audit}
        write_json(out/'result.json',result,immutable=True);existing[label]=fm_verify_benchmark(seed,label,data,lock,final_done)
        require(existing[label] is not None,'Official-val receipt readback failed')
        print('Official-val verified seed',seed,label,'| Mean',metric['Mean'])
    del mr,state,reduced,inputs,encoder,adapter;gc.collect();return existing

def fm_summary(package,lock):
    data=package.data('benchmark');rows=[];paired=[];costs=[]
    for seed in (42,43,44):
        done=fm_verify_training(seed,package,lock);require(done is not None,'Missing final MR result')
        results={label:fm_verify_benchmark(seed,label,data,lock,done) for label in ('adapter','adapter_mr')};require(all(results.values()),'Missing official-val result')
        values={}
        for label,result in results.items():
            metrics=result['metrics'];values[label]={**metrics['overall'],'Mean':metrics['Mean']};rows.append({'seed':seed,'model':label,**values[label]})
        paired.append({'seed':seed,**{metric:values['adapter_mr'][metric]-values['adapter'][metric] for metric in ('R@1','R@5','R@10','R@50','Mean')}})
        costs.append({'seed':seed,'MR_epochs':lock['mr_epochs'],**done['result']['timing'],'cost_scope':'incremental MR only; excludes frozen encoder/Adapter training and cache/preflight/evaluation'})
    rows=pd.DataFrame(rows);paired=pd.DataFrame(paired);stats=[]
    for label in ('adapter','adapter_mr'):
        for metric in ('R@1','R@5','R@10','R@50','Mean'):
            x=rows[rows.model==label][metric];stats.append({'model':label,'metric':metric,'mean':float(x.mean()),'sample_std':float(x.std(ddof=1)),'n':3})
    for metric in ('R@1','R@5','R@10','R@50','Mean'):
        x=paired[metric];stats.append({'model':'paired_adapter_mr_minus_adapter','metric':metric,'mean':float(x.mean()),'sample_std':float(x.std(ddof=1)),'n':3})
    write_csv(ROOT/'official_val_per_seed.csv',rows);write_csv(ROOT/'official_val_paired_deltas.csv',paired);write_csv(ROOT/'official_val_mean_std.csv',pd.DataFrame(stats));write_csv(ROOT/'MR_incremental_timing.csv',pd.DataFrame(costs))
    payload={'passed':True,'complete_three_seeds':True,'scope':'official-val fixed common development-selected MR epoch; no official-val model selection','mr_epochs':lock['mr_epochs'],'lock_sha256':fingerprint(lock),'per_seed':rows.to_dict('records'),'paired_deltas':paired.to_dict('records'),'mean_sample_std':stats,'incremental_MR_timing':costs,'development_results_kept_separate':True,'significance_or_SOTA_claim':False}
    write_json(ROOT/'official_val_summary.json',payload,immutable=True)
    require(read_json(ROOT/'official_val_summary.json')==payload,'Official summary readback failed')
    compact=['# Final Adapter + MR — official-val','',f'MR common epoch: {lock["mr_epochs"]}; seeds42/43/44; all6016 queries; locked full category galleries.','']
    for label in ('adapter','adapter_mr','paired_adapter_mr_minus_adapter'):
        r=next(x for x in stats if x['model']==label and x['metric']=='Mean');compact.append(f'- {label}: Mean {r["mean"]:.6f} ± {r["sample_std"]:.6f} (sample std; paired delta in percentage points).')
    compact+=['','Development and official-val are separate. MR time is incremental, not total pipeline cost. No significance/SOTA claim. Details: JSON/CSV and per-seed receipts.']
    write_bytes(ROOT/'summary.md','\n'.join(compact).encode())
    from IPython.display import display
    display(rows);display(paired);display(pd.DataFrame(stats));return payload

def fm_progress(phase,seed=None,status='running',error=None):
    checkpoint=ROOT/'final'/f'seed_{seed}'/'last.pt' if seed is not None else None
    step=None
    if checkpoint is not None and checkpoint.with_suffix('.json').is_file():step=read_json(checkpoint.with_suffix('.json')).get('step')
    write_json(ROOT/'progress.json',{'phase':phase,'seed':seed,'status':status,'last_checkpoint':str(checkpoint) if checkpoint is not None and checkpoint.is_file() else None,'step':step,'error':error,'updated_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())})

def run_final_mr_phase(phase):
    require(phase in ('preflight','lock','final','benchmark','summary'),'Unknown final MR phase')
    FINAL_CFG['phase']=phase
    with fm_scope(42):
        seed=None;fm_progress(phase)
        try:
            package=FinalMRPreparation(FINAL_CFG['preparation_root'])
            if phase=='preflight':
                result=fm_resources()
                for seed in (42,43,44):fm_source_metadata(seed,package)
            elif phase=='lock':result=fm_lock(package)
            else:
                lock=fm_read_lock(package)
                write_json(ROOT/'campaign_config.json',{'schema':'visfit-final-mr-campaign-v1','recipe':fm_recipe(),'lock_sha256':fingerprint(lock),'seeds':[42,43,44],'MR_epochs':lock['mr_epochs'],'training_rows':18000,'output_root':str(ROOT),'source_root':FINAL_CFG['source_full_root']},immutable=True)
                write_json(ROOT/'source_provenance.json',FINAL_SOURCES,immutable=True)
                write_json(ROOT/'implementations'/(FINAL_MR_CODE_SHA256+'.json'),{'sha256':FINAL_MR_CODE_SHA256,'scope':'cache/train/benchmark definition sources excluding code pin assignment','pilot_model_sha256':PILOT_MODEL_SHA256,'pilot_runner_sha256':PILOT_RUNNER_SHA256},immutable=True)
                if phase=='final':
                    result={}
                    for seed in (42,43,44):
                        CFG['seed']=seed;fm_progress(phase,seed);result[seed]=fm_train_seed(seed,package,lock)
                elif phase=='benchmark':
                    completed={s:fm_verify_training(s,package,lock) for s in (42,43,44)};require(all(completed.values()),'Final training incomplete')
                    result={}
                    for seed in (42,43,44):
                        CFG['seed']=seed;fm_progress(phase,seed);result[seed]=fm_benchmark_seed(seed,package,lock,completed[seed])
                else:result=fm_summary(package,lock)
            fm_progress(phase,status='verified');print('Phase verified:',phase);return result
        except BaseException as error:
            write_bytes(ROOT/'errors'/(phase+'_'+str(seed)+'_'+uuid.uuid4().hex+'.txt'),traceback.format_exc().encode());fm_progress(phase,seed,status='blocked',error=type(error).__name__+': '+str(error));raise

def run_final_mr_pipeline():
    import fcntl
    local=Path(FINAL_CFG['local_root'])/FINAL_CFG['campaign_id'];local.mkdir(parents=True,exist_ok=True)
    with (local/'active_pipeline.lock').open('a') as guard:
        try:fcntl.flock(guard,fcntl.LOCK_EX|fcntl.LOCK_NB)
        except BlockingIOError:raise RuntimeError('Another final MR pipeline is active; do not launch a duplicate') from None
        try:
            run_final_mr_phase('preflight');FINAL_CFG['enable_lock']=True;run_final_mr_phase('lock');FINAL_CFG['enable_lock']=False
            FINAL_CFG['enable_training']=True;run_final_mr_phase('final');FINAL_CFG['enable_training']=False
            FINAL_CFG['enable_benchmark']=True;run_final_mr_phase('benchmark');FINAL_CFG['enable_benchmark']=False
            return run_final_mr_phase('summary')
        finally:
            for key in ('enable_lock','enable_training','enable_benchmark'):FINAL_CFG[key]=False
            fcntl.flock(guard,fcntl.LOCK_UN)


In [ ]:
# Historical initial final-pipeline diagnostic output retained.
# Superseded only in benchmark verification/CSV reading by the explicit dispatcher below.
# No executable legacy driver here; use the final pipeline control cell.


In [ ]:
# Historical diagnostic output retained; no automatic inference or artifact writes.
if RUN_FINAL_DIAGNOSTICS:
    # Diagnostic only: no optimizer, checkpoint updates or acceptance of mismatched metrics.
    with fm_scope(42):
        package=FinalMRPreparation(FINAL_CFG['preparation_root']);data=package.data('benchmark');lock=fm_read_lock(package)
        failed=read_json(ROOT/'benchmark/seed_42/adapter/MR_off_reproduction.json');print('Failed reproduction:',failed)
        diagnostic=ROOT/'diagnostics'/('seed42_'+uuid.uuid4().hex);diagnostic.mkdir(parents=True)
        base,tokenizer,encoder,adapter=fm_load_models(42,package);images=fm_images(package,base,data)
        inputs=FinalFrozenInputs(42,'benchmark',data,encoder,adapter,images,tokenizer);reduced=FinalReducedInputs(42,data,inputs,images,encoder,adapter)
        native_metric,native_pred,_=retrieval_evaluate(adapter,data,images,inputs.batch,'adapter',0);write_csv(diagnostic/'native_per_query.csv',native_pred)
        keys=['query_id','category','reference_image_id','target_image_id'];source=pd.read_csv(FINAL_SOURCES[42]['benchmark_source']['prediction_path'],dtype=str)
        compare=native_pred.merge(source[keys+['target_rank']],on=keys,validate='one_to_one',suffixes=('_native','_source'))
        native_changed=compare[compare.target_rank_native.astype(int)!=compare.target_rank_source.astype(int)];write_csv(diagnostic/'native_rank_differences.csv',native_changed)
        refs=sorted(set(data['validation'].reference_image_id));audit=read_json(Path(FINAL_CFG['source_full_root'])/'benchmark/seed_42/image_feature_audit.json')
        pool_root=Path(FINAL_CFG['source_full_root'])/'shared_reference_pool'/audit['reference_pool'];original_pool=FinalReadOnlyCache(pool_root,'reference_pooled',refs,LOCAL/'diagnostic_original_pool')
        a=images.take('reference_pooled',refs);b=original_pool.take(refs);pool_delta=np.abs(a-b);pool_diff=np.any(a!=b,axis=1)
        mr=fm_fresh(42).eval().requires_grad_(False);off=FinalMROff(mr);max_abs=0.;unequal=0;count=0
        for c in CATEGORIES:
            f=data['validation'][data['validation'].category==c]
            for start in range(0,len(f),32):
                rows=f.iloc[start:start+32]
                with torch.no_grad():
                    q1=query_forward(adapter,inputs.batch(rows,False),'adapter');q2=query_forward(off,reduced.batch(rows,'D',False),'adapter')
                max_abs=max(max_abs,float((q1-q2).abs().max().cpu()));unequal+=int((q1!=q2).any(1).sum().cpu());count+=len(rows)
        report={'native_Mean':native_metric['Mean'],'source_Mean':FINAL_SOURCES[42]['benchmark_source']['metrics']['Mean'],'native_source_rank_difference_count':len(native_changed),'cached_off_native_query_max_abs':max_abs,'cached_off_native_unequal_queries':unequal,'queries_compared':count,'reference_pool_max_abs':float(pool_delta.max()),'reference_pool_different_rows':int(pool_diff.sum()),'original_pool_fingerprint':audit['reference_pool'],'diagnostic_path':str(diagnostic)}
        write_json(diagnostic/'report.json',report);print('Diagnostic report:',report)
        print('Native rank differences (first5):',native_changed.head(5).to_dict('records'))


In [ ]:
# Historical diagnostic output retained; no automatic inference or artifact writes.
if RUN_FINAL_DIAGNOSTICS:
    # Diagnose score arithmetic only. Never use an alternative engine to select results.
    def diagnose_score_arithmetic(data,images,inputs,query_id):
        category=query_id.split(':')[0];f=data['validation'][data['validation'].category==category];position=np.flatnonzero(f.query_id.to_numpy()==query_id);require(len(position)==1,'Ambiguous diagnostic query')
        start=int(position[0])//32*32;rows=f.iloc[start:start+32];local=int(position[0])-start
        gallery=data['gallery'][data['gallery'].category==category].sort_values('gallery_index',kind='stable');ids=gallery.image_id.tolist();target=ids.index(rows.iloc[local].target_image_id)
        vectors=images.take('target_pooled',ids);vectors=vectors/np.linalg.norm(vectors,axis=-1,keepdims=True)
        with torch.inference_mode():q=query_forward(inputs.adapter,inputs.batch(rows,False),'adapter').cpu().numpy()
        q=q/np.linalg.norm(q,axis=-1,keepdims=True);batch_scores=q@vectors.T;single=q[local:local+1]@vectors.T;double=q[local:local+1].astype(np.float64)@vectors.astype(np.float64).T
        def rank(s):return int(1+(s>s[target]).sum()+((s==s[target])&(np.arange(len(s))<target)).sum())
        near=np.argsort(np.abs(batch_scores[local]-batch_scores[local,target]),kind='stable')[:8]
        result={'query_id':query_id,'target_image_id':ids[target],'rank_current_FP32_batch32':rank(batch_scores[local]),'rank_same_query_FP32_batch1':rank(single[0]),'rank_FP64_diagnostic_only':rank(double[0]),'source_rank':int(pd.read_csv(FINAL_SOURCES[42]['benchmark_source']['prediction_path'],dtype=str).set_index('query_id').loc[query_id,'target_rank']),'near_target':[{'image_id':ids[i],'score_FP32':float(batch_scores[local,i]),'gap_FP32_to_target':float(batch_scores[local,i]-batch_scores[local,target]),'gap_FP64_to_target':float(double[0,i]-double[0,target]),'gallery_index':int(i)} for i in near],'no_protocol_or_result_engine_changed':True}
        return result
    with fm_scope(42):
        precision_report=diagnose_score_arithmetic(data,images,inputs,'shirt:val:000693')
        write_json(diagnostic/'score_arithmetic.json',precision_report);print(precision_report)


In [ ]:
VERIFIED_BENCHMARK_SHA256 = 'e36cde9b336018382a26dd24629a8744b7505f906581ce6b279f7ebc02235f0e'
# Correct source-contract verification. Original failed artifacts/guard remain intact.
# No change to cache, MR weights, training fingerprint, evaluator, metric tolerance or ranking engine.
def verified_benchmark_root():return ROOT/'official_val_v2'

@torch.inference_mode()
def verify_native_off_pair(inputs,reduced,images,data,seed):
    core=fm_fresh(seed).eval().requires_grad_(False);off=FinalMROff(core);compared=0
    for c in CATEGORIES:
        frame=data['validation'][data['validation'].category==c]
        for start in range(0,len(frame),32):
            rows=frame.iloc[start:start+32];native=query_forward(inputs.adapter,inputs.batch(rows,False),'adapter');cached=query_forward(off,reduced.batch(rows,'D',False),'adapter')
            require(torch.equal(native,cached),'Native Adapter vs MR-off query differs in evaluator inference context')
            compared+=len(rows)
    require(compared==6016,'Incomplete native/MR-off equivalence check')
    source_audit=read_json(Path(FINAL_CFG['source_full_root'])/'benchmark'/f'seed_{seed}'/'image_feature_audit.json')
    refs=sorted(set(data['validation'].reference_image_id));path=Path(FINAL_CFG['source_full_root'])/'shared_reference_pool'/source_audit['reference_pool']
    original=FinalReadOnlyCache(path,'reference_pooled',refs,LOCAL/'diagnostic_original_pool')
    require(np.array_equal(images.take('reference_pooled',refs),original.take(refs)),'Source/current pooled image features differ')
    return {'passed':True,'native_MR_off_bitwise_equal_queries':compared,'pooled_source_bitwise_equal_references':len(refs),'source_pool_fingerprint':source_audit['reference_pool'],'evaluator_context':'torch.inference_mode; unchanged batches32'}

def verify_reported_source_control(seed,metric,pred,data,out,pair_proof):
    require(pair_proof['passed'] and pair_proof['native_MR_off_bitwise_equal_queries']==6016,'Native MR-off equivalence required')
    source=FINAL_SOURCES[seed]['benchmark_source'];record=fm_pinned(source)
    require(record['phase']=='benchmark' and record['label']=='adapter' and record['seed']==seed and record['data']==fm_data_id(data),'Source benchmark mapping/protocol differs')
    require(record['encoder_asset']==FINAL_SOURCES[seed]['encoder'] and record['adapter_asset']==FINAL_SOURCES[seed]['adapter'] and record['lock_sha256']==SOURCE_FINAL_LOCK['fingerprint'] and not record['selected_by_official_val'],'Source checkpoint/lock provenance differs')
    verify_receipts(record['file_receipts']);require(file_hash(source['prediction_path'])==source['prediction_sha256'],'Historical per-query file changed')
    old=pd.read_csv(source['prediction_path'],dtype=str,keep_default_na=False);metric_equal(metrics_from_predictions(old,data),source['metrics']);metric_equal(metric,source['metrics'])
    keys=['query_id','category','reference_image_id','target_image_id'];joined=pred.merge(old[keys+['target_rank']],on=keys,validate='one_to_one',suffixes=('_current','_historical'))
    require(len(joined)==6016 and set(pred.query_id)==set(data['validation'].query_id),'Source query coverage/mapping differs')
    current=joined.target_rank_current.astype(int);historical=joined.target_rank_historical.astype(int)
    # Stronger than aggregate metric equality: no hit changes or compensating errors at any reported cutoff.
    hits={f'Hit@{k}':int(((current<=k)!=(historical<=k)).sum()) for k in (1,5,10,50)}
    require(all(count==0 for count in hits.values()),'Source per-query reported Hit@K differs; do not relax metric tolerance')
    changes=joined[current!=historical];write_csv(out/'historical_rank_differences.csv',changes)
    proof={'passed':True,'metric_tolerance_pp':1e-10,'metric_policy':'Original unchanged metric_equal: all category R@1/5/10/50 and Mean; strict abs difference<1e-10','per_query_hit_difference_counts':hits,'historical_rank_difference_count':len(changes),'historical_rank_differences':changes.to_dict('records'),'native_pair':pair_proof,'source_mean':source['metrics']['Mean'],'current_mean':metric['Mean'],'note':'Historical non-cutoff ranks are retained as diagnostics; not rewritten or asserted bitwise reproducible across runtime. Reported hits and metrics must match exactly under source metric tolerance.'}
    write_json(out/'MR_off_reproduction.json',proof);return proof

def read_verified_ranks(path):
    frame=pd.read_csv(path,dtype={'query_id':str,'category':str,'reference_image_id':str,'target_image_id':str},keep_default_na=False)
    numeric=pd.to_numeric(frame['target_rank'],errors='raise');require(np.isfinite(numeric).all() and (numeric>0).all() and np.equal(numeric,np.floor(numeric)).all(),'Invalid target rank schema')
    frame['target_rank']=numeric.astype(np.int64);return frame

def verify_benchmark_receipt_v2(seed,label,data,lock,done):
    out=verified_benchmark_root()/f'seed_{seed}'/label;path=out/'result.json'
    if not path.is_file():return None
    r=read_json(path);require(r['passed'] and r['seed']==seed and r['label']==label and r['phase']=='benchmark' and r['data']==fm_data_id(data),'Verified benchmark data/label differs')
    require(r['MR_final_sha256']==done['file_receipts'][str(ROOT/'final'/f'seed_{seed}'/'final.pt')] and r['lock_sha256']==fingerprint(lock) and r['benchmark_implementation_sha256']==VERIFIED_BENCHMARK_SHA256,'Verified benchmark checkpoint/code/lock differs')
    require(r['source_encoder']==FINAL_SOURCES[seed]['encoder'] and r['source_adapter']==FINAL_SOURCES[seed]['adapter'] and not r['selected_by_official_val'],'Verified benchmark source differs')
    verify_receipts(r['file_receipts']);pred=read_verified_ranks(out/'per_query_results.csv');metric_equal(metrics_from_predictions(pred,data),r['metrics'])
    verify_topk(pd.read_csv(out/'top_k_results.csv',dtype={'query_id':str,'image_id':str}),pred,data,50)
    if label=='adapter':verify_reported_source_control(seed,r['metrics'],pred,data,out,r['native_pair_proof'])
    return r

def benchmark_seed_v2(seed,package,lock,done):
    require(FINAL_CFG['enable_benchmark'] and not FINAL_CFG['enable_training'],'Benchmark only; training must be disabled')
    data=package.data('benchmark');existing={label:verify_benchmark_receipt_v2(seed,label,data,lock,done) for label in ('adapter','adapter_mr')}
    if all(existing.values()):return existing
    base,tokenizer,encoder,adapter=fm_load_models(seed,package);frozen=fm_frozen(base,encoder,adapter);images=fm_images(package,base,data)
    inputs=FinalFrozenInputs(seed,'benchmark',data,encoder,adapter,images,tokenizer);reduced=FinalReducedInputs(seed,data,inputs,images,encoder,adapter)
    pair=verify_native_off_pair(inputs,reduced,images,data,seed)
    state=committed_checkpoint_load(ROOT/'final'/f'seed_{seed}'/'final.pt',fingerprint(done['binding']));required_checkpoint(state,done['binding'])
    mr=fm_fresh(seed);mr.load_state_dict(state['model'],strict=True);mr.eval().requires_grad_(False);before=tensor_hash(mr);batch=lambda rows,training=False,r=reduced:r.batch(rows,'D',False)
    for label,model in [('adapter',FinalMROff(mr)),('adapter_mr',mr)]:
        if existing[label]:continue
        out=verified_benchmark_root()/f'seed_{seed}'/label;started=time.perf_counter();metric,receipts=export_evaluation(out,model,data,images,batch,'adapter')
        pred=read_verified_ranks(out/'per_query_results.csv')
        verify_topk(pd.read_csv(out/'top_k_results.csv',dtype={'query_id':str,'image_id':str}),pred,data,50)
        metric_equal(metrics_from_predictions(pred,data),metric)
        if label=='adapter':verify_reported_source_control(seed,metric,pred,data,out,pair)
        require(before==tensor_hash(mr) and frozen==fm_frozen(base,encoder,adapter),'Inference changed frozen weights')
        record={'passed':True,'phase':'benchmark','seed':seed,'label':label,'data':fm_data_id(data),'metrics':metric,'source_encoder':FINAL_SOURCES[seed]['encoder'],'source_adapter':FINAL_SOURCES[seed]['adapter'],'MR_final_sha256':done['file_receipts'][str(ROOT/'final'/f'seed_{seed}'/'final.pt')],'lock_sha256':fingerprint(lock),'benchmark_implementation_sha256':VERIFIED_BENCHMARK_SHA256,'training_implementation_sha256':FINAL_MR_CODE_SHA256,'data_extension_sha256':FINAL_MR_DATA_EXTENSION_SHA256,'selected_by_official_val':False,'file_receipts':{str(out/name):sha for name,sha in receipts.items()},'native_pair_proof':pair,'evaluation_seconds':time.perf_counter()-started,'input_audit':inputs.audit,'cache':{'text':inputs.key,'reduced':reduced.key,'vision':images.key}}
        write_json(out/'result.json',record,immutable=True);existing[label]=verify_benchmark_receipt_v2(seed,label,data,lock,done)
        print('Official-val verified',seed,label,'Mean',metric['Mean'])
    del mr,state,reduced,inputs,encoder,adapter;gc.collect();return existing

def summarize_verified_benchmark(package,lock):
    data=package.data('benchmark');rows=[];paired=[];costs=[];source_differences={}
    for seed in (42,43,44):
        done=fm_verify_training(seed,package,lock);require(done is not None,'Missing final MR seed')
        results={label:verify_benchmark_receipt_v2(seed,label,data,lock,done) for label in ('adapter','adapter_mr')};require(all(results.values()),'Paired official-val incomplete')
        values={}
        for label,r in results.items():values[label]={**r['metrics']['overall'],'Mean':r['metrics']['Mean']};rows.append({'seed':seed,'model':label,**values[label]})
        paired.append({'seed':seed,**{k:values['adapter_mr'][k]-values['adapter'][k] for k in ('R@1','R@5','R@10','R@50','Mean')}})
        costs.append({'seed':seed,'MR_epochs':lock['mr_epochs'],**done['result']['timing'],'scope':'incremental MR only; excludes source encoder/Adapter training, cache/preflight/evaluation'})
        source_differences[seed]=read_json(verified_benchmark_root()/f'seed_{seed}'/'adapter/MR_off_reproduction.json')
    per=pd.DataFrame(rows);delta=pd.DataFrame(paired);stats=[]
    for label in ('adapter','adapter_mr','paired_delta'):
        for metric in ('R@1','R@5','R@10','R@50','Mean'):
            v=delta[metric] if label=='paired_delta' else per[per.model==label][metric]
            stats.append({'model':label,'metric':metric,'mean':float(v.mean()),'sample_std':float(v.std(ddof=1)),'n':3})
    for name,frame in [('official_val_per_seed.csv',per),('official_val_paired_deltas.csv',delta),('official_val_mean_std.csv',pd.DataFrame(stats)),('MR_incremental_timing.csv',pd.DataFrame(costs))]:write_csv(ROOT/name,frame)
    summary={'passed':True,'complete_three_seeds':True,'scope':'official-val; common development-selected MR budget fixed before evaluation','mr_epochs':lock['mr_epochs'],'lock_sha256':fingerprint(lock),'per_seed':rows,'paired_deltas':paired,'mean_sample_std':stats,'incremental_MR_timing':costs,'source_reproduction':source_differences,'benchmark_root':str(verified_benchmark_root()),'benchmark_implementation_sha256':VERIFIED_BENCHMARK_SHA256,'development_results_kept_separate':True,'significance_or_SOTA_claim':False}
    write_json(ROOT/'official_val_summary.json',summary,immutable=True)
    lines=['# Final Adapter + MR — official-val','',f'Common MR budget: {lock["mr_epochs"]} epoch; seeds42/43/44; all6016 queries and full locked galleries.','']
    for label in ('adapter','adapter_mr','paired_delta'):
        r=next(x for x in stats if x['model']==label and x['metric']=='Mean');lines.append(f'- {label}: Mean {r["mean"]:.6f} ± {r["sample_std"]:.6f}; sample std, delta in percentage points.')
    lines+=['','All reported per-query hits and source metrics reproduced; historical non-cutoff rank differences remain in JSON/CSV. MR time is incremental, not total pipeline cost. No significance/SOTA claim.']
    write_bytes(ROOT/'summary.md','\n'.join(lines).encode())
    from IPython.display import display
    display(per);display(delta);display(pd.DataFrame(stats));return summary

def run_verified_benchmark():
    require(not FINAL_CFG['enable_training'],'No training during benchmark resume')
    with fm_scope(42):
        package=FinalMRPreparation(FINAL_CFG['preparation_root']);lock=fm_read_lock(package);done={s:fm_verify_training(s,package,lock) for s in (42,43,44)};require(all(done.values()),'All three final MR checkpoints must verify first')
        write_json(ROOT/'implementations'/('benchmark_'+VERIFIED_BENCHMARK_SHA256+'.json'),{'sha256':VERIFIED_BENCHMARK_SHA256,'data_extension_sha256':FINAL_MR_DATA_EXTENSION_SHA256,'training_code_sha256_unchanged':FINAL_MR_CODE_SHA256,'scope':'correct source metric contract plus all-query native/MR-off exact equivalence, per-query hit equality, historical rank drift disclosure'},immutable=True)
        seed=None
        try:
            for seed in (42,43,44):
                CFG['seed']=seed;fm_progress('benchmark',seed);benchmark_seed_v2(seed,package,lock,done[seed])
            result=summarize_verified_benchmark(package,lock);fm_progress('summary',status='verified');return result
        except BaseException as error:
            write_bytes(ROOT/'errors'/('verified_benchmark_'+str(seed)+'_'+uuid.uuid4().hex+'.txt'),traceback.format_exc().encode());fm_progress('benchmark',seed,status='blocked',error=type(error).__name__+': '+str(error));raise
        finally:
            for key in ('enable_lock','enable_training','enable_benchmark'):FINAL_CFG[key]=False

def run_complete_final_mr_pipeline():
    import fcntl
    local=Path(FINAL_CFG['local_root'])/FINAL_CFG['campaign_id'];local.mkdir(parents=True,exist_ok=True)
    with (local/'active_pipeline.lock').open('a') as guard:
        try:fcntl.flock(guard,fcntl.LOCK_EX|fcntl.LOCK_NB)
        except BlockingIOError:raise RuntimeError('A final MR pipeline is already active') from None
        try:
            run_final_mr_phase('preflight');FINAL_CFG['enable_lock']=True;run_final_mr_phase('lock');FINAL_CFG['enable_lock']=False
            FINAL_CFG['enable_training']=True;run_final_mr_phase('final');FINAL_CFG['enable_training']=False
            FINAL_CFG['enable_benchmark']=True;return run_verified_benchmark()
        finally:
            for key in ('enable_lock','enable_training','enable_benchmark'):FINAL_CFG[key]=False
            fcntl.flock(guard,fcntl.LOCK_UN)


In [ ]:
# Output below belongs to the authorized True run on2026-11-20.
# Safe default restored to False; retained output was not produced under this False setting.
RUN_FINAL_PIPELINE = False
if RUN_FINAL_PIPELINE:
    VERIFIED_RESULT = run_complete_final_mr_pipeline()
else:
    print('Final pipeline is opt-in; no training/evaluation started.')


In [ ]:
# Completed-artifact readback only; no training or evaluation rerun.
RUN_FINAL_PIPELINE=False
RUN_FINAL_LOCK=False
RUN_PILOT=False
for key in ('enable_lock','enable_training','enable_benchmark'):FINAL_CFG[key]=False
with fm_scope(42):
    package=FinalMRPreparation(FINAL_CFG['preparation_root']);lock=fm_read_lock(package);data=package.data('benchmark');summary=read_json(ROOT/'official_val_summary.json')
    require(summary['passed'] and summary['complete_three_seeds'] and summary['mr_epochs']==lock['mr_epochs'] and summary['lock_sha256']==fingerprint(lock),'Summary incomplete or wrong lock')
    require(len(summary['per_seed'])==6 and len(summary['paired_deltas'])==3 and all(s['n']==3 for s in summary['mean_sample_std']),'Incomplete paired statistics')
    reports={};receipts={str(ROOT/'final_mr_lock.json'):file_hash(ROOT/'final_mr_lock.json'),str(ROOT/'official_val_summary.json'):file_hash(ROOT/'official_val_summary.json')}
    historical_changes=0
    for seed in (42,43,44):
        done=fm_verify_training(seed,package,lock);require(done is not None,'Missing final checkpoint')
        test=read_json(ROOT/'final'/f'seed_{seed}'/'test_report.json');smoke=test['smoke']
        require(test['passed'] and smoke['passed'] and not test['official_val_used'] and not smoke['official_val_accessed'],'Preflight leaked official-val or failed')
        require(smoke['resume_epoch_step_cursor_model_optimizer_RNG_exact'] and smoke['query_target_independent'] and smoke['teacher_detached'],'Missing required preflight evidence')
        require(test['frozen_hashes']==done['frozen_hashes'],'Frozen hash provenance changed')
        result={label:verify_benchmark_receipt_v2(seed,label,data,lock,done) for label in ('adapter','adapter_mr')};require(all(result.values()),'Incomplete official-val pair')
        reproduction=read_json(verified_benchmark_root()/f'seed_{seed}'/'adapter/MR_off_reproduction.json')
        require(reproduction['passed'] and all(v==0 for v in reproduction['per_query_hit_difference_counts'].values()) and reproduction['native_pair']['native_MR_off_bitwise_equal_queries']==6016,'Source control verification incomplete')
        historical_changes+=reproduction['historical_rank_difference_count']
        reports[seed]={'preflight':test,'source_progress':read_json(ROOT/'final'/f'seed_{seed}'/'source_progress_audit.json'),'MR_epochs':done['result']['epoch'],'MR_steps':done['result']['step'],'MR_timing':done['result']['timing'],'initial_sha256':done['result']['initial_sha256'],'frozen_hashes':done['frozen_hashes'],'MR_off_reproduction':reproduction,'n_query':6016,'gallery_counts':result['adapter']['metrics']['gallery_counts'],'final_sources':{'encoder':done['binding']['source_encoder'],'adapter':done['binding']['source_adapter']}}
        for path in [ROOT/'final'/f'seed_{seed}'/'completion.json',verified_benchmark_root()/f'seed_{seed}'/'adapter/result.json',verified_benchmark_root()/f'seed_{seed}'/'adapter_mr/result.json']:
            receipts[str(path)]=file_hash(path)
    verify_receipts(receipts)
    FINAL_READBACK={'passed':True,'scope':'three-seed fixed-budget final MR and paired official-val; separate from development','MR_epochs':lock['mr_epochs'],'lock_sha256':fingerprint(lock),'training_code_sha256':FINAL_MR_CODE_SHA256,'benchmark_code_sha256':VERIFIED_BENCHMARK_SHA256,'data_extension_sha256':FINAL_MR_DATA_EXTENSION_SHA256,'flags_false':all(not FINAL_CFG[k] for k in ('enable_lock','enable_training','enable_benchmark')),'seed_tests':reports,'receipt_map':receipts,'historical_all_ranks_identical':historical_changes==0,'historical_non_cutoff_rank_difference_count':historical_changes,'reported_per_query_hits_identical':True,'metric_tolerance_unchanged_pp':1e-10,'initial_guard_fix':'Original source contract verifies reported metrics; initial added all-rank historical bitwise rule was overstrict. Historical rank differences disclosed, no score/rank rewriting. Require exact current native/MR-off query equality and exact per-query reported hit equality.','CSV_fix':'Strict numeric integer positive target_rank on serialized CSV readback. Failed prior output folders retained.','cost_scope':'MR incremental only, not total source encoder/Adapter training cost','statistics':summary['mean_sample_std'],'no_significance_or_SOTA_claim':True}
    require(FINAL_READBACK['flags_false'],'Flags remain enabled');write_json(ROOT/'test_report.json',FINAL_READBACK,immutable=True)
    write_json(ROOT/'implementations'/('data_'+FINAL_MR_DATA_EXTENSION_SHA256+'.json'),{'sha256':FINAL_MR_DATA_EXTENSION_SHA256,'source_anchors':FINAL_PREPARATION_ANCHORS,'final_training_binding_compatibility':'verified unchanged by strict completion/receipt validation for all3 seeds; benchmark carries original full source anchor set'},immutable=True)
    require(read_json(ROOT/'test_report.json')['passed'],'Final test report readback failed')
    from IPython.display import Markdown,display
    display(Markdown((ROOT/'summary.md').read_text()))
    print('Verified: all3 final checkpoints and all6 benchmark receipts; reported source hits/metrics exact; native/MR-off full query equality. Historical non-cutoff rank differences:',historical_changes)
    print('Execution flags False. Artifact root:',ROOT)
